# Bootstrap Consolidated: damage-function benchmarking in one notebook

This notebook is a self-contained framework for testing damage/vulnerability
functions against a *known* ground truth. A hazard variable and a loss
variable are simulated jointly from a copula with fixed, exact marginal
distributions, so every quantity of interest -- conditional means, tail
expectations, VaR, ES -- has a closed form or a quadrature-computable exact
value. Repeatedly drawing bootstrap subsamples from that population and
fitting each candidate damage function on every subsample turns "how good is
this method?" into a measurable question: compare its bootstrap distribution
of estimates against the exact answer.

Every piece of reusable logic (distributions, copula, estimators, evaluation
engine, tables, plots) is defined once, in the sections below, with no
external package dependency -- everything needed to run this notebook is
either a standard scientific-Python library or defined in an earlier cell of
this same file. Running a new scenario only ever means editing **Part 2
(Configuration)** and re-running the notebook — the metaparameters, which
hazard/loss distributions to use, which copula, and which methods (including
whether each trains on the raw subsample or the binned summary) are all
declared there.

## Contents

0. This overview
1. Imports
2. Configuration (metaparameters, distributions, copula, methods)
3. Plot style
4. Distributions (hazard/loss margins, extensible registry)
5. Copula (Pickands dependence functions, extensible registry)
6. Theoretical conditional targets (exact `E[L|H=h]`, `E[L|H>=h]` by quadrature)
7. Damage-function / methodology estimators (Quadratic, Sigmoid, Isotonic, IDR),
   and binning
8. Bootstrap evaluation engine
9. Reporting and plotting
10. Runs: (a) Light-scale, (b) Full-scale sweep (flag-guarded), (c)
    tail-heaviness sensitivity sweep, (d) extended diagnostic figures
    (upper-tail comparison, hazard-loss risk diagram, IDR conditional law)
11. Notes
12. HANZE real-data case study (interactive dashboard on real flood-loss data)



## Part 1 — Imports

External libraries only, no local package dependency.

**Cell below:** the scientific-Python stack (`numpy`, `pandas`, `scipy`,
`matplotlib`, `sklearn`) plus three *optional* dependencies wrapped in
`try/except ImportError` so the notebook still imports cleanly without them:
`isodistrreg` (needed only when the `"IDR"` method is actually fit), `h5py`
(needed only for `data_backend="hdf5"`, Part 10b), and `joblib` (needed only
to parallelize the bootstrap loop across subsamples — without it,
`run_fullbaseline_predictions_both` falls back to a plain sequential loop).
`DATA_DIR`/`RESULTS_DIR`/`PLOTS_DIR` are the sibling `data/`/`results/`/`plots/` folders (located portably, wherever the notebook happens to be run from) where the optional HDF5 population/subsample cache, saved result tables, and saved figures are written, respectively. A fifth optional import, `ipywidgets`, is only needed for Part 12's interactive dashboard.


In [ ]:
from __future__ import annotations

import copy
import re
import warnings
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Callable, Iterable, Literal

import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.optimize import OptimizeWarning, brentq, curve_fit, isotonic_regression as _pava, minimize, minimize_scalar
from scipy.special import betainc, betaln, expit, gammaln, logsumexp
from scipy.stats import gaussian_kde, t as student_t

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

from sklearn.isotonic import IsotonicRegression

try:
    from IPython.display import HTML, clear_output, display
except ImportError:  # plain interpreter
    HTML = lambda text: text
    clear_output = lambda wait=False: None
    display = print

try:
    from isodistrreg import IDR as _OFFICIAL_IDR
except ImportError:
    _OFFICIAL_IDR = None

try:
    import h5py
except ImportError:
    h5py = None

try:
    from joblib import Parallel, delayed
except ImportError:
    Parallel = None
    delayed = None

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

warnings.filterwarnings("ignore", category=RuntimeWarning)

def _find_project_root(anchor: str = "data") -> Path:
    """Locate the project root (containing data/, results/, plots/) by
    walking up from the working directory -- portable whether Jupyter starts
    in the repo root or the notebook's own folder."""
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / anchor).is_dir():
            return candidate
    return cwd


# Sibling data/results/plots folders where generated population caches (Part
# 8), saved result tables (Part 9/10b), and saved figures (Part 3) go.
PROJECT_ROOT = _find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
PLOTS_DIR = PROJECT_ROOT / "plots"


## Part 2 — Configuration

Everything a user is expected to change lives in this Part: metaparameters,
which hazard/loss distributions to use, which copula, and which methods (and
whether each trains on the raw subsample or the binned summary). Config
objects only reference distributions/copulas/methods **by name** (strings) —
the registries those names resolve against are populated later, in Parts 4, 5
and 7, by `register_distribution`/`register_copula`/`register_method` calls.
A distribution/copula/method that isn't built in yet can be added by writing
a small class/function and calling the matching `register_...` — no other
cell needs to change.

This Part is split into a few pieces: the small "spec" dataclasses that just
carry a registry name and parameters, reference tables of what's currently
registered (so you don't have to read Parts 4/5/7 first to know what's
available), the "mechanics" dataclasses that hold the actual bootstrap
metaparameters, and finally two ready-to-use presets.


In [ ]:
@dataclass(frozen=True)
class DistributionSpec:
    """A distribution chosen by registry name plus its parameters.

    Used identically for the hazard slot and the loss slot -- nothing here is
    hazard-only or loss-only by construction.
    """

    name: str
    params: dict = field(default_factory=dict)


@dataclass(frozen=True)
class CopulaSpec:
    name: str
    params: dict = field(default_factory=dict)
    mirror: bool | None = None  # None -> registry default for this copula


@dataclass(frozen=True)
class MethodConfig:
    name: str
    train_on: Literal["raw", "binned"] | None = None  # None -> registry default
    fit_kwargs: dict = field(default_factory=dict)


### Available distributions

Referenced by name from `DistributionSpec`, used identically for the hazard
slot and the loss slot -- nothing ties a distribution to one or the other.
Each row is what's registered in Part 4:

| Name | Parameters | What it is |
|---|---|---|
| `"weibull_min"` | `c` (shape), `loc`, `scale` | The usual choice for the hazard margin here -- a Weibull severity distribution. `c<1` gives a heavy-ish, decreasing hazard rate (e.g. `c=2/3` for a normalised precipitation-event severity). |
| `"lognorm"` | `s` (shape = sigma of the underlying normal), `loc`, `scale` (`scale = exp(mu)`) | Lognormal losses -- moderately heavy right tail, finite moments of every order. Larger `s` means a heavier tail. |
| `"pareto"` | `b` (tail index / shape), `loc`, `scale` | Classic power-law losses. Mean is finite only for `b > 1`; the tail gets heavier as `b` decreases toward 1. |
| `"dagum"` | `alpha` (tail index), `p` (body shape), `b` (scale, default 1.0) | A margin whose tail heaviness (`alpha`) and body shape (`p`) are controlled independently -- two losses with the same `alpha` and different `p` share the same asymptotic tail but look very different at a realistic sample size. Mean finite only for `alpha > 1`. |
| `"gpd"` | `alpha` (tail index = `1/xi`), `scale` (default 1.0) | A Generalised Pareto distribution used directly as a loss or hazard margin, parametrised by tail index rather than shape. Mean finite only for `alpha > 1`. |

To add a new distribution: write a class exposing `cdf`/`ppf`/`mean`/`rvs`
(optionally `sf`/`isf`/a closed-form `expected_shortfall`), and call
`register_distribution("your_name", factory)` in Part 4. Nothing else needs
to change -- `DistributionSpec("your_name", dict(...))` will work immediately.


### Available copulas

Referenced by name from `CopulaSpec`, registered in Part 5:

| Name | Parameters | What it is |
|---|---|---|
| `"gumbel"` | `theta` (dependence, default 2.0; `theta=1` is independence, larger is stronger dependence) | A symmetric extreme-value copula -- hazard and loss are equally "in charge" of the joint tail behavior. |
| `"lorenz"` | none (`mirror` defaults to `True`) | An asymmetric power-law extreme-value copula. Mirroring orients which margin the asymmetry favors; the default matches how it's used throughout this notebook. |

`CopulaSpec(name, params, mirror=None)` -- `mirror=None` uses the registered
default; pass `True`/`False` explicitly to override it for a given scenario.

To add a new copula: write its Pickands dependence function `A(t)` plus first
and second derivatives, and call `register_copula("your_name", builder)` in
Part 5.


### Available methods

Referenced by name from `MethodConfig`, registered in Part 7.
`default_train_on` is what each method uses unless a `MethodConfig.train_on`
override says otherwise; `reports` controls which results tables the method
contributes to (most contribute to all three: pointwise, tail, and marginal
VaR/ES).

| Name | What it fits | `default_train_on` | Key parameters |
|---|---|---|---|
| `"Quadratic"` | A parabola $y=ax^2+bx+c$ (either orientation -- no sign constraint on $a$ beyond significance), kept only if the curvature is statistically significant | binned | `alpha` (significance level, default 0.05), `intercept` |
| `"Sigmoid"` | A logistic curve fit to rescaled losses, kept only if the steepness is significant | binned | `alpha` (default 0.05), `maxfev`, `p0` |
| `"Isotonic"` | A monotone (non-decreasing) curve, no significance screening | binned | none |
| `"Isotonic (raw)"` | The same monotone curve, trained on the unbinned pairs -- a control arm for isolating what binning costs | raw | none |
| `"IDR"` | The full conditional loss *distribution* (not just its mean), via isotonic distributional regression | raw | `close_at_max` |

To add a new method: write a fit function/class and register a `MethodSpec`
in Part 7.

In [ ]:
@dataclass(frozen=True)
class ScenarioConfig:
    """The data-generating process: hazard, loss, copula, population size, seed."""

    seed: int
    N: int
    hazard: DistributionSpec
    loss: DistributionSpec
    copula: CopulaSpec


@dataclass(frozen=True)
class RunConfig:
    """Bootstrap/evaluation mechanics: subsample size, iterations, methods."""

    n: int
    iterations: int
    bins: int
    var_levels: tuple
    methods: tuple
    data_backend: Literal["memory", "hdf5"] = "memory"
    n_jobs: int | None = None
    bin_summary_func: Callable = np.mean
    bin_x: Literal["midpoint", "mean"] = "midpoint"
    collect_diagnostics: bool = False
    rebuild_cache: bool = False


@dataclass(frozen=True)
class StudyConfig:
    scenario: ScenarioConfig
    run: RunConfig
    label: str | None = None

    def cache_tag(self) -> str:
        s, r = self.scenario, self.run
        return (
            f"{s.copula.name.lower()}_{s.hazard.name.lower()}_{s.loss.name.lower()}_"
            f"N{s.N}_n{r.n}_iter{r.iterations}_bins{r.bins}"
        )


### Metaparameters

These live on `ScenarioConfig`/`RunConfig` and control precision vs. runtime:

- **`N`** -- population size for the synthetic "true" hazard/loss universe.
  Larger `N` makes the full-sample sanity checks (Part 10) tighter, but is
  mostly a memory/generation-time cost, since the exact ground truth (Part 6)
  is computed by quadrature, not from this sample.
- **`n`** -- subsample size per bootstrap resample: how much data each fitted
  method actually sees. This is the realistic "size of one real dataset."
- **`iterations`** -- number of bootstrap resamples. More iterations give a
  tighter estimate of each method's bias/variance, at roughly linear extra
  runtime (parallelizable across `n_jobs` if `joblib` is installed).
- **`bins`** -- number of probability bins for methods trained on
  `"binned"` data (Part 7f).
- **`var_levels`** -- the hazard quantiles $q$ every table/plot reports at.
  Note: no empirical quantile is defined above $1-1/n$, so a `q` above that
  bound will simply repeat the subsample-maximum-based estimate (see Part
  10a's reading note).
- **`data_backend`** -- `"memory"` regenerates the population every call
  (fine for small `N`); `"hdf5"` persists the population and subsample
  indices to disk, keyed by `StudyConfig.cache_tag()`, so an expensive
  population is only ever generated once.
- **`rebuild_cache`** -- when `True`, bypasses both caches this notebook
  keeps: the `"hdf5"` population/subsample cache above, and (Part 9)
  `load_or_run_bootstrap_study`'s reuse of an already-saved
  `results/pointwise_<tag>.csv`/`tail_<tag>.csv` pair, which otherwise
  skips redoing the fitting pass -- the dominant cost of a full-scale
  run -- for a scenario whose results already exist on disk.
- **`collect_diagnostics`** -- when `True`, also records a per-subsample,
  per-method fit-status table (why a fit was rejected or errored), at some
  extra bookkeeping cost.


In [ ]:
# The five estimators reported throughout this notebook (and the paper):
# Quadratic, Sigmoid, Isotonic, Isotonic (raw), IDR.
_DEFAULT_METHODS = (
    MethodConfig("Quadratic"),
    MethodConfig("Sigmoid"),
    MethodConfig("Isotonic"),
    MethodConfig("Isotonic (raw)"),
    MethodConfig("IDR"),
)


def _plot_methods(method_names):
    """Identity pass-through kept as the single choke point every plotting
    call site routes its method list through, so a future method-visibility
    toggle only needs to change this one function."""
    return list(method_names)

LIGHT_PRESET = StudyConfig(
    scenario=ScenarioConfig(
        seed=1,
        N=100_000,
        hazard=DistributionSpec("weibull_min", dict(c=2 / 3, loc=0.0, scale=1.0)),
        loss=DistributionSpec("lognorm", dict(s=2.5, loc=0.0, scale=1.0)),
        copula=CopulaSpec("gumbel", dict(theta=2.0)),
    ),
    run=RunConfig(
        n=500,
        iterations=100,
        bins=50,
        var_levels=(0.95, 0.99, 0.999, 0.9999),
        methods=_DEFAULT_METHODS,
        data_backend="memory",
        collect_diagnostics=True,
    ),
    label="light",
)

FULL_PRESET = StudyConfig(
    scenario=ScenarioConfig(
        seed=123,
        N=500_000,
        hazard=DistributionSpec("weibull_min", dict(c=2 / 3, loc=0.0, scale=1.0)),
        loss=DistributionSpec("pareto", dict(b=2.5, loc=0.0, scale=1.0)),
        copula=CopulaSpec("lorenz", dict(), mirror=True),
    ),
    run=RunConfig(
        n=500,
        iterations=100_000,
        bins=10,
        var_levels=(0.95, 0.99, 0.999, 0.9999),
        methods=_DEFAULT_METHODS,
        data_backend="hdf5",
        collect_diagnostics=True,
    ),
    label="full",
)

# The one variable every "Run" section in Part 10 reads by default.
ACTIVE = LIGHT_PRESET


### Building your own scenario

Presets are just `StudyConfig` instances -- build a variant with
`dataclasses.replace` instead of writing one from scratch. For example, to
keep everything about `LIGHT_PRESET` except swap the loss margin to Dagum:

```python
from dataclasses import replace

my_config = replace(
    LIGHT_PRESET,
    scenario=replace(LIGHT_PRESET.scenario, loss=DistributionSpec("dagum", dict(alpha=2.0, p=1.0))),
)
```

Part 10a does exactly this. Any field on `ScenarioConfig`/`RunConfig` can be
overridden the same way -- copula, methods, `var_levels`, `iterations`, and
so on.


## Part 3 — Plot style

Shared `mpl.rcParams`, color palette, and small legend/label helpers used by
every figure in this notebook, defined once here.

In [ ]:
mpl.rcParams.update({
    "figure.facecolor": "white",
    "figure.dpi": 130,
    "savefig.dpi": 300,
    "savefig.facecolor": "white",
    "savefig.bbox": "tight",
    "axes.facecolor": "white",
    "font.family": "DejaVu Serif",
    "mathtext.fontset": "stix",
    "text.usetex": False,
    "font.size": 12,
    "axes.titlesize": 12,
    "axes.labelsize": 17,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.edgecolor": "black",
    "axes.linewidth": 0.9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 4.0,
    "ytick.major.size": 4.0,
    "legend.frameon": False,
    "legend.fontsize": 11.5,
    "lines.linewidth": 2.0,
})

# One color identity for every estimator this notebook registers.
METHOD_COLORS = {
    "Quadratic": "#2e92ff",
    "Sigmoid": "#ff6b5f",
    "Isotonic": "#45c3a3",
    "Isotonic (raw)": "#f0a202",
    "IDR": "#7b4dff",
    "Empirical subsample": "#777777",
    "Truth": "#111111",
}
REFERENCE_COLORS = {
    "Hazard": METHOD_COLORS["Quadratic"],
    "Loss": METHOD_COLORS["Sigmoid"],
    "Empirical": METHOD_COLORS["Isotonic"],
    "Truth": METHOD_COLORS["Truth"],
}
mpl.rcParams["axes.prop_cycle"] = mpl.cycler(
    color=[
        METHOD_COLORS["Quadratic"],
        METHOD_COLORS["Sigmoid"],
        METHOD_COLORS["Isotonic"],
        METHOD_COLORS["IDR"],
    ]
)


def _q_to_pct_labels(levels):
    return [rf"${q * 100:g}\%$" for q in np.asarray(levels, dtype=float)]


def _palette_for(method_names):
    default = mpl.rcParams["axes.prop_cycle"].by_key().get("color", [])
    return {
        name: METHOD_COLORS.get(name, default[i % len(default)] if default else None)
        for i, name in enumerate(method_names)
    }


# Global figure-saving switch. Off by default -- flip to True to write every
# figure this notebook produces to PLOTS_DIR (created on first use).
SAVE_FIGURES = False


def save_figure(fig, name: str) -> None:
    """Write `fig` to PLOTS_DIR/{name}.png, if SAVE_FIGURES is on."""
    if not SAVE_FIGURES:
        return
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(PLOTS_DIR / f"{name}.png")

## Part 4 — Distributions

A common margin interface (`cdf`/`ppf`/`mean`/`rvs`, plus `sf`/`isf`/
`expected_shortfall`) used identically for the hazard slot and the loss slot,
backed by one registry (`DISTRIBUTION_REGISTRY`). SciPy frozen distributions
already provide most of this; `DistributionAdapter` fills in whatever is
missing (numerically, via the same logit-Gauss-Legendre quadrature used
throughout this notebook's tail integrals) and lets a closed-form expected
shortfall be attached where one exists.

Two margins that are not in `scipy.stats` are implemented here in full:
`Dagum` (Burr Type III) and `GPD` (Generalised Pareto as a loss margin).
Both are written with loss margins in mind, but nothing in their interface or
in the registry ties them to the loss slot — either can be used as the hazard
margin too.

To add a new distribution: write a class exposing `cdf`/`ppf`/`mean`/`rvs`
(optionally `sf`/`isf`/closed-form `expected_shortfall`), wrap it in
`DistributionAdapter`, and call `register_distribution("name", factory)` --
that one call is the only change needed.

**Cell below:** the numerical building blocks the rest of the notebook's
tail integrals reuse, plus the registry machinery.

*Logit-Gauss-Legendre quadrature.* To integrate a function of $v\in(0,1)$
stably all the way to the endpoints, substitute $v=\operatorname{sigmoid}(t)=
\frac{1}{1+e^{-t}}$ for $t\in[-T,T]$, so that
$$\int_0^1 f(v)\,dv=\int_{-T}^{T} f\big(\operatorname{sigmoid}(t)\big)\,v(1-v)\,dt,
\qquad v(1-v)=\frac{dv}{dt},$$
evaluated by $n$-point Gauss–Legendre quadrature on $[-T,T]$. This is the
same substitution used by `ConditionalTargets` (Part 6), just with different
$(T,n)$.

*Generic expected shortfall.* `DistributionAdapter.expected_shortfall(q)`
falls back, when no closed form is registered, to
$$\mathrm{ES}_q(X)=\frac{1}{1-q}\int_q^1 \mathrm{VaR}_u(X)\,du
=\int_0^1 F_X^{-1}\!\big(q+(1-q)v\big)\,dv,$$
evaluated with the same quadrature.


In [ ]:
# ---------------------------------------------------------------------------
# Stable quadrature on (0, 1), reused by the distribution ES fallback, the
# theoretical conditional targets (Part 6), and the EVT tail integrals (Part 7).
# ---------------------------------------------------------------------------

def _sigmoid_v(t: np.ndarray) -> np.ndarray:
    out = np.empty_like(t, dtype=float)
    pos = t >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-t[pos]))
    et = np.exp(t[~pos])
    out[~pos] = et / (1.0 + et)
    return out


def _logit_gauss_integrate_01(f_of_v, n_quad: int = 256, T: float = 25.0) -> float:
    """integral_0^1 f(v) dv via v = sigmoid(t), t in [-T, T], Gauss-Legendre nodes."""
    x, w = np.polynomial.legendre.leggauss(n_quad)
    t = T * x
    v = _sigmoid_v(t)
    dv_dt = v * (1.0 - v)
    return float(T * np.sum(w * f_of_v(v) * dv_dt))


def _es_from_sorted_atoms(values_sorted: np.ndarray, probs: np.ndarray, levels) -> np.ndarray:
    """Coherent (superquantile) ES of a discrete law with sorted support.

    ES_q = (1/(1-q)) * integral_q^1 VaR_u du, computed *exactly* for a
    discrete law rather than approximated: the atom straddling the VaR_q
    threshold is weighted only by the fraction of its probability mass lying
    strictly above q, not its full probability. This matters because every
    closed-form or quadrature theoretical ES elsewhere in this notebook
    (`ES_lognormal`, `ES_pareto`, `dagum_var_es`, `gpd_var_es`,
    `DistributionAdapter`'s quadrature fallback) computes this same coherent
    definition -- using a cruder estimator here (e.g. a plain mean of every
    point at or above VaR_q, which over-counts the boundary atom) would
    benchmark the empirical estimate against a target it isn't actually
    trying to estimate, inflating the apparent bias at small sample sizes and
    high quantiles.
    """
    values_sorted = np.asarray(values_sorted, dtype=float)
    probs = np.asarray(probs, dtype=float)
    F = np.clip(np.cumsum(probs), 0.0, 1.0)
    levels_arr = np.atleast_1d(np.asarray(levels, dtype=float))
    out = np.full(levels_arr.shape, np.nan, dtype=float)
    for j, q in enumerate(levels_arr):
        denom = 1.0 - q
        if denom <= 0:
            continue
        k = int(np.searchsorted(F, q, side="left"))
        if k >= values_sorted.size:
            continue
        frac = max(float(F[k]) - q, 0.0)
        num = frac * values_sorted[k]
        if k + 1 < values_sorted.size:
            num += float(probs[k + 1:] @ values_sorted[k + 1:])
        out[j] = num / denom
    return out


DISTRIBUTION_REGISTRY: dict[str, Callable[..., Any]] = {}


def register_distribution(name: str, factory: Callable[..., Any], *, overwrite: bool = False) -> None:
    if not overwrite and name in DISTRIBUTION_REGISTRY:
        raise ValueError(f"Distribution '{name}' is already registered.")
    DISTRIBUTION_REGISTRY[name] = factory


def get_distribution(name: str, **params) -> Any:
    if name not in DISTRIBUTION_REGISTRY:
        raise KeyError(f"Unknown distribution '{name}'. Registered: {sorted(DISTRIBUTION_REGISTRY)}")
    return DISTRIBUTION_REGISTRY[name](**params)


class DistributionAdapter:
    """Wraps any cdf/ppf/mean/rvs object, filling in sf/isf/expected_shortfall.

    `sf`/`isf` fall back to `1 - cdf`/`ppf(1 - q)` when the wrapped object
    doesn't provide them. `expected_shortfall` uses `closed_form_es(q)` when
    given, else a numerical upper-tail integral of the quantile function,
    ES_q = integral_0^1 VaR(q + (1-q) v) dv.
    """

    def __init__(self, dist: Any, name: str = "", closed_form_es: Callable[..., Any] | None = None):
        self._dist = dist
        self.name = name
        self._closed_form_es = closed_form_es

    def __repr__(self):
        return f"DistributionAdapter({self.name}, {self._dist!r})"

    def cdf(self, x):
        return self._dist.cdf(x)

    def ppf(self, q):
        return self._dist.ppf(q)

    def mean(self):
        return self._dist.mean()

    def rvs(self, size=1, random_state=None):
        return self._dist.rvs(size=size, random_state=random_state)

    def sf(self, x):
        if hasattr(self._dist, "sf"):
            return self._dist.sf(x)
        return 1.0 - self.cdf(x)

    def isf(self, q):
        if hasattr(self._dist, "isf"):
            return self._dist.isf(q)
        return self.ppf(1.0 - np.asarray(q, dtype=float))

    def expected_shortfall(self, q):
        if self._closed_form_es is not None:
            return self._closed_form_es(q)
        q_arr = np.atleast_1d(np.asarray(q, dtype=float))
        out = np.empty_like(q_arr)
        for i, qi in enumerate(q_arr):
            out[i] = _logit_gauss_integrate_01(lambda v, qi=qi: self.ppf(qi + (1.0 - qi) * v))
        return out if out.size > 1 else float(out[0])


### Closed-form margins: Weibull, lognormal, Pareto

`ES_lognormal`/`ES_pareto` are closed-form upper-tail expected shortfalls,
attached to the `lognorm`/`pareto`
`DistributionAdapter`s as `closed_form_es` so `expected_shortfall(q)` is exact
rather than falling back to quadrature:
$$\mathrm{ES}_u^{\text{lognormal}}=\frac{e^{\mu+\sigma^2/2}\,\Phi(\sigma-z_u)}{1-u},
\qquad z_u=\Phi^{-1}(u),$$
$$\mathrm{ES}_u^{\text{Pareto}}=\frac{b}{b-1}\,\mathrm{VaR}_u,\qquad b>1
\ \text{(else the mean, and ES, is infinite)}.$$
`weibull_min`/`lognorm`/`pareto` are otherwise thin `DistributionAdapter`
wrappers around the matching frozen `scipy.stats` distribution.


In [ ]:
# ---------------------------------------------------------------------------
# SciPy-backed margins: Weibull (hazard), lognormal and Pareto (loss), each
# with a closed-form upper-tail expected shortfall.
# ---------------------------------------------------------------------------

def ES_lognormal(u, s, loc=0.0, scale=1.0):
    """Upper-tail ES for X ~ lognorm(s, loc, scale) at level(s) u in (0,1)."""
    u = np.asarray(u, dtype=float)
    z = stats.norm.ppf(u)
    mu = np.log(scale)
    es0 = np.exp(mu + 0.5 * s**2) * stats.norm.cdf(s - z) / (1.0 - u)
    return loc + es0


def ES_pareto(u, b, loc=0.0, scale=1.0):
    """Upper-tail ES for X ~ pareto(b, loc, scale) at level(s) u in (0,1). Valid for b > 1."""
    u = np.asarray(u, dtype=float)
    var_u = stats.pareto.ppf(u, b, loc=0, scale=scale)
    es_u = (b * var_u) / (b - 1)
    return loc + es_u


def _weibull_min_factory(**params):
    return DistributionAdapter(stats.weibull_min(**params), name="weibull_min")


def _lognorm_factory(**params):
    s = params["s"]
    loc = params.get("loc", 0.0)
    scale = params.get("scale", 1.0)
    return DistributionAdapter(
        stats.lognorm(**params),
        name="lognorm",
        closed_form_es=lambda q: ES_lognormal(q, s, loc=loc, scale=scale),
    )


def _pareto_factory(**params):
    b = params["b"]
    loc = params.get("loc", 0.0)
    scale = params.get("scale", 1.0)
    return DistributionAdapter(
        stats.pareto(**params),
        name="pareto",
        closed_form_es=lambda q: ES_pareto(q, b, loc=loc, scale=scale),
    )


register_distribution("weibull_min", _weibull_min_factory)
register_distribution("lognorm", _lognorm_factory)
register_distribution("pareto", _pareto_factory)


### Custom loss margins: Dagum and generalised Pareto

Two margins not in `scipy.stats`, implemented in full below. Both are
written with *loss* margins in mind, but
nothing in their interface (`cdf`/`sf`/`ppf`/`isf`/`mean`/`rvs`) or in the
registry restricts them to that slot.

**Dagum** (Burr Type III, unit scale):
$$F(x)=\Big(1+(x/b)^{-\alpha}\Big)^{-p},\qquad x>0.$$
The tail is regularly varying with index $\alpha$ (finite mean iff
$\alpha>1$); $p$ shapes the body independently of the tail, so two margins
with the same $\alpha$ and different $p$ share the same asymptotic tail but
look very different at $n=500$ — useful for separating "tail heaviness" from
"body shape" in the sensitivity sweep (Part 10c). VaR/ES are closed-form via
the regularized incomplete beta function $B(z;a,b)$.

**Generalised Pareto as a loss (or hazard) margin**, parametrised by tail
index $\alpha=1/\xi$ (threshold-stable):
$$S(x)=\Big(1+\xi\,x/\sigma\Big)^{-1/\xi},\qquad \xi=1/\alpha.$$
Both classes evaluate their far tail via `.isf(s)` (in terms of the survival
probability) rather than `.ppf(1-s)`, for the same precision reason as
`ConditionalTargets` (Part 6).


In [ ]:
# ---------------------------------------------------------------------------
# Dagum (Burr Type III) margin.
#
# F(x) = (1 + (x/b)^(-alpha))^(-p), x > 0, unit scale by default. The upper tail
# is regularly varying with index alpha (S(x) ~ p x^(-alpha)); the mean exists
# exactly when alpha > 1. p shapes the body: two losses with the same alpha and
# different p share the same asymptotic tail but look very different in a
# finite sample -- useful for separating "tail heaviness" from "body shape"
# effects in the sensitivity sweep (Part 10c).
# ---------------------------------------------------------------------------

class Dagum:
    """Dagum type I with unit scale; scipy-frozen-distribution-like interface."""

    def __init__(self, alpha, p, b=1.0):
        if alpha <= 0 or p <= 0 or b <= 0:
            raise ValueError("alpha, p and b must be positive")
        self.alpha, self.p, self.b = float(alpha), float(p), float(b)

    def __repr__(self):
        return f"Dagum(alpha={self.alpha:g}, p={self.p:g}, b={self.b:g})"

    def _zpow(self, x):
        x = np.asarray(x, float)
        return np.power(np.where(x > 0, x, np.inf) / self.b, -self.alpha)

    def cdf(self, x):
        out = np.exp(-self.p * np.log1p(self._zpow(x)))
        return np.where(np.asarray(x, float) > 0, out, 0.0)

    def sf(self, x):
        return -np.expm1(-self.p * np.log1p(self._zpow(x)))

    def ppf(self, q):
        q = np.asarray(q, float)
        return self.b * np.power(np.expm1(-np.log(q) / self.p), -1.0 / self.alpha)

    def isf(self, s):
        """Upper quantile from the survival probability -- full precision in the far tail."""
        s = np.asarray(s, float)
        return self.b * np.power(np.expm1(-np.log1p(-s) / self.p), -1.0 / self.alpha)

    def mean(self):
        if self.alpha <= 1.0:
            return np.inf
        a, p = self.alpha, self.p
        return self.b * np.exp(gammaln(p + 1 / a) + gammaln(1 - 1 / a) - gammaln(p))

    def rvs(self, size=1, random_state=None):
        rng = random_state if hasattr(random_state, "uniform") else np.random.default_rng(random_state)
        return self.ppf(rng.uniform(size=size))


def dagum_var_es(alpha, p, levels, b=1.0):
    """Closed-form VaR and ES via the incomplete beta function.

    ES_q = b p B(1 - q^(1/p); 1 - 1/alpha, p + 1/alpha) / (1 - q). Letting
    q -> 0 returns the mean, which is the check run in Part 10's population
    validation.
    """
    levels = np.asarray(levels, float)
    var = b * np.power(np.expm1(-np.log(levels) / p), -1.0 / alpha)
    if alpha <= 1.0:
        return var, np.full_like(var, np.inf)
    A, B = 1.0 - 1.0 / alpha, p + 1.0 / alpha
    z = -np.expm1(np.log(levels) / p)  # 1 - q^(1/p)
    es = b * p * betainc(A, B, z) * np.exp(betaln(A, B)) / (1.0 - levels)
    return var, es


def _dagum_factory(**params):
    alpha, p, b = params["alpha"], params["p"], params.get("b", 1.0)
    return DistributionAdapter(
        Dagum(alpha, p, b), name="dagum",
        closed_form_es=lambda q: dagum_var_es(alpha, p, q, b)[1],
    )


register_distribution("dagum", _dagum_factory)


# ---------------------------------------------------------------------------
# Generalised Pareto as a loss margin.
# Parametrised by tail index alpha = 1/xi (threshold-stable, POT-friendly by
# construction) rather than shape xi directly.
# ---------------------------------------------------------------------------

class GPD:
    """Generalised Pareto loss margin, parametrised by the tail index."""

    def __init__(self, alpha, scale=1.0):
        if alpha <= 0 or scale <= 0:
            raise ValueError("alpha and scale must be positive")
        self.alpha, self.scale = float(alpha), float(scale)
        self.xi = 1.0 / self.alpha
        self.sigma = self.scale

    def __repr__(self):
        return f"GPD(alpha={self.alpha:g}, xi={self.xi:g}, scale={self.scale:g})"

    def cdf(self, x):
        x = np.asarray(x, float)
        return np.where(x > 0, -np.expm1(-self.alpha * np.log1p(
            self.xi * np.where(x > 0, x, 0.0) / self.sigma)), 0.0)

    def sf(self, x):
        x = np.asarray(x, float)
        return np.where(x > 0, np.exp(-self.alpha * np.log1p(
            self.xi * np.where(x > 0, x, 0.0) / self.sigma)), 1.0)

    def ppf(self, q):
        q = np.asarray(q, float)
        return self.sigma / self.xi * np.expm1(-self.xi * np.log1p(-q))

    def isf(self, s):
        """Upper quantile from the survival probability -- exact in the far tail."""
        s = np.asarray(s, float)
        return self.sigma / self.xi * np.expm1(-self.xi * np.log(s))

    def mean(self):
        return np.inf if self.alpha <= 1.0 else self.sigma / (1.0 - self.xi)

    def rvs(self, size=1, random_state=None):
        rng = random_state if hasattr(random_state, "uniform") else np.random.default_rng(random_state)
        return self.isf(rng.uniform(size=size))


def gpd_var_es(alpha, scale, levels):
    """Closed-form VaR and ES. Letting q -> 0 returns scale / (1 - xi), the mean."""
    levels = np.asarray(levels, float)
    xi = 1.0 / alpha
    var = scale / xi * np.expm1(-xi * np.log1p(-levels))
    if alpha <= 1.0:
        return var, np.full_like(var, np.inf)
    return var, (var + scale) / (1.0 - xi)


def _gpd_factory(**params):
    alpha, scale = params["alpha"], params.get("scale", 1.0)
    return DistributionAdapter(
        GPD(alpha, scale), name="gpd",
        closed_form_es=lambda q: gpd_var_es(alpha, scale, q)[1],
    )


register_distribution("gpd", _gpd_factory)

print("Registered distributions:", sorted(DISTRIBUTION_REGISTRY))


## Part 5 — Copula

Extreme-value copulas, `C(u,v) = exp(r A(t))` with `r = log(u) + log(v)`,
`t = log(u)/r`, specified by a Pickands dependence function `A`. The
`Pickands` class bundles `A` with its first two analytic derivatives and a
`mirror` flag; `COPULA_REGISTRY` maps a name to a builder of `(A, Ap, App)`,
and `make_pickands` assembles the `Pickands` object a `CopulaSpec` asks for.

Sampling and the copula density are computed in log-safe form: points are
carried as `(u, z)` with `v = expit(z)`, so `1 - v = expit(-z)` keeps full
double precision on **both** tails of the loss margin -- losses are drawn via
`.isf(1-v)`, never `.ppf(v)` on a value clipped to `1 - 1e-16`. This numerical
convention matters most for the tail-heaviness sensitivity sweep (Part 10c).

New copula families are added the same way as distributions: write `A`, `Ap`,
`App` and call `register_copula`.

**Cell below:** the two built-in copula families, expressed through their
Pickands dependence function $A(t)$, and the machinery that turns a name into
a usable `Pickands` object.

An extreme-value copula is $C(u,v)=\exp\!\big(rA(t)\big)$ with
$r=\log u+\log v$ and $t=\log u / r$. Two families are registered:

- **Gumbel:** $A(t)=\big(t^\theta+(1-t)^\theta\big)^{1/\theta}$, dependence
  parameter $\theta\ge1$ ($\theta=1$ is independence, $\theta\to\infty$ is
  comonotonicity).
- **Lorenz** (power-law, Cirillo & Fontanari 2025, $\beta=3$):
  $A(t)=1+t-K^{-1}(t)$ where $K(x)=\tfrac12(x^3+x)$; $K^{-1}$ is the real
  Cardano root, so $A'$ and $A''$ are exact closed forms rather than finite
  differences.

`Pickands.mirror=True` uses $A(1-t)$ instead of $A(t)$ — irrelevant for the
symmetric Gumbel family, but how the asymmetric Lorenz copula is oriented
throughout this notebook (`register_copula("lorenz", ..., default_mirror=True)`).
Since $\frac{d}{dt}A(1-t)=-A'(1-t)$ but $\frac{d^2}{dt^2}A(1-t)=A''(1-t)$ (the
double negative from the chain rule cancels for the second derivative),
`Ap` negates under mirroring and `App` does **not** — this asymmetry is a
common transcription trap and is checked explicitly in this notebook's audit.


In [ ]:
# ---------------------------------------------------------------------------
# Pickands dependence functions: Gumbel, and the power-law "Lorenz" copula
# (Cirillo and Fontanari 2025, Eq. 18, beta = 3). K^{-1} is the real Cardano
# root of K(x) = (x^3 + x)/2, so its derivatives are exact.
# ---------------------------------------------------------------------------

def gumbel_A(t, theta):
    t = np.clip(np.asarray(t, float), 0.0, 1.0)
    return (t**theta + (1.0 - t) ** theta) ** (1.0 / theta)


def gumbel_Ap(t, theta):
    t = np.clip(np.asarray(t, float), 0.0, 1.0)
    S = t**theta + (1.0 - t) ** theta
    return S ** (1.0 / theta - 1.0) * (t ** (theta - 1.0) - (1.0 - t) ** (theta - 1.0))


def gumbel_App(t, theta):
    t = np.clip(np.asarray(t, float), 1e-300, 1.0 - 1e-16)
    S = t**theta + (1.0 - t) ** theta
    return (theta - 1.0) * S ** (1.0 / theta - 2.0) * (t * (1.0 - t)) ** (theta - 2.0)


def _lorenz3_root(t):
    t = np.clip(np.asarray(t, float), 0.0, 1.0)
    s = np.sqrt(t * t + 1.0 / 27.0)
    return np.cbrt(t + s) + np.cbrt(t - s)


def lorenz3_A(t):
    return 1.0 + np.clip(np.asarray(t, float), 0.0, 1.0) - _lorenz3_root(t)


def lorenz3_Ap(t):
    x = _lorenz3_root(t)
    return 1.0 - 2.0 / (3.0 * x * x + 1.0)


def lorenz3_App(t):
    x = _lorenz3_root(t)
    return 24.0 * x / (3.0 * x * x + 1.0) ** 3


class Pickands:
    """Pickands dependence function with its first two derivatives.

    `mirror=True` uses A(1-t); irrelevant for the symmetric Gumbel copula,
    and how the asymmetric Lorenz copula is oriented in this notebook.
    """

    def __init__(self, A, Ap, App, mirror=False, name=""):
        self._A, self._Ap, self._App, self.mirror, self.name = A, Ap, App, mirror, name

    def A(self, t):
        return self._A(1.0 - t) if self.mirror else self._A(t)

    def Ap(self, t):
        return -self._Ap(1.0 - t) if self.mirror else self._Ap(t)

    def App(self, t):
        return self._App(1.0 - t) if self.mirror else self._App(t)

    def upper_tail_dependence(self) -> float:
        return float(2.0 - 2.0 * self.A(0.5))


COPULA_REGISTRY: dict[str, dict[str, Any]] = {}


def register_copula(name: str, builder: Callable[..., tuple], *, default_mirror: bool = False, overwrite: bool = False) -> None:
    name = name.lower()
    if not overwrite and name in COPULA_REGISTRY:
        raise ValueError(f"Copula '{name}' is already registered.")
    COPULA_REGISTRY[name] = {"builder": builder, "default_mirror": default_mirror}


def make_pickands(name: str, mirror: bool | None = None, **params) -> Pickands:
    name = name.lower()
    if name not in COPULA_REGISTRY:
        raise KeyError(f"Unknown copula '{name}'. Registered: {sorted(COPULA_REGISTRY)}")
    entry = COPULA_REGISTRY[name]
    A, Ap, App = entry["builder"](**params)
    use_mirror = entry["default_mirror"] if mirror is None else bool(mirror)
    return Pickands(A, Ap, App, mirror=use_mirror, name=f"{name}({params})")


def build_pickands(spec: "CopulaSpec") -> Pickands:
    """Build a Pickands object from a CopulaSpec (Part 2)."""
    return make_pickands(spec.name, mirror=spec.mirror, **spec.params)


def _gumbel_builder(theta: float = 2.0):
    return (lambda t: gumbel_A(t, theta), lambda t: gumbel_Ap(t, theta), lambda t: gumbel_App(t, theta))


def _lorenz_builder():
    return (lorenz3_A, lorenz3_Ap, lorenz3_App)


register_copula("gumbel", _gumbel_builder, default_mirror=False)
register_copula("lorenz", _lorenz_builder, default_mirror=True)

print("Registered copulas:", sorted(COPULA_REGISTRY))


### Log-safe copula quantities and exact sampling

Points are carried as $(u,z)$ with $v=\operatorname{expit}(z)$ instead of as
$(u,v)$ directly, so that $1-v=\operatorname{expit}(-z)$ stays exact even when
$v$ itself would round to $1.0$ in float64 — this is what lets the wide
quadrature domain in Part 6 work at all.

With $r=\log u+\log v$, $t=\log u/r$, $B_u=A(t)+(1-t)A'(t)$,
$B_v=A(t)-tA'(t)$:
$$\frac{\partial C}{\partial u}=P(V\le v\mid U=u)=e^{rA(t)-\log u}\,B_u,\qquad
\frac{\partial C}{\partial v}=P(U\le u\mid V=v)=e^{rA(t)-\log v}\,B_v,$$
$$c(u,v)=\frac{\partial^2 C}{\partial u\,\partial v}
=\frac{C}{uv}\left(B_uB_v-\frac{t(1-t)A''(t)}{r}\right).$$
`sample_ev_copula` draws $U\sim\mathrm{Unif}(0,1)$ and an independent
$W\sim\mathrm{Unif}(0,1)$, then solves $\partial C/\partial u(U,V)=W$ for $V$
by bisection on $z$ — exact conditional-inversion sampling, monotone and
derivative-free (never a Newton solve that could diverge).


In [ ]:
# ---------------------------------------------------------------------------
# Copula quantities in log-safe form. Points are carried as (u, z) with
# v = expit(z); 1 - v is recovered as expit(-z), never as 1 - v computed
# directly on a value near 1.
# ---------------------------------------------------------------------------

def _logv(z):
    return -np.logaddexp(0.0, -np.asarray(z, float))  # log(expit(z))


def _log1mv(z):
    return -np.logaddexp(0.0, np.asarray(z, float))  # log(1 - expit(z))


def _rt(u, z):
    lu = np.log(u)
    lv = _logv(z)
    r = lu + lv
    return lu, lv, r, lu / r


def cond_cdf_v_given_u(u, z, pk: Pickands):
    """P(V <= v | U = u) = dC/du, evaluated at v = expit(z)."""
    lu, lv, r, t = _rt(u, z)
    Bu = pk.A(t) + (1.0 - t) * pk.Ap(t)
    return np.exp(r * pk.A(t) - lu) * np.clip(Bu, 0.0, None)


def cond_cdf_u_given_v(u, z, pk: Pickands):
    """P(U <= u | V = v) = dC/dv."""
    lu, lv, r, t = _rt(u, z)
    Bv = pk.A(t) - t * pk.Ap(t)
    return np.exp(r * pk.A(t) - lv) * np.clip(Bv, 0.0, None)


def copula_density(u, z, pk: Pickands):
    lu, lv, r, t = _rt(u, z)
    Bu = pk.A(t) + (1.0 - t) * pk.Ap(t)
    Bv = pk.A(t) - t * pk.Ap(t)
    dens = Bu * Bv - t * (1.0 - t) * pk.App(t) / r
    return np.exp(r * pk.A(t) - lu - lv) * np.clip(dens, 0.0, None)


def sample_ev_copula(n: int, pk: Pickands, rng, n_bisect: int = 90, z_lo: float = -745.0, z_hi: float = 745.0):
    """Exact conditional-inversion sampling, vectorised.

    Returns (u, z) with v = expit(z). dC/du is increasing in v, so a plain
    bisection on z is monotone, derivative-free, and cannot diverge.
    """
    u = rng.uniform(size=n)
    w = rng.uniform(size=n)
    lo = np.full(n, z_lo)
    hi = np.full(n, z_hi)
    for _ in range(n_bisect):
        mid = 0.5 * (lo + hi)
        take_hi = cond_cdf_v_given_u(u, mid, pk) >= w
        hi = np.where(take_hi, mid, hi)
        lo = np.where(take_hi, lo, mid)
    return u, 0.5 * (lo + hi)


## Part 6 — Theoretical conditional targets

`ConditionalTargets` computes the exact ground truth `d(h) = E[L|H=h]` and
`tail_mean(h) = E[L|H>=h]` by Gauss-Legendre quadrature on the logit scale,
built directly on the `Pickands` object from Part 5 -- so there is only one
copy of the Pickands/density formulas in the whole notebook, used everywhere
they're needed, with nothing to drift out of sync. Losses are evaluated via
`.isf(1 - v)`, keeping full precision deep in the tail.

**Cell below:** `ConditionalTargets`, which turns a `Pickands` object plus a
hazard/loss margin pair into the two *exact* ground-truth curves the whole
bootstrap study is benchmarked against:
$$d(h)=\mathbb E[L\mid H=h]=\int_0^1 Q_L(v)\,c(u,v)\,dv,\qquad u=F_H(h),$$
$$m(h)=\mathbb E[L\mid H\ge h]=\int_0^1 Q_L(v)\,\frac{1-\partial C/\partial v(u,v)}{1-u}\,dv,$$
where $c(u,v)=\partial^2C/\partial u\,\partial v$ is the copula density (Part 5)
and $Q_L=F_L^{-1}$ is the loss quantile function. Both integrals use the same
logit-scale quadrature as Part 4, but evaluated via `.isf(1-v)` rather than
`.ppf(v)` — the notebook's audit confirmed this matters: at wide quadrature
domains ($T=40$), `v` can round to exactly $1.0$ in float64, and `ppf(1.0)`
returns $\infty$/`NaN` where `isf` stays exact.


In [ ]:
class ConditionalTargets:
    """Exact (numerically integrated) conditional targets.

    d(h)         = E[L | H = h]  = integral_0^1 Q_L(v) c(u,v) dv,       u = F_H(h)
    tail_mean(h) = E[L | H >= h] = integral_0^1 Q_L(v) (1 - dC/dv)/(1-u) dv
    """

    def __init__(self, pk: Pickands, hazard_dist, loss_dist, n_nodes: int = 512, T: float = 40.0):
        self.pk, self.hazard_dist, self.loss_dist = pk, hazard_dist, loss_dist
        x, w = np.polynomial.legendre.leggauss(int(n_nodes))
        self.z = T * x
        self.w = T * w
        self.jac = np.exp(_logv(self.z) + _log1mv(self.z))  # dv/dz
        self.QL = loss_dist.isf(np.exp(_log1mv(self.z)))  # Q_L(v)
        finite = np.isfinite(self.QL)
        if not finite.all():
            self.QL = np.where(finite, self.QL, 0.0)

    def _integrate(self, u, weights_fn):
        u = np.atleast_1d(np.asarray(u, float))
        U = u[:, None]
        Z = self.z[None, :]
        integrand = self.QL[None, :] * weights_fn(U, Z) * self.jac[None, :]
        return np.nansum(integrand * self.w[None, :], axis=1)

    def d(self, h):
        """E[L | H = h]."""
        u = self.hazard_dist.cdf(np.asarray(h, float))
        out = self._integrate(u, lambda U, Z: copula_density(U, Z, self.pk))
        return out if np.ndim(h) else float(out[0])

    def tail_mean(self, h):
        """E[L | H >= h]."""
        u = np.atleast_1d(self.hazard_dist.cdf(np.asarray(h, float)))
        out = self._integrate(u, lambda U, Z: (1.0 - cond_cdf_u_given_v(U, Z, self.pk)) / (1.0 - U))
        return out if np.ndim(h) else float(out[0])


@dataclass
class BuiltScenario:
    """Everything derived from a ScenarioConfig: distributions, copula, exact targets."""

    hazard_dist: Any
    loss_dist: Any
    pickands: Pickands
    targets: ConditionalTargets


def build_scenario(scenario: "ScenarioConfig") -> BuiltScenario:
    hazard_dist = get_distribution(scenario.hazard.name, **scenario.hazard.params)
    loss_dist = get_distribution(scenario.loss.name, **scenario.loss.params)
    pickands = build_pickands(scenario.copula)
    targets = ConditionalTargets(pickands, hazard_dist, loss_dist)
    return BuiltScenario(hazard_dist=hazard_dist, loss_dist=loss_dist, pickands=pickands, targets=targets)


## Part 7 — Damage-function / methodology estimators

Every estimator below sees the same subsample of `n` pairs. `METHOD_REGISTRY`
is a declarative registry: each `MethodSpec` names a fit function, whether it
trains on the raw subsample or the probability-binned summary by default, and
whether it produces a mean curve or a full conditional distribution
(distributional methods can answer VaR/ES/mean queries directly; mean-curve
methods cannot). A `MethodConfig` in a `RunConfig` (Part 2) can override the
`train_on` default per run.

**7a-7c: classical mean-damage curves** (Quadratic, Sigmoid, Isotonic). The
quadratic fit has no linear fallback: a rejected significance test returns
`NaN` everywhere rather than silently downgrading to a different model, so
`"Quadratic"` means the same thing in every run this notebook produces.
Isotonic is also registered a second time as `"Isotonic (raw)"`, trained on
the unbinned pairs -- a control arm that isolates the cost of binning.

**7d: IDR**, the official `isodistrreg`-backed isotonic distributional
regression estimator -- the only method here that returns a full conditional
CDF rather than a mean curve.

**7e: EVT/POT tail models** -- the only estimators that can place probability
mass beyond the largest observed loss. The IDR-bulk variants reuse the
official IDR backend from 7d (via a small adapter) rather than a second,
separate implementation.

**7f: binning**, the equal-probability binning used by every "binned" method.

**Cell below:** the shared plumbing every damage-function estimator in Parts
7a-7e is registered through.

*Binning* (`bin_summary`) turns $n$ raw $(h,y)$ pairs into $B$ summary points
of (as close to) equal empirical probability $1/B$ each, using the empirical
quantile function $\hat F_H^{-1}(k/B)$ as bin edges; the bin's $y$-value is the
mean loss inside it, and its $x$-value is either the edge midpoint
(`bin_x="midpoint"`, the default used throughout this notebook) or the mean
hazard inside the bin (`bin_x="mean"`, an alternative convention) — the two
give materially different fits in the tail bin.

*`MethodSpec`* is a small dataclass: a name, a fit callable, whether it
defaults to `"raw"` or `"binned"` training, and which of
`{"pointwise","tail","marginal"}` tables it's meaningful in (`reports`) — e.g.
a method whose tail behavior doesn't depend on $h$ at all is marked
`reports={"marginal"}` so it isn't plotted as a (trivially flat)
pointwise/tail curve.


In [ ]:
# ---------------------------------------------------------------------------
# 7f. Equal-probability binning, used by every "binned" method. Bins are
# contiguous blocks after sorting by hazard, so each bin carries (as close to)
# equal empirical probability.
# ---------------------------------------------------------------------------

def probability_based_binning(data: np.ndarray, num_bins: int):
    if data.ndim != 2 or data.shape[1] != 2:
        raise ValueError("data must have shape (n, 2) with columns (x, y).")
    if num_bins < 1:
        raise ValueError("num_bins must be >= 1.")
    n = data.shape[0]
    if n == 0:
        raise ValueError("data is empty.")

    sorted_data = data[np.argsort(data[:, 0])]
    probs = np.linspace(0.0, 1.0, num_bins + 1)
    edges_x = np.quantile(sorted_data[:, 0], probs, method="inverted_cdf")

    bin_edges = np.linspace(0, n, num_bins + 1, dtype=int)
    binned_data = [sorted_data[bin_edges[i]:bin_edges[i + 1]] for i in range(num_bins)]
    return binned_data, edges_x


def bin_summary(data: np.ndarray, num_bins: int, summary_func=np.mean, bin_x: str = "midpoint"):
    """Return (x_summary, y_summary) per bin.

    `bin_x="midpoint"` (the default used throughout this notebook) uses the
    midpoint of the two empirical-quantile bin edges; `bin_x="mean"` uses the
    mean hazard inside the bin. The two give materially different fits in
    the tail bin, since a single extreme observation can move the midpoint
    convention's top design point but not the mean convention's.
    """
    binned_rows, edges_x = probability_based_binning(data, num_bins)
    ys = [b[:, 1] for b in binned_rows]
    y_summary = np.array([summary_func(y) if len(y) else np.nan for y in ys])
    if bin_x == "mean":
        x_summary = np.array([b[:, 0].mean() if len(b) else np.nan for b in binned_rows])
    else:
        x_summary = 0.5 * (edges_x[:-1] + edges_x[1:])
    return x_summary, y_summary


# ---------------------------------------------------------------------------
# Method registry.
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class MethodSpec:
    name: str
    kind: Literal["mean_curve", "distributional"]
    fit: Callable[..., Any]
    default_train_on: Literal["raw", "binned"]
    reports: frozenset = field(default_factory=lambda: frozenset({"pointwise", "tail", "marginal"}))
    fit_kwargs: dict = field(default_factory=dict)


METHOD_REGISTRY: dict[str, MethodSpec] = {}


def register_method(spec: MethodSpec, *, overwrite: bool = False) -> None:
    if not overwrite and spec.name in METHOD_REGISTRY:
        raise ValueError(f"Method '{spec.name}' is already registered.")
    METHOD_REGISTRY[spec.name] = spec


def build_methods_and_train_on(method_configs: Iterable["MethodConfig"]) -> tuple[dict[str, Any], dict[str, str], dict[str, MethodSpec]]:
    """Resolve a RunConfig's MethodConfig tuple into the (methods, train_on)
    dicts the evaluation engine (Part 8) consumes, plus the resolved specs."""
    methods: dict[str, Any] = {}
    train_on: dict[str, str] = {}
    specs: dict[str, MethodSpec] = {}
    for cfg in method_configs:
        if cfg.name not in METHOD_REGISTRY:
            raise KeyError(f"Unknown method '{cfg.name}'. Registered: {sorted(METHOD_REGISTRY)}")
        spec = METHOD_REGISTRY[cfg.name]
        methods[cfg.name] = spec.fit
        train_on[cfg.name] = cfg.train_on or spec.default_train_on
        specs[cfg.name] = spec
    return methods, train_on, specs


### 7a. Quadratic damage function, with a significance screen

Ordinary least squares of $y=ax^2+bx+c$ on the (binned, by default) sample,
**accepted only if** the quadratic coefficient is statistically distinguishable
from zero:
$$t=\frac{\hat a}{\mathrm{SE}(\hat a)},\qquad
\text{accept iff } p\text{-value}=2\,\big(1-\Phi_{t,\,\mathrm{dof}}(|t|)\big)<\alpha.$$
Only the quadratic coefficient $a$ is significance-tested -- $b$ and $c$ are
fit but never screened -- and a rejected fit returns `NaN` everywhere rather
than downgrading to a different model. This makes `"Quadratic"` mean the same
thing in every run this notebook produces.


In [ ]:
# ---------------------------------------------------------------------------
# 7a. Quadratic regression with a significance check on the quadratic term.
# Accepted iff the quadratic coefficient's two-sided t-test p-value < alpha;
# no significance test on the other coefficients, and -- deliberately -- no
# linear fallback.
# ---------------------------------------------------------------------------

def fit_quadratic_checked(x, y, alpha=0.05, intercept=True):
    x = np.asarray(x, dtype=float).ravel()
    y = np.asarray(y, dtype=float).ravel()
    if x.shape != y.shape:
        raise ValueError("x and y must have the same shape.")

    finite = np.isfinite(x) & np.isfinite(y)
    x_fit, y_fit = x[finite], y[finite]

    def nan_result(message):
        return (np.nan, np.nan, np.nan, np.full(x_fit.shape, np.nan, dtype=float), "error", message)

    n_parameters = 3 if intercept else 2
    if x_fit.size <= n_parameters:
        return nan_result("not enough finite data points for the quadratic fit.")

    def ols_beta_tstats(X, response):
        n, p = X.shape
        dof = n - p
        if dof <= 0 or np.linalg.matrix_rank(X) < p:
            return np.full(p, np.nan), np.full(p, np.nan), dof
        beta, *_ = np.linalg.lstsq(X, response, rcond=None)
        residuals = response - X @ beta
        residual_variance = float(residuals @ residuals) / dof
        try:
            xtx_inv = np.linalg.inv(X.T @ X)
        except np.linalg.LinAlgError:
            return beta, np.full(p, np.nan), dof
        variances = np.maximum(residual_variance * np.diag(xtx_inv), 0.0)
        standard_errors = np.sqrt(variances)
        t_statistics = np.full(p, np.nan, dtype=float)
        positive_se = standard_errors > 0
        t_statistics[positive_se] = beta[positive_se] / standard_errors[positive_se]
        zero_se_nonzero_beta = (standard_errors == 0) & (beta != 0)
        t_statistics[zero_se_nonzero_beta] = np.sign(beta[zero_se_nonzero_beta]) * np.inf
        return beta, t_statistics, dof

    def two_sided_p(t_statistic, dof):
        if not np.isfinite(t_statistic):
            return np.nan
        return float(2.0 * student_t.sf(np.abs(t_statistic), dof))

    if intercept:
        X_quadratic = np.column_stack((x_fit**2, x_fit, np.ones_like(x_fit)))
    else:
        X_quadratic = np.column_stack((x_fit**2, x_fit))

    beta, t_statistics, dof = ols_beta_tstats(X_quadratic, y_fit)
    if not np.all(np.isfinite(beta)):
        return nan_result("the quadratic design matrix is rank-deficient or has no residual degrees of freedom.")

    a, b = float(beta[0]), float(beta[1])
    c = float(beta[2]) if intercept else 0.0
    p_a = two_sided_p(t_statistics[0], dof)
    accepted = np.isfinite(p_a) and p_a < alpha

    if accepted:
        y_hat = a * x_fit**2 + b * x_fit + c
        return a, b, c, y_hat, "quadratic", ""
    return nan_result("the quadratic coefficient failed the significance criterion.")


def fit_damage_function_quadratic(hazard_data, loss_data, alpha=0.05, intercept=True):
    """Fit once and return a callable predictor h -> m_hat(h)."""
    a, b, c, _, _, _ = fit_quadratic_checked(hazard_data, loss_data, alpha=alpha, intercept=intercept)

    def predictor(h):
        h = np.asarray(h, dtype=float)
        return a * h**2 + b * h + c

    return predictor


def damage_function_quadratic(h, hazard_data, loss_data, alpha=0.05, intercept=True):
    return fit_damage_function_quadratic(hazard_data, loss_data, alpha=alpha, intercept=intercept)(h)


register_method(MethodSpec("Quadratic", kind="mean_curve", fit=damage_function_quadratic, default_train_on="binned"))


### 7b. Sigmoid damage function

A logistic curve fit to **min-max-standardised** losses, then rescaled back:
$$\hat y_{\text{std}}=\operatorname{expit}\big(k(x-x_0)\big)=\frac{1}{1+e^{-k(x-x_0)}},
\qquad \hat y = \hat y_{\text{std}}\cdot(\max y-\min y)+\min y.$$
Only the steepness $k$ is significance-screened (two-sided $t$-test at
$\alpha$) — screening the inflection point $x_0$ would test a null with no
real interpretation. Because the upper asymptote is fixed at the *largest
observed value in the training data*, this curve can never predict a loss
larger than the largest value it was shown, which is the main reason it
underperforms badly on higher VaR levels.


In [ ]:
# ---------------------------------------------------------------------------
# 7b. Sigmoidal regression on min-max-standardised losses, rescaled back.
# Only the steepness k is significance-screened.
# ---------------------------------------------------------------------------

def _sigmoid_unit_scale(x, k, x0):
    return expit(k * (np.asarray(x, dtype=float) - x0))


def _make_nan_predictor():
    def f(h):
        h = np.asarray(h, dtype=float)
        return np.full_like(h, np.nan, dtype=float)
    return f


def _make_scaled_sigmoid(k, x0, min_loss, max_loss, clip_std=True):
    scale = max_loss - min_loss
    if scale == 0:
        const = float(min_loss)

        def f(h):
            h = np.asarray(h, dtype=float)
            return np.full_like(h, const, dtype=float)
        return f

    def f(h):
        h = np.asarray(h, dtype=float)
        y = _sigmoid_unit_scale(h, k, x0)
        if clip_std:
            y = np.clip(y, 0.0, 1.0)
        return y * scale + min_loss
    return f


def fit_and_rescale_sigmoid(hazards, losses, p0=None, maxfev=10000, clip_std=True, alpha=0.05):
    hazards = np.asarray(hazards, dtype=float).reshape(-1)
    losses = np.asarray(losses, dtype=float).reshape(-1)
    if hazards.size != losses.size:
        return _make_nan_predictor(), np.full(2, np.nan, dtype=float)

    valid = np.isfinite(hazards) & np.isfinite(losses)
    hazards, losses = hazards[valid], losses[valid]
    nan_popt = np.full(2, np.nan, dtype=float)

    if hazards.size < 3 or np.unique(hazards).size < 2:
        return _make_nan_predictor(), nan_popt

    min_loss, max_loss = float(np.min(losses)), float(np.max(losses))
    scale = max_loss - min_loss
    if scale == 0:
        x0 = float(np.median(hazards))
        predictor = _make_scaled_sigmoid(k=0.0, x0=x0, min_loss=min_loss, max_loss=max_loss, clip_std=clip_std)
        return predictor, np.array([0.0, x0], dtype=float)

    std_losses = (losses - min_loss) / scale
    if p0 is None:
        p0 = (1.0, float(np.median(hazards)))

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", OptimizeWarning)
            popt, pcov, _, _, ier = curve_fit(
                _sigmoid_unit_scale, hazards, std_losses, p0=p0, maxfev=maxfev, full_output=True,
            )
        converged = ier in (1, 2, 3, 4)
        finite_parameters = np.all(np.isfinite(popt))
        if not converged or not finite_parameters:
            return _make_nan_predictor(), nan_popt

        fitted_std = _sigmoid_unit_scale(hazards, *popt)
        if not np.all(np.isfinite(fitted_std)):
            return _make_nan_predictor(), nan_popt

        dof = hazards.size - popt.size
        k_se = np.sqrt(pcov[0, 0]) if np.isfinite(pcov[0, 0]) else np.nan
        if dof <= 0 or not np.isfinite(k_se) or k_se <= 0:
            return _make_nan_predictor(), nan_popt

        k_t_statistic = popt[0] / k_se
        k_p_value = float(2.0 * student_t.sf(np.abs(k_t_statistic), dof))
        if not np.isfinite(k_p_value) or k_p_value >= alpha:
            return _make_nan_predictor(), nan_popt
    except (RuntimeError, ValueError, TypeError, FloatingPointError, OverflowError, np.linalg.LinAlgError):
        return _make_nan_predictor(), nan_popt

    predictor = _make_scaled_sigmoid(k=float(popt[0]), x0=float(popt[1]), min_loss=min_loss, max_loss=max_loss, clip_std=clip_std)
    return predictor, np.asarray(popt, dtype=float)


def fit_damage_function_sigmoid(hazard_data, loss_data, **fit_kwargs):
    predictor, _ = fit_and_rescale_sigmoid(hazard_data, loss_data, **fit_kwargs)
    return predictor


def damage_function_sigmoid(h, hazard_data, loss_data, **fit_kwargs):
    return fit_damage_function_sigmoid(hazard_data, loss_data, **fit_kwargs)(h)


register_method(MethodSpec("Sigmoid", kind="mean_curve", fit=damage_function_sigmoid, default_train_on="binned"))


### 7c. Isotonic regression (binned, and a raw control arm)

Monotone (non-decreasing) least-squares regression, fit via `sklearn`'s
pool-adjacent-violators implementation:
$$\hat m=\operatorname*{arg\,min}_{m \text{ non-decreasing}}\ \sum_i\big(y_i-m(x_i)\big)^2,$$
piecewise-linear between fitted knots, clipped (not extrapolated) outside the
training range. Registered **twice**: `"Isotonic"` (binned, matching the
other classical curves) and `"Isotonic (raw)"` (trained on the unbinned $n$
pairs) — a control arm that isolates the cost of binning: the gap between the
two is "what binning costs," and the further
gap to IDR (next cell) is "what modelling the whole conditional distribution
buys you beyond the conditional mean."


In [ ]:
# ---------------------------------------------------------------------------
# 7c. Isotonic regression. Registered twice: the default "Isotonic" (binned)
# and a "Isotonic (raw)" control arm trained on
# the unbinned pairs -- comparing the two isolates the cost of binning, and
# comparing "Isotonic (raw)" with IDR isolates the value of modelling the
# conditional distribution instead of the conditional mean.
# ---------------------------------------------------------------------------

def fit_damage_function_isotonic(x_train, y_train):
    model = IsotonicRegression(increasing=True, out_of_bounds="clip")
    model.fit(x_train, y_train)
    return model.predict


def damage_function_isotonic(h, x_train, y_train):
    return fit_damage_function_isotonic(x_train, y_train)(h)


register_method(MethodSpec("Isotonic", kind="mean_curve", fit=damage_function_isotonic, default_train_on="binned"))
register_method(MethodSpec("Isotonic (raw)", kind="mean_curve", fit=damage_function_isotonic, default_train_on="raw"))


### 7d. Isotonic Distributional Regression (IDR), official backend

Unlike every other estimator here, IDR (Henzi, Ziegel & Gneiting, 2021)
targets the **whole conditional distribution** $F_{L\mid H=h}$, not just its
mean — obtained by isotonic-regressing the indicator $\mathbf 1\{L\le z\}$
against $H$, independently for every candidate loss threshold $z$, under a
monotonicity-in-$H$ constraint. This wraps the official `isodistrreg` Python
bindings rather than re-deriving the algorithm.

Exceedance and marginal distributions are **mixtures of conditional CDFs**,
never averages of conditional quantiles:
$$F_{L\mid H\ge h_0}(z)\approx\frac{1}{|\{i:H_i\ge h_0\}|}\sum_{i:H_i\ge h_0}F_{L\mid H=H_i}(z),
\qquad
F_L(z)\approx\frac1n\sum_{i=1}^n F_{L\mid H=H_i}(z).$$
Given a discrete law with sorted atoms $z_1<z_2<\dots$, probabilities $p_j$ and
CDF $F_j=\sum_{i\le j}p_i$, let $k=\min\{j:F_j\ge q\}$, so that $z_k=\mathrm{VaR}_q$.
The mean and the coherent (superquantile) expected shortfall are
$$\mathbb E[L]=\sum_j p_j z_j,\qquad
\mathrm{ES}_q=\frac{1}{1-q}\int_q^1\mathrm{VaR}_u\,du
=\frac{1}{1-q}\Big[(F_k-q)\,z_k+\sum_{j>k}p_j z_j\Big].$$
The atom at $\mathrm{VaR}_q$ therefore contributes only the part of its mass lying
above level $q$, namely $F_k-q$, not its full mass $p_k$. This is the exact
computation in `_es_from_sorted_atoms` (Part 4) and `IDRDiscreteDistribution.expected_shortfall()`
below, and the same definition used by every theoretical ES in this notebook.
A plain tail average $\sum_{j\ge k}p_jz_j\big/\sum_{j\ge k}p_j$, which counts the
boundary atom in full, would give a different functional whenever $F_{k-1}<q<F_k$.
**A floating-point subtlety in `IDRDiscreteDistribution.quantile()`/`.expected_shortfall()`:**
in-sample, the marginal mixture $\hat F_{\text{IDR}}=\frac1n\sum_i \hat F(\cdot\mid H=H_i)$ is, by
isotonic regression's own mass-balance property, mathematically forced to equal the plain empirical
CDF of the training losses — so IDR's marginal VaR/ES can never *genuinely* differ from the empirical
baseline in-sample. At quantile levels $q$ where $q\cdot n$ is an exact integer (e.g. $q=0.95$ or
$q=0.99$ at $n=500$), the empirical CDF hits $q$ *exactly* on one atom, with zero margin on either
side; IDR's fitted CDF at that same atom is only accurate to floating-point precision ($\sim10^{-8}$),
so it can round a hair below $q$ instead of landing on it exactly, silently pushing the discrete
search onto the *next* atom up — a full order-statistic jump caused by a $10^{-8}$ rounding error.
`_IDR_QUANTILE_TIE_EPS` (defined just below) absorbs exactly that noise: it is four to five orders of
magnitude smaller than the real $1/n$ gap between atoms, so it can never merge two genuinely different
quantile rungs, only a tie with its own numerical noise. This only ever affects the raw VaR/ES
*value* at those tie levels — `.mean()`/`.variance()`, and everything built on them (the pointwise and
tail conditional-expectation tables), never perform this kind of boundary search and are unaffected.


In [ ]:
# ---------------------------------------------------------------------------
# 7d. Official-IDR-backed univariate damage/risk estimation, wrapping the
# official Python bindings from `isodistrreg`. Exposes
# E[L|H=h], E[L|H>=h], VaR_q(L), ES_q(L) via the correct distributional
# aggregation: the exceedance/marginal distributions are empirical mixtures of
# the fitted conditional CDFs, never averages of conditional VaRs.
# ---------------------------------------------------------------------------

def _import_official_idr():
    if _OFFICIAL_IDR is None:
        raise ImportError(
            "The official IDR backend is missing. Install the Python bindings from "
            "the official AlexanderHenzi/isodistrreg repository, then retry. "
            "Expected import: 'from isodistrreg import IDR'."
        )
    return _OFFICIAL_IDR


def _finite_training_arrays(hazard_data, loss_data):
    h = np.asarray(hazard_data, dtype=float).reshape(-1)
    l = np.asarray(loss_data, dtype=float).reshape(-1)
    if h.shape != l.shape:
        raise ValueError("hazard_data and loss_data must have the same shape.")
    ok = np.isfinite(h) & np.isfinite(l)
    h, l = h[ok], l[ok]
    if h.size == 0:
        raise ValueError("No finite hazard/loss observations.")
    return h, l


def _finite_source_array(hazard_data):
    h = np.asarray(hazard_data, dtype=float).reshape(-1)
    return h[np.isfinite(h)]


# Tolerance for the discrete quantile search in quantile()/expected_shortfall()
# below. IDR's fitted CDF equals the plain empirical CDF only up to floating-
# point solver noise (~1e-8); at a level q where q*n is an exact integer (e.g.
# q=0.95, q=0.99 at n=500), the empirical CDF lands exactly on q with zero
# margin, so that noise can push the fitted value a hair below q and bump the
# search to the next atom up. EPS is ~1e4-1e5x smaller than the real 1/n gap
# between atoms, so it only ever absorbs this noise -- it can't merge two
# genuinely different quantile rungs. See the markdown note above.
_IDR_QUANTILE_TIE_EPS = 1e-6


@dataclass
class IDRDiscreteDistribution:
    """Discrete distribution represented by values and CDF rows."""

    thresholds: np.ndarray
    cdfs: np.ndarray
    close_at_max: bool = True

    def __post_init__(self) -> None:
        thresholds = np.asarray(self.thresholds, dtype=float).reshape(-1)
        cdfs = np.asarray(self.cdfs, dtype=float)
        if thresholds.size == 0:
            raise ValueError("thresholds must be non-empty.")
        if cdfs.ndim == 1:
            cdfs = cdfs.reshape(1, -1)
        if cdfs.ndim != 2 or cdfs.shape[1] != thresholds.size:
            raise ValueError(f"cdfs must have shape (n_queries, n_thresholds); got {cdfs.shape} for {thresholds.size} thresholds.")

        order = np.argsort(thresholds, kind="mergesort")
        thresholds = thresholds[order]
        cdfs = cdfs[:, order]
        clean = np.clip(cdfs, 0.0, 1.0)
        finite_rows = np.all(np.isfinite(clean), axis=1)
        if np.any(finite_rows):
            clean[finite_rows] = np.maximum.accumulate(clean[finite_rows], axis=1)
            if self.close_at_max:
                clean[finite_rows, -1] = 1.0
        self.thresholds = thresholds
        self.cdfs = clean

    def _complete_rows(self) -> np.ndarray:
        return np.all(np.isfinite(self.cdfs), axis=1) & np.isclose(self.cdfs[:, -1], 1.0, rtol=1e-10, atol=1e-10)

    def probabilities(self) -> np.ndarray:
        masses = np.full_like(self.cdfs, np.nan, dtype=float)
        finite_rows = np.all(np.isfinite(self.cdfs), axis=1)
        if not np.any(finite_rows):
            return masses
        raw = np.diff(np.column_stack([np.zeros(np.count_nonzero(finite_rows), dtype=float), self.cdfs[finite_rows]]), axis=1)
        masses[finite_rows] = np.maximum(raw, 0.0)
        return masses

    def mean(self) -> np.ndarray:
        masses = self.probabilities()
        out = np.full(masses.shape[0], np.nan, dtype=float)
        valid = self._complete_rows()
        if np.any(valid):
            out[valid] = np.sum(masses[valid] * self.thresholds[None, :], axis=1)
        return out

    def variance(self) -> np.ndarray:
        masses = self.probabilities()
        mean = self.mean()
        out = np.full(masses.shape[0], np.nan, dtype=float)
        valid = self._complete_rows() & np.isfinite(mean)
        if np.any(valid):
            second = np.sum(masses[valid] * (self.thresholds[None, :] ** 2), axis=1)
            out[valid] = np.maximum(second - mean[valid] ** 2, 0.0)
        return out

    def quantile(self, q):
        q_arr = np.asarray(q, dtype=float)
        was_scalar = q_arr.ndim == 0
        q_flat = q_arr.reshape(-1)
        out = np.full((self.cdfs.shape[0], q_flat.size), np.nan, dtype=float)
        for i, row in enumerate(self.cdfs):
            if not np.all(np.isfinite(row)):
                continue
            for j, q_val in enumerate(q_flat):
                if not np.isfinite(q_val):
                    continue
                q_clip = float(np.clip(q_val, 0.0, 1.0))
                if q_clip > row[-1] and not np.isclose(row[-1], 1.0, rtol=1e-10, atol=1e-10):
                    continue
                idx = int(np.searchsorted(row, q_clip - _IDR_QUANTILE_TIE_EPS, side="left"))  # tie-break tolerance, see note above
                idx = min(idx, self.thresholds.size - 1)
                out[i, j] = self.thresholds[idx]
        return out[:, 0] if was_scalar else out

    def value_at_risk(self, q):
        return self.quantile(q)

    def expected_shortfall(self, q):
        """ES_q = (1/(1-q)) * integral_q^1 VaR_u du, the coherent tail risk
        measure -- computed exactly for this discrete law by weighting the
        atom straddling VaR_q by only the fraction of its probability mass
        lying above q, not its full mass (see `_es_from_sorted_atoms`, Part
        4, for why this convention matters). Matches every closed-form and
        quadrature theoretical ES elsewhere in this notebook.
        """
        q_arr = np.asarray(q, dtype=float)
        was_scalar = q_arr.ndim == 0
        q_flat = q_arr.reshape(-1)
        masses = self.probabilities()
        complete = self._complete_rows()
        out = np.full((self.cdfs.shape[0], q_flat.size), np.nan, dtype=float)
        for i, row in enumerate(self.cdfs):
            if not np.all(np.isfinite(row)) or not complete[i]:
                continue
            row_mass = masses[i]
            for j, q_val in enumerate(q_flat):
                if not np.isfinite(q_val):
                    continue
                q_clip = float(np.clip(q_val, 0.0, 1.0))
                if q_clip >= 1.0:
                    out[i, j] = self.thresholds[-1]
                    continue
                idx = int(np.searchsorted(row, q_clip - _IDR_QUANTILE_TIE_EPS, side="left"))  # tie-break tolerance, see note above
                idx = min(idx, self.thresholds.size - 1)
                denom = 1.0 - q_clip
                frac = max(float(row[idx]) - q_clip, 0.0)
                num = frac * self.thresholds[idx]
                if idx + 1 < self.thresholds.size:
                    num += float(np.sum(row_mass[idx + 1:] * self.thresholds[idx + 1:]))
                out[i, j] = num / denom if denom > 0 else self.thresholds[-1]
        return out[:, 0] if was_scalar else out


class OfficialIDRDamageModel:
    """Univariate damage model backed by official `isodistrreg.IDR`."""

    def __init__(self, close_at_max: bool = True) -> None:
        self.close_at_max = close_at_max
        self.model_ = None
        self.thresholds_ = None
        self.hazard_train_ = None
        self.loss_train_ = None

    def fit(self, hazard_data, loss_data) -> "OfficialIDRDamageModel":
        h, l = _finite_training_arrays(hazard_data, loss_data)
        IDR = _import_official_idr()
        try:
            model = IDR(X=h, y=l)
        except TypeError:
            model = IDR(l, h)
        thresholds = getattr(model, "thresholds", None)
        if thresholds is None:
            thresholds = np.unique(l)
        thresholds = np.asarray(thresholds, dtype=float).reshape(-1)
        if thresholds.size == 0:
            raise ValueError("The fitted IDR backend returned no loss thresholds.")
        self.model_ = model
        self.thresholds_ = thresholds
        self.hazard_train_ = h
        self.loss_train_ = l
        return self

    def _require_fitted(self):
        if self.model_ is None or self.thresholds_ is None:
            raise RuntimeError("The IDR model must be fitted before prediction.")
        return self.model_

    @staticmethod
    def _query_array(h_new):
        return np.asarray(h_new, dtype=float).reshape(-1)

    @staticmethod
    def _call_with_univariate_fallback(fn, h, *args):
        try:
            return fn(h, *args)
        except Exception as first_exc:
            try:
                return fn(h.reshape(-1, 1), *args)
            except Exception:
                raise first_exc

    def cdf_matrix(self, h_new):
        model = self._require_fitted()
        h_arr = self._query_array(h_new)
        cdfs = np.full((h_arr.size, self.thresholds_.size), np.nan, dtype=float)
        finite = np.isfinite(h_arr)
        if not np.any(finite):
            return cdfs
        values = self._call_with_univariate_fallback(model.cdf, h_arr[finite])
        values = np.asarray(values, dtype=float)
        if values.ndim == 1:
            values = values.reshape(1, -1)
        expected = (np.count_nonzero(finite), self.thresholds_.size)
        if values.shape != expected:
            raise ValueError(f"Official IDR cdf(...) returned shape {values.shape}, expected {expected}.")
        cdfs[finite, :] = values
        return cdfs

    # cdf(h) / thresholds -- kept as a small, stable public interface for any
    # downstream consumer that wants IDR's bulk distribution.
    def cdf(self, h_new):
        return self.cdf_matrix(h_new)

    @property
    def thresholds(self):
        return self.thresholds_

    def conditional_distribution(self, h_new) -> IDRDiscreteDistribution:
        return IDRDiscreteDistribution(thresholds=self.thresholds_, cdfs=self.cdf_matrix(h_new), close_at_max=self.close_at_max)

    def expected_loss(self, h_new):
        return self.conditional_distribution(h_new).mean()

    def exceedance_distribution(self, h_thresholds, hazard_data=None) -> IDRDiscreteDistribution:
        self._require_fitted()
        if hazard_data is None:
            if self.hazard_train_ is None:
                raise RuntimeError("The IDR model does not have stored training hazards.")
            h_source = self.hazard_train_
        else:
            h_source = _finite_source_array(hazard_data)
        h_thresholds_arr = np.asarray(h_thresholds, dtype=float).reshape(-1)
        cdf_rows = np.full((h_thresholds_arr.size, self.thresholds_.size), np.nan, dtype=float)
        if h_source.size == 0:
            return IDRDiscreteDistribution(thresholds=self.thresholds_, cdfs=cdf_rows, close_at_max=self.close_at_max)
        source_cdfs = self.cdf_matrix(h_source)
        for i, h0 in enumerate(h_thresholds_arr):
            if not np.isfinite(h0):
                continue
            idx = h_source >= h0
            if np.any(idx):
                with np.errstate(all="ignore"):
                    cdf_rows[i, :] = np.nanmean(source_cdfs[idx], axis=0)
        return IDRDiscreteDistribution(thresholds=self.thresholds_, cdfs=cdf_rows, close_at_max=self.close_at_max)

    # Alias expected by _tail_summary's dispatch (Part 8).
    tail_distribution = exceedance_distribution

    def marginal_distribution(self, hazard_data=None) -> IDRDiscreteDistribution:
        self._require_fitted()
        if hazard_data is None:
            if self.hazard_train_ is None:
                raise RuntimeError("The IDR model does not have stored training hazards.")
            h_source = self.hazard_train_
        else:
            h_source = _finite_source_array(hazard_data)
        cdf_row = np.full((1, self.thresholds_.size), np.nan, dtype=float)
        if h_source.size > 0:
            with np.errstate(all="ignore"):
                cdf_row[0, :] = np.nanmean(self.cdf_matrix(h_source), axis=0)
        return IDRDiscreteDistribution(thresholds=self.thresholds_, cdfs=cdf_row, close_at_max=self.close_at_max)

    def predict(self, h_new):
        return self.expected_loss(h_new)

    def __call__(self, h_new):
        return self.predict(h_new)


register_method(MethodSpec("IDR", kind="distributional", fit=OfficialIDRDamageModel, default_train_on="raw"))


## Part 8 — Bootstrap evaluation engine

The core subsample-fitting and pointwise/tail/marginal-risk aggregation loop.
This is the single evaluation implementation used by every run in Part 10,
with an optional `collect_diagnostics: bool` flag on
`run_fullbaseline_predictions_both`: off by default (zero extra cost), on to
additionally capture a per-method, per-subsample fit-status/message table
(why a fit was rejected or errored).

Two extensions support any *aggregate-oriented* model exposing
`.mean(h)`, `.mixture_mean(h_values)`, and/or `.mixture_var_es(h_sample, levels)`,
rather than the per-query
`conditional_distribution`/`tail_distribution`/`marginal_distribution` protocol
IDR exposes:
- `_tail_summary` also recognizes a `mixture_mean` hook (used for the tail
  target, `E[L|H>=VaR_q(H)]`);
- `_marginal_risk_summary` also recognizes a `mixture_var_es` hook (used for
  the marginal VaR/ES table).

Both fall back to the existing sample-based path when neither hook is present,
so IDR and the mean-curve methods are unaffected.

**Cell below:** nan-safe aggregate metrics reused throughout the evaluation
engine, plus the adapter that lets a plain callable behave like a fitted
model.

Given $S$ bootstrap estimates $\hat\theta_1,\dots,\hat\theta_S$ of a true
value $\theta$:
$$\overline{\hat\theta}=\frac1S\sum_i\hat\theta_i,\quad
\mathrm{Bias}=\overline{\hat\theta}-\theta,\quad
\%\text{error}=100\cdot\frac{\mathrm{Bias}}{|\theta|},$$
$$\mathrm{RMSE}=\sqrt{\mathbb E\big[(\hat\theta-\theta)^2\big]},\qquad
\mathrm{MAE}=\mathbb E\big[|\hat\theta-\theta|\big].$$
`decompose_error` additionally reports the bias–variance identity
$$\mathrm{MSE}=\mathrm{Bias}^2+\mathrm{Var}(\hat\theta),$$
computed post-hoc from the same per-subsample arrays the evaluation loop
already returns.


In [ ]:
# ---------------------------------------------------------------------------
# Nan-safe bootstrap metrics.
# ---------------------------------------------------------------------------

def _finite_count(a, axis=0):
    return np.sum(np.isfinite(a), axis=axis)


def _nanmean_safe(a, axis=0):
    a = np.asarray(a, dtype=float)
    cnt = _finite_count(a, axis=axis)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        out = np.nanmean(a, axis=axis)
    return np.where(cnt > 0, out, np.nan)


def _nanvar_safe(a, axis=0):
    a = np.asarray(a, dtype=float)
    cnt = _finite_count(a, axis=axis)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        out = np.nanvar(a, axis=axis)
    return np.where(cnt > 0, out, np.nan)


def compute_variance(estimates, axis=0):
    return np.asarray(_nanvar_safe(np.asarray(estimates, dtype=float), axis=axis), dtype=float)


def compute_pct_error(estimates, truth, axis=0, eps=1e-12):
    est = np.asarray(estimates, dtype=float)
    tru = np.asarray(truth, dtype=float)
    mean_estimate = _nanmean_safe(est, axis=axis)
    denom = np.maximum(np.abs(tru), eps)
    return np.asarray(100.0 * (mean_estimate - tru) / denom, dtype=float)


def compute_rmse(estimates, truth, axis=0):
    est, tru = np.asarray(estimates, dtype=float), np.asarray(truth, dtype=float)
    return np.asarray(np.sqrt(_nanmean_safe((est - tru) ** 2, axis=axis)), dtype=float)


def compute_mae(estimates, truth, axis=0):
    est, tru = np.asarray(estimates, dtype=float), np.asarray(truth, dtype=float)
    return np.asarray(_nanmean_safe(np.abs(est - tru), axis=axis), dtype=float)


def summarize_bootstrap(estimates, truth, axis=0, eps=1e-12):
    est, tru = np.asarray(estimates, dtype=float), np.asarray(truth, dtype=float)
    return {
        "mean_estimate": np.asarray(_nanmean_safe(est, axis=axis), dtype=float),
        "variance": compute_variance(est, axis=axis),
        "pct_error": compute_pct_error(est, tru, axis=axis, eps=eps),
        "rmse": compute_rmse(est, tru, axis=axis),
        "mae": compute_mae(est, tru, axis=axis),
        "truth": tru,
        "valid_count": np.asarray(_finite_count(est, axis=axis), dtype=float),
    }


def decompose_error(est_matrix, truth_by_level):
    """Bias^2 + variance = MSE decomposition for one method's (S, L) estimate
    matrix against its (L,) truth vector. Computed post-hoc from the same
    per-subsample arrays run_fullbaseline_predictions_both already returns --
    no extra per-subsample state is needed."""
    est = np.asarray(est_matrix, dtype=float)
    truth = np.asarray(truth_by_level, dtype=float)
    mean_est = _nanmean_safe(est, axis=0)
    bias = mean_est - truth
    variance = compute_variance(est, axis=0)
    mse = compute_rmse(est, truth, axis=0) ** 2
    return pd.DataFrame({
        "truth": truth, "mean_estimate": mean_est, "bias": bias,
        "bias_sq": bias**2, "variance": variance, "mse": mse,
        "bias_sq_plus_variance": bias**2 + variance,
    })


@dataclass
class FunctionDamageModel:
    """Adapter exposing a predict(h) API from a plain damage-function callable."""

    fn: Callable[[np.ndarray], np.ndarray]

    def predict(self, h):
        return np.asarray(self.fn(h), dtype=float)


def _sample_var_es(sample, levels):
    """Empirical VaR and coherent ES for a one-dimensional sample.

    VaR is the empirical inverse-CDF quantile. ES is the coherent
    (superquantile) tail risk measure computed via `_es_from_sorted_atoms`
    (Part 4), treating the sample as a discrete uniform law -- not a plain
    mean of every point at or above VaR_q, which would over-count the
    boundary point and target a different, more biased functional than the
    theoretical ES this is benchmarked against.
    """
    sample = np.asarray(sample, dtype=float)
    levels = np.asarray(levels, dtype=float)
    var = np.quantile(sample, levels, method="inverted_cdf")
    sorted_sample = np.sort(sample)
    probs = np.full(sorted_sample.size, 1.0 / sorted_sample.size, dtype=float)
    es = _es_from_sorted_atoms(sorted_sample, probs, levels)
    return np.asarray(var, dtype=float), es


def _subsample_loss_var_es(losses, levels):
    return _sample_var_es(losses, levels)


def summarize(var_levels, ce_full, ce_by_method, var_sub, var_full):
    out = {}
    for name, est_mat in ce_by_method.items():
        out[name] = {}
        for j, lvl in enumerate(var_levels):
            est = np.asarray(est_mat[:, j], dtype=float)
            m = summarize_bootstrap(est[:, None], np.asarray([ce_full[j]], dtype=float), axis=0)
            out[name][lvl] = dict(
                mean=float(m["mean_estimate"][0]), std=float(np.sqrt(m["variance"][0])),
                bias=float(m["mean_estimate"][0] - ce_full[j]), rmse=float(m["rmse"][0]), mae=float(m["mae"][0]),
            )
    var_stats = {
        lvl: dict(full=float(var_full[j]), mean=float(np.nanmean(var_sub[:, j])), std=float(np.nanstd(var_sub[:, j], ddof=1)))
        for j, lvl in enumerate(var_levels)
    }
    return out, var_stats


### `_fit_predictors_for_subsample`: one dispatch, three method shapes

Every entry in the `methods` dict registered via `MethodSpec.fit` is one of:
a **legacy callable** `fn(h_query, x_train, y_train) -> predictions` (the
classical curve fits, Parts 7a-7c); a **class** (`OfficialIDRDamageModel`, or
one of the `_make_evt_class` variants), instantiated with zero args and then
`.fit(x_train, y_train)`; or an already-constructed model exposing
`.fit`/`.predict`. The dispatch order matters: Python classes satisfy
`hasattr(cls, "fit")` (unbound methods are visible on the class itself), so
they correctly skip the "legacy callable" branch and fall through to
`isinstance(method, type)` — but a plain zero-argument *function* would be
misread as a legacy `fn(h,x,y)` callable and called with the wrong
arguments, which is exactly why every method is registered as a class
rather than a factory closure. `train_on[name]` selects raw pairs vs. the Part 7f binned
summary per method. With `collect_diagnostics=True`, a fit that raises, or
that a model marks `.failed=True` (a rejected significance screen or a
non-converged GPD fit), is recorded rather than silently propagated.


In [ ]:
def _fit_predictors_for_subsample(methods, train_on, h, y, xb, yb, *, fit_kwargs_map=None, collect_diagnostics=False):
    """Fit each method once for a subsample and return fitted estimators.

    Supported method entries:
    - legacy callable: fn(h_query, x_train, y_train, **kwargs) -> predictions
    - a class (`isinstance(method, type)`), instantiated as `method(**kwargs)`,
      then `.fit(x_train, y_train)` -> object with `.predict(...)`
    - an already-constructed model/factory exposing `.fit(...)`/`.predict(...)`

    `fit_kwargs_map` (name -> dict) supplies each method's `MethodConfig.fit_kwargs`
    override, applied at the point the method is actually invoked.

    `idr_cache` (local to one call) lets any method whose `.fit(...)` accepts
    an `idr_cache=` keyword share one IDR fit across every method that needs
    it on this subsample, instead of each fitting its own -- seeded here
    whenever a plain `OfficialIDRDamageModel` (the standalone "IDR" method)
    is fit.

    When `collect_diagnostics=True`, also returns a list of per-method
    `{"method": name, "status": "ok"|"error", "message": str}` records.
    """
    predictors = {}
    diagnostics = []
    fit_kwargs_map = fit_kwargs_map or {}
    idr_cache = {}

    for name, method in methods.items():
        mode = train_on.get(name, "raw")
        if mode == "binned":
            if xb is None or yb is None:
                raise ValueError(f"Method '{name}' requested binned fitting but binned data is missing.")
            x_train, y_train = xb, yb
        else:
            x_train, y_train = h, y
        kwargs = fit_kwargs_map.get(name, {})

        status, message = "ok", ""
        try:
            if callable(method) and not hasattr(method, "fit"):
                def make_legacy_predictor(fn, xtr, ytr, kw):
                    return lambda hq: np.asarray(fn(np.asarray(hq, dtype=float), xtr, ytr, **kw), dtype=float)

                predictors[name] = FunctionDamageModel(make_legacy_predictor(method, x_train, y_train, kwargs))
                if collect_diagnostics and np.any(~np.isfinite(predictors[name].predict(x_train[:1] if len(x_train) else np.array([0.0])))):
                    status, message = "rejected", "legacy fit returned NaN (significance screen failed)"
            else:
                model = method
                if isinstance(method, type):
                    model = method(**kwargs)
                if not hasattr(model, "fit"):
                    raise TypeError(f"Method '{name}' is neither a legacy callable nor a model with fit(...).")
                try:
                    fitted = model.fit(x_train, y_train, idr_cache=idr_cache)
                except TypeError:
                    fitted = model.fit(x_train, y_train)
                if not hasattr(fitted, "predict"):
                    raise TypeError(f"Method '{name}' fit(...) did not return an object with predict(...).")
                predictors[name] = fitted
                if isinstance(fitted, OfficialIDRDamageModel):
                    idr_cache[(id(x_train), id(y_train))] = fitted
                if collect_diagnostics and getattr(fitted, "failed", False):
                    status, message = "rejected", str(getattr(fitted, "reason", "") or "fit rejected")
        except Exception as exc:
            status, message = "error", f"{type(exc).__name__}: {exc}"
            predictors[name] = FunctionDamageModel(lambda hq: np.full(np.asarray(hq, dtype=float).shape, np.nan, dtype=float))

        if collect_diagnostics:
            diagnostics.append({"method": name, "status": status, "message": message, "train_on": mode, "n_train": int(np.asarray(x_train).size)})

    if collect_diagnostics:
        return predictors, diagnostics
    return predictors, None

### Per-query summaries: pointwise, tail, and marginal risk

Three functions turn one fitted model into the three quantities the results
tables (Part 9) need, each preferring the richest API the model actually
exposes and falling back gracefully:

- **Pointwise** (`_pointwise_summary`): if the model exposes a full
  `conditional_distribution(h)` (IDR), use its mean/VaR/ES; otherwise use
  `.predict(h)` as a point estimate with VaR/ES left `NaN`.
- **Tail** (`_tail_summary`): prefer an IDR-style `tail_distribution`
  (exact mixture over $\{i:H_i\ge h_q\}$); else an aggregate-oriented
  model's `mixture_mean` hook; else fall back to
  $$\widehat{\mathbb E}[\hat m(H)\mid H\ge h_q]=\frac{1}{|\{i:H_i\ge h_q\}|}\sum_{i:H_i\ge h_q}\hat m(H_i),$$
  computed for *every* threshold in one pass via cached reverse cumulative
  sums (`_tail_means_from_cached_index`) rather than re-summing per threshold.
- **Marginal risk** (`_marginal_risk_summary`): prefer `marginal_distribution`
  (IDR) or an aggregate-oriented model's `mixture_var_es` hook; else empirical
  VaR/ES of the predicted-damage sample. The `mixture_mean`/`mixture_var_es`
  hooks are this notebook's extensibility point for any future model whose
  native API is aggregate-oriented rather than per-query.


In [ ]:
def _prepare_tail_indexing(h, thresholds):
    """Precompute sorting and threshold lookup indices, method-independent
    and reused across all methods for a given subsample."""
    order = np.argsort(h)
    hs = h[order]
    idxs = np.searchsorted(hs, thresholds, side="left")
    return order, idxs


def _tail_means_from_cached_index(yhat_all, order, idxs, out_shape):
    """E[yhat | H >= threshold] for many thresholds using cached indexing."""
    ys = np.asarray(yhat_all, dtype=float)[order]
    ys_rev = ys[::-1]
    sum_rev = np.cumsum(np.nan_to_num(ys_rev, nan=0.0))[::-1]
    cnt_rev = np.cumsum(~np.isnan(ys_rev))[::-1]
    out = np.full(out_shape, np.nan, dtype=float)
    valid = idxs < ys.size
    if np.any(valid):
        idxv = idxs[valid]
        with np.errstate(invalid="ignore", divide="ignore"):
            out[valid] = sum_rev[idxv] / cnt_rev[idxv]
    return out


def _tail_variances_from_cached_index(yhat_all, order, idxs, out_shape):
    ys = np.asarray(yhat_all, dtype=float)[order]
    ys_rev = ys[::-1]
    finite_rev = ~np.isnan(ys_rev)
    sum_rev = np.cumsum(np.nan_to_num(ys_rev, nan=0.0))[::-1]
    sumsq_rev = np.cumsum(np.nan_to_num(ys_rev, nan=0.0) ** 2)[::-1]
    cnt_rev = np.cumsum(finite_rev)[::-1]
    out = np.full(out_shape, np.nan, dtype=float)
    valid = idxs < ys.size
    if np.any(valid):
        idxv = idxs[valid]
        cntv, sumv, sumsqv = cnt_rev[idxv], sum_rev[idxv], sumsq_rev[idxv]
        positive = cntv > 0
        vals = np.full(cntv.shape, np.nan, dtype=float)
        if np.any(positive):
            mean = sumv[positive] / cntv[positive]
            vals[positive] = np.maximum(sumsqv[positive] / cntv[positive] - mean**2, 0.0)
        out[valid] = vals
    return out


def _predict_conditional_mean(model, h_query):
    """Return conditional-mean predictions, preferring distributional APIs."""
    h_arr = np.asarray(h_query, dtype=float).reshape(-1)
    if hasattr(model, "conditional_means") and callable(getattr(model, "conditional_means")):
        values = model.conditional_means(h_arr)
    else:
        values = model.predict(h_arr)
    return np.asarray(values, dtype=float).reshape(-1)


def _paired_distribution_values(values, n):
    """Extract one q-specific value for each of n threshold rows."""
    arr = np.asarray(values, dtype=float)
    if arr.ndim == 0:
        return np.full(n, float(arr), dtype=float)
    if arr.ndim == 1:
        if arr.size == n:
            return arr.astype(float, copy=False)
        if arr.size == 1:
            return np.full(n, float(arr[0]), dtype=float)
    if arr.ndim == 2:
        if arr.shape == (n, n):
            return np.diag(arr)
        if arr.shape[0] == n and arr.shape[1] == 1:
            return arr[:, 0]
        if arr.shape[0] == 1 and arr.shape[1] == n:
            return arr[0, :]
    raise ValueError(f"Cannot pair distribution values with shape {arr.shape} to {n} rows.")


def _pointwise_summary(model, h_query, var_levels):
    """Pointwise estimates, using full conditional distributions when available."""
    h_arr = np.asarray(h_query, dtype=float).reshape(-1)
    levels = np.asarray(var_levels, dtype=float).reshape(-1)
    n = h_arr.size
    summary = {
        "mean": _predict_conditional_mean(model, h_arr),
        "variance": np.full(n, np.nan, dtype=float),
        "value_at_risk": np.full(n, np.nan, dtype=float),
        "expected_shortfall": np.full(n, np.nan, dtype=float),
        "stat_source": "predicted_damage_sample",
    }
    dist_fn = getattr(model, "conditional_distribution", None) or getattr(model, "conditional_distributions", None)
    if not callable(dist_fn):
        return summary
    dist = dist_fn(h_arr)
    if hasattr(dist, "mean") and callable(getattr(dist, "mean")):
        summary["mean"] = np.asarray(dist.mean(), dtype=float).reshape(-1)
    if hasattr(dist, "variance") and callable(getattr(dist, "variance")):
        summary["variance"] = np.asarray(dist.variance(), dtype=float).reshape(-1)
    q_fn = getattr(dist, "value_at_risk", None) or getattr(dist, "quantile", None)
    if callable(q_fn):
        summary["value_at_risk"] = _paired_distribution_values(q_fn(levels), n)
    es_fn = getattr(dist, "expected_shortfall", None)
    if callable(es_fn):
        summary["expected_shortfall"] = _paired_distribution_values(es_fn(levels), n)
    summary["stat_source"] = "conditional_distribution"
    return summary


def _tail_summary(model, h_sample, h_thresholds, var_levels, order, idxs):
    """Tail estimates: prefer an IDR-style `tail_distribution`, then an
    EVT-style `mixture_mean` hook, then fall back to sample aggregation of
    pointwise predictions."""
    h = np.asarray(h_sample, dtype=float).reshape(-1)
    thresholds = np.asarray(h_thresholds, dtype=float).reshape(-1)
    levels = np.asarray(var_levels, dtype=float).reshape(-1)
    n = thresholds.size

    tail_dist_fn = getattr(model, "tail_distribution", None)
    if callable(tail_dist_fn):
        dist = tail_dist_fn(thresholds, hazard_data=h)
        summary = {
            "mean": np.asarray(dist.mean(), dtype=float).reshape(-1),
            "variance": np.asarray(dist.variance(), dtype=float).reshape(-1),
            "value_at_risk": np.full(n, np.nan, dtype=float),
            "expected_shortfall": np.full(n, np.nan, dtype=float),
            "stat_source": "tail_distribution",
        }
        q_fn = getattr(dist, "value_at_risk", None) or getattr(dist, "quantile", None)
        if callable(q_fn):
            summary["value_at_risk"] = _paired_distribution_values(q_fn(levels), n)
        es_fn = getattr(dist, "expected_shortfall", None)
        if callable(es_fn):
            summary["expected_shortfall"] = _paired_distribution_values(es_fn(levels), n)
        return summary

    mixture_mean_fn = getattr(model, "mixture_mean", None)
    if callable(mixture_mean_fn):
        mean = np.array([float(mixture_mean_fn(h[h >= thr])) if np.any(h >= thr) else np.nan for thr in thresholds])
        return {
            "mean": mean, "variance": np.full(n, np.nan, dtype=float),
            "value_at_risk": np.full(n, np.nan, dtype=float), "expected_shortfall": np.full(n, np.nan, dtype=float),
            "stat_source": "mixture_mean",
        }

    pred_all = _predict_conditional_mean(model, h)
    value_at_risk, expected_shortfall = _sample_var_es(pred_all, levels)
    return {
        "mean": _tail_means_from_cached_index(pred_all, order, idxs, thresholds.shape),
        "variance": _tail_variances_from_cached_index(pred_all, order, idxs, thresholds.shape),
        "value_at_risk": value_at_risk, "expected_shortfall": expected_shortfall,
        "stat_source": "predicted_damage_sample",
    }


def _marginal_risk_summary(model, h_sample, var_levels):
    """VaR/ES for the model-implied marginal loss distribution: prefer an
    IDR-style `marginal_distribution`, then an EVT-style `mixture_var_es`
    hook, then fall back to VaR/ES of the predicted conditional means."""
    h = np.asarray(h_sample, dtype=float).reshape(-1)
    levels = np.asarray(var_levels, dtype=float).reshape(-1)
    n = levels.size

    marginal_dist_fn = getattr(model, "marginal_distribution", None)
    if callable(marginal_dist_fn):
        dist = marginal_dist_fn(hazard_data=h)
        summary = {
            "mean": np.full(n, np.nan, dtype=float), "variance": np.full(n, np.nan, dtype=float),
            "value_at_risk": np.full(n, np.nan, dtype=float), "expected_shortfall": np.full(n, np.nan, dtype=float),
            "stat_source": "marginal_distribution",
        }
        if hasattr(dist, "mean") and callable(getattr(dist, "mean")):
            summary["mean"] = _paired_distribution_values(dist.mean(), n)
        if hasattr(dist, "variance") and callable(getattr(dist, "variance")):
            summary["variance"] = _paired_distribution_values(dist.variance(), n)
        q_fn = getattr(dist, "value_at_risk", None) or getattr(dist, "quantile", None)
        if callable(q_fn):
            summary["value_at_risk"] = _paired_distribution_values(q_fn(levels), n)
        es_fn = getattr(dist, "expected_shortfall", None)
        if callable(es_fn):
            summary["expected_shortfall"] = _paired_distribution_values(es_fn(levels), n)
        return summary

    mixture_var_es_fn = getattr(model, "mixture_var_es", None)
    if callable(mixture_var_es_fn):
        var, es = mixture_var_es_fn(h, levels)
        return {
            "mean": np.full(n, np.nan, dtype=float), "variance": np.full(n, np.nan, dtype=float),
            "value_at_risk": np.asarray(var, dtype=float), "expected_shortfall": np.asarray(es, dtype=float),
            "stat_source": "mixture_var_es",
        }

    pred_all = _predict_conditional_mean(model, h)
    value_at_risk, expected_shortfall = _sample_var_es(pred_all, levels)
    variance = float(np.nanvar(pred_all)) if np.any(np.isfinite(pred_all)) else np.nan
    return {
        "mean": np.full(n, float(np.nanmean(pred_all)) if np.any(np.isfinite(pred_all)) else np.nan, dtype=float),
        "variance": np.full(n, variance, dtype=float),
        "value_at_risk": value_at_risk, "expected_shortfall": expected_shortfall,
        "stat_source": "predicted_damage_sample",
    }


### The bootstrap loop, and `run_fullbaseline_predictions_both`

`_evaluate_subsample_records_both` is what runs *once per bootstrap subsample*:
compute the empirical hazard VaR $\hat q_\alpha=\hat F_H^{-1}(\alpha)$ at every
requested level, bin if any registered method needs it, fit every method
(previous cells), then call the three per-query summaries above and append
one result row per (method, VaR level, target) triple.
`run_fullbaseline_predictions_both` (behind an optional `collect_diagnostics`
flag — see this Part's header) repeats that across all $S$ subsamples,
optionally in parallel via `joblib`
(`_execute_parallel`, falling back to a sequential loop if `joblib` isn't
installed or `n_jobs=None`), then aggregates each method's $S\times L$
estimate matrix against the true value via `summarize` (mean, std, bias,
RMSE, MAE per VaR level — same formulas as this Part's header cell).


In [ ]:
def _evaluate_subsample_records_both(h, y, methods, train_on, var_levels, n_bins, bin_summary_fn,
                                      h_theory_true, true_full_pointwise, true_full_tail,
                                      bin_summary_func=np.mean, bin_x="midpoint", collect_diagnostics=False,
                                      fit_kwargs_map=None):
    loss_var_sub, loss_es_sub = _subsample_loss_var_es(y, var_levels)

    xb = yb = None
    need_binned = any(train_on.get(m, "raw") == "binned" for m in methods)
    if need_binned:
        xb, yb = bin_summary_fn(np.column_stack([h, y]), num_bins=n_bins, summary_func=bin_summary_func, bin_x=bin_x)
        ok = np.isfinite(xb) & np.isfinite(yb)
        xb, yb = xb[ok], yb[ok]

    predictors, fit_diag = _fit_predictors_for_subsample(methods, train_on, h, y, xb, yb,
                                                          fit_kwargs_map=fit_kwargs_map, collect_diagnostics=collect_diagnostics)
    hq = np.quantile(h, var_levels, method="inverted_cdf")
    tail_order, tail_idxs = _prepare_tail_indexing(h, hq)
    # Empirical hazard-tail conditional mean of the raw subsample losses --
    # free to compute here since tail_order/tail_idxs are already built for
    # the per-method model predictions below; this is the model-free
    # "Empirical subsample" baseline for the Tail conditional expectation
    # table (build_summary_tables).
    tail_mean_sub = _tail_means_from_cached_index(y, tail_order, tail_idxs, hq.shape)

    est_pointwise = {name: np.full(var_levels.shape, np.nan, dtype=float) for name in methods}
    est_tail = {name: np.full(var_levels.shape, np.nan, dtype=float) for name in methods}
    records_pointwise, records_tail = [], []

    for name, model in predictors.items():
        point_summary = _pointwise_summary(model, hq, var_levels)
        tail_summary = _tail_summary(model, h, hq, var_levels, tail_order, tail_idxs)
        risk_summary = _marginal_risk_summary(model, h, var_levels)

        pred_hq = np.asarray(point_summary["mean"], dtype=float)
        pred_tail = np.asarray(tail_summary["mean"], dtype=float)
        est_pointwise[name] = pred_hq
        est_tail[name] = pred_tail

        for j, lvl in enumerate(var_levels):
            base = dict(
                method=name, subsample_id=0, var_level=float(lvl), h_var_sub=float(hq[j]),
                loss_var_sub=float(loss_var_sub[j]), loss_es_sub=float(loss_es_sub[j]),
                tail_mean_sub=float(tail_mean_sub[j]),
                damage_var_sub=float(risk_summary["value_at_risk"][j]), damage_es_sub=float(risk_summary["expected_shortfall"][j]),
                damage_variance_sub=float(risk_summary["variance"][j]), damage_stat_source=str(risk_summary["stat_source"]),
                h_theory_true=float(h_theory_true[j]),
            )
            records_pointwise.append({**base, "prediction": float(pred_hq[j]), "true_full": float(true_full_pointwise[j]), "target": "pointwise"})
            records_tail.append({**base, "prediction": float(pred_tail[j]), "true_full": float(true_full_tail[j]), "target": "tail"})

    return hq, est_pointwise, est_tail, records_pointwise, records_tail, fit_diag


def _execute_parallel(func, args_list, n_jobs):
    if n_jobs is None or n_jobs == 1 or Parallel is None:
        return [func(*args) for args in args_list]
    return Parallel(n_jobs=n_jobs)(delayed(func)(*args) for args in args_list)


def run_fullbaseline_predictions_both(hazard_sub, losses_sub, methods, train_on, var_levels, n_bins,
                                      *, h_theory_true, true_full_pointwise, true_full_tail,
                                      n_jobs=None, bin_summary_func=np.mean, bin_x="midpoint",
                                      collect_diagnostics=False, fit_kwargs_map=None):
    """Evaluate pointwise and tail targets in one subsample loop.

    `collect_diagnostics=False` (default) has zero extra cost.
    `collect_diagnostics=True` additionally returns a per-subsample,
    per-method fit-status table.
    """
    hazard_sub = np.asarray(hazard_sub, dtype=float)
    losses_sub = np.asarray(losses_sub, dtype=float)
    var_levels = np.asarray(var_levels, dtype=float)
    h_theory_true = np.asarray(h_theory_true, dtype=float)
    true_full_pointwise = np.asarray(true_full_pointwise, dtype=float)
    true_full_tail = np.asarray(true_full_tail, dtype=float)

    S, _ = hazard_sub.shape
    L = len(var_levels)
    var_sub_out = np.full((S, L), np.nan, dtype=float)
    est_pointwise = {name: np.full((S, L), np.nan, dtype=float) for name in methods}
    est_tail = {name: np.full((S, L), np.nan, dtype=float) for name in methods}
    records_pointwise, records_tail, diagnostics = [], [], []

    args_list = [
        (hazard_sub[i], losses_sub[i], methods, train_on, var_levels, n_bins, bin_summary,
         h_theory_true, true_full_pointwise, true_full_tail, bin_summary_func, bin_x, collect_diagnostics,
         fit_kwargs_map)
        for i in range(S)
    ]
    results = _execute_parallel(_evaluate_subsample_records_both, args_list, n_jobs=n_jobs)

    for i, (hq, est_pt, est_tl, rec_pt, rec_tl, fit_diag) in enumerate(results):
        var_sub_out[i] = hq
        for name in methods:
            est_pointwise[name][i, :] = est_pt[name]
            est_tail[name][i, :] = est_tl[name]
        for row in rec_pt:
            row["subsample_id"] = int(i)
        for row in rec_tl:
            row["subsample_id"] = int(i)
        records_pointwise.extend(rec_pt)
        records_tail.extend(rec_tl)
        if collect_diagnostics and fit_diag:
            for row in fit_diag:
                diagnostics.append({**row, "subsample_id": int(i)})

    summary_pointwise, var_stats = summarize(var_levels, true_full_pointwise, est_pointwise, var_sub_out, h_theory_true)
    summary_tail, _ = summarize(var_levels, true_full_tail, est_tail, var_sub_out, h_theory_true)

    return {
        "var_levels": var_levels, "h_theory_true": h_theory_true, "var_sub": var_sub_out,
        "pointwise": {"true_full": true_full_pointwise, "pred_by_method": est_pointwise, "summary": summary_pointwise, "records": records_pointwise},
        "tail": {"true_full": true_full_tail, "pred_by_method": est_tail, "summary": summary_tail, "records": records_tail},
        "var_stats": var_stats,
        "diagnostics": pd.DataFrame(diagnostics) if collect_diagnostics else None,
    }

### Generating the population and bootstrap subsamples

The actual data-generating process this whole study benchmarks against:
draw $U\sim\mathrm{Unif}(0,1)$, invert the copula's conditional CDF
$\partial C/\partial u(U,\cdot)$ at an independent uniform to get $V$ (Part
5's `sample_ev_copula`), then transform each uniform through its own margin's
quantile function — $H=F_H^{-1}(U)$ directly, but $L$ via
$L=Q_L\big(\operatorname{expit}(z)\big)=F_L^{-1}(v)$ computed as
`loss_dist.isf(1 - v)` for the tail-precision reasons noted in Part 6.
`draw_subsamples` then repeatedly samples $n$ *without replacement* from the
size-$N$ population, `iterations` times.

`generate_population_and_subsamples` is the one place `data_backend` is
resolved: `"memory"` regenerates in-process every call (used by Light and
the sensitivity sweep, where the population is small/cheap); `"hdf5"`
persists the population and the subsample **index** arrays (not the
subsampled data itself — the indices are enough to reconstruct it) to a
cache file keyed by `StudyConfig.cache_tag()`, so the expensive Full-scale
population (Part 10b) is only ever generated once per `(scenario, n,
iterations)` combination.


In [ ]:
# ---------------------------------------------------------------------------
# Population/subsample generation and the top-level orchestrator,
# run_bootstrap_study(config) -> BootstrapResult. This is the single call
# every "Run" section in Part 10 makes; it wires the in-memory generation path
# (used by Light and the sensitivity sweep) and an optional HDF5-cached path
# (used by the full-scale run) behind one `data_backend` switch.
# ---------------------------------------------------------------------------

@dataclass
class PopulationData:
    hazard: np.ndarray
    losses: np.ndarray


@dataclass
class SubsampleData:
    hazard_sub: np.ndarray  # shape (iterations, n)
    losses_sub: np.ndarray  # shape (iterations, n)


def generate_population(built: BuiltScenario, N: int, rng) -> PopulationData:
    u, z = sample_ev_copula(N, built.pickands, rng)
    hazard = built.hazard_dist.ppf(u)
    losses = built.loss_dist.isf(np.exp(_log1mv(z)))
    return PopulationData(hazard=hazard, losses=losses)


def draw_subsamples(population: PopulationData, n: int, iterations: int, rng) -> SubsampleData:
    N = population.hazard.size
    if n > N:
        raise ValueError(f"Subsample size n={n} cannot exceed population size N={N}.")
    idx = np.empty((iterations, n), dtype=np.int64)
    for i in range(iterations):
        idx[i, :] = rng.choice(N, size=n, replace=False)
    return SubsampleData(hazard_sub=population.hazard[idx], losses_sub=population.losses[idx])


def _hdf5_cache_path(config: "StudyConfig", tag_suffix: str = "") -> Path:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    return DATA_DIR / f"{config.cache_tag()}{tag_suffix}.h5"


def generate_population_and_subsamples(config: "StudyConfig", built: BuiltScenario, tag_suffix: str = ""):
    scenario, run = config.scenario, config.run

    if run.data_backend == "hdf5":
        if h5py is None:
            raise ImportError("h5py is required for data_backend='hdf5'.")
        path = _hdf5_cache_path(config, tag_suffix)
        if path.exists() and not getattr(run, "rebuild_cache", False):
            with h5py.File(path, "r") as f:
                population = PopulationData(hazard=f["hazard"][:], losses=f["losses"][:])
                idx = f["sub_indices"][:]
            print(f"Loaded cached population + subsample indices from {path}")
            return population, SubsampleData(hazard_sub=population.hazard[idx], losses_sub=population.losses[idx])

        rng = np.random.default_rng(scenario.seed)
        population = generate_population(built, scenario.N, rng)
        sub_rng = np.random.default_rng(scenario.seed + 1)
        idx = np.empty((run.iterations, run.n), dtype=np.int64)
        for i in range(run.iterations):
            idx[i, :] = sub_rng.choice(scenario.N, size=run.n, replace=False)
        with h5py.File(path, "w") as f:
            f.create_dataset("hazard", data=population.hazard, compression="gzip")
            f.create_dataset("losses", data=population.losses, compression="gzip")
            f.create_dataset("sub_indices", data=idx, compression="gzip")
        print(f"Generated and cached population + subsample indices to {path}")
        return population, SubsampleData(hazard_sub=population.hazard[idx], losses_sub=population.losses[idx])

    rng = np.random.default_rng(scenario.seed)
    population = generate_population(built, scenario.N, rng)
    sub_rng = np.random.default_rng(scenario.seed + 1)
    subsamples = draw_subsamples(population, run.n, run.iterations, sub_rng)
    return population, subsamples


@dataclass
class BootstrapResult:
    config: "StudyConfig"
    built: BuiltScenario
    population: PopulationData
    h_theory_true: np.ndarray
    output: dict
    point_df: pd.DataFrame
    tail_df: pd.DataFrame
    diagnostics: pd.DataFrame | None


def run_bootstrap_study(config: "StudyConfig", *, cache_tag_suffix: str = "") -> BootstrapResult:
    built = build_scenario(config.scenario)
    population, subsamples = generate_population_and_subsamples(config, built, tag_suffix=cache_tag_suffix)

    methods, train_on, specs = build_methods_and_train_on(config.run.methods)
    # Per-run MethodConfig.fit_kwargs overrides, merged over each method's
    # registry-level MethodSpec.fit_kwargs default -- threaded through to
    # _fit_predictors_for_subsample (Part 8), which applies them at the
    # point each method is actually constructed/called.
    fit_kwargs_map = {cfg.name: {**specs[cfg.name].fit_kwargs, **cfg.fit_kwargs} for cfg in config.run.methods}
    var_levels = np.asarray(config.run.var_levels, dtype=float)
    h_theory_true = built.hazard_dist.ppf(var_levels)
    true_full_pointwise = np.asarray(built.targets.d(h_theory_true), dtype=float)
    true_full_tail = np.asarray(built.targets.tail_mean(h_theory_true), dtype=float)

    output = run_fullbaseline_predictions_both(
        subsamples.hazard_sub, subsamples.losses_sub, methods, train_on, var_levels, config.run.bins,
        h_theory_true=h_theory_true, true_full_pointwise=true_full_pointwise, true_full_tail=true_full_tail,
        n_jobs=config.run.n_jobs, bin_summary_func=config.run.bin_summary_func, bin_x=config.run.bin_x,
        collect_diagnostics=config.run.collect_diagnostics, fit_kwargs_map=fit_kwargs_map,
    )
    point_df = pd.DataFrame(output["pointwise"]["records"])
    tail_df = pd.DataFrame(output["tail"]["records"])

    return BootstrapResult(
        config=config, built=built, population=population, h_theory_true=h_theory_true,
        output=output, point_df=point_df, tail_df=tail_df, diagnostics=output["diagnostics"],
    )

## Part 9 — Reporting and plotting

`build_summary_tables(result)` assembles one Hazard-VaR table, one Loss-VaR table, one
pointwise-conditional-expectation table, one Loss-ES table, one
tail-conditional-expectation table, each row's method filtered by that
method's `MethodSpec.reports` (a method contributing to only a subset of
the tables is possible, though every currently-registered method reports
to all three).

`plot_grouped_boxplots` and `plot_sanity_check` replace the corresponding
cells, callable identically regardless of which scenario produced `result`.

**Cell below:** `_summary_record`/`_method_order`/`build_summary_tables` —
the consolidated results-table builder. For
each (quantity, method, VaR level $q$) triple it reports the bootstrap mean,
median, std, bias, RMSE, MAE (formulas in Part 8's header) against the
theoretical truth, filtered per method by that `MethodSpec`'s `reports` set.


In [ ]:
def _summary_record(quantity, estimator, level, values, truth, status=None):
    """`status`, when given, is a per-value array of `_fit_predictors_for_subsample`'s
    fit-status strings ("ok"/"rejected"/"error", Part 8) aligned with `values`,
    element-for-element, used to attribute each NaN to its cause instead of
    lumping every NaN into one count:
      - "rejected": the method's own significance screen or `.failed` gate
        discarded this iteration's fit entirely (Quadratic/Sigmoid's p-value
        test) -- a deliberate modeling decision, not a numerical failure.
      - "error": the fit itself raised an exception.
      - "other": a NaN with no matching diagnostics -- e.g. `result.diagnostics`
        is None (a cache-loaded result, or `collect_diagnostics=False`), or
        this row is a non-fitted quantity like the empirical-subsample
        baseline. In principle also covers a fit that reported "ok" but
        still produced a NaN somewhere downstream.
    With `status=None` (the default), every NaN is counted as "other",
    matching this function's previous, unsplit behaviour.
    """
    values = np.asarray(values, dtype=float)
    finite = values[np.isfinite(values)]
    truth = float(truth)
    nan_mask = ~np.isfinite(values)
    nan_count = int(nan_mask.sum())

    if status is not None:
        status = np.asarray(status, dtype=object)
        n_rejected = int(np.sum(nan_mask & (status == "rejected")))
        n_error = int(np.sum(nan_mask & (status == "error")))
        n_other = nan_count - n_rejected - n_error
    else:
        n_rejected = n_error = 0
        n_other = nan_count

    if finite.size == 0:
        mean = median = std = bias = pct_error = median_pct_error = rmse = mae = np.nan
    else:
        errors = finite - truth
        mean, median = float(np.mean(finite)), float(np.median(finite))
        std = float(np.std(finite, ddof=1)) if finite.size > 1 else 0.0
        bias = float(np.mean(errors))
        pct_error = 100.0 * bias / truth if np.isfinite(truth) and truth != 0 else np.nan
        median_pct_error = 100.0 * (median - truth) / truth if np.isfinite(truth) and truth != 0 else np.nan
        rmse = float(np.sqrt(np.mean(errors**2)))
        mae = float(np.mean(np.abs(errors)))

    return {
        "Quantity": quantity, "Estimator": estimator, "VaR level": float(level), "Theory": truth,
        "Bootstrap mean": mean, "Bootstrap median": median, "Std": std, "Bias": bias,
        "% error": pct_error, "Median % error": median_pct_error, "RMSE": rmse, "MAE": mae,
        "NaN count": nan_count, "NaN (screen-rejected)": n_rejected, "NaN (fit error)": n_error, "NaN (other)": n_other,
    }


def _method_order(df, method_names):
    available = list(df["method"].dropna().unique())
    registered = [name for name in method_names if name in available]
    return registered + [name for name in available if name not in registered]


def build_summary_tables(result: BootstrapResult) -> dict[str, pd.DataFrame]:
    """One consolidated set of bootstrap results tables, filtered by each
    registered method's declared `reports` capability."""
    _, _, specs = build_methods_and_train_on(result.config.run.methods)
    point_df, tail_df = result.point_df, result.tail_df
    built = result.built

    # Per-(method, subsample_id) fit status ("ok"/"rejected"/"error"), used
    # to split each method's "NaN count" below by cause -- None (and so
    # every method falling back to `_status_for`'s None) when this result
    # was built with `collect_diagnostics=False` or loaded from a cache that
    # predates the diagnostics table.
    status_by_method = (
        {m: g.set_index("subsample_id")["status"] for m, g in result.diagnostics.groupby("method")}
        if result.diagnostics is not None else {}
    )

    def _status_for(method, rows_at):
        lookup = status_by_method.get(method)
        if lookup is None:
            return None
        return rows_at["subsample_id"].map(lookup).to_numpy()

    levels = np.sort(point_df["var_level"].unique().astype(float))
    hazard_truth = np.asarray(built.hazard_dist.ppf(levels), dtype=float)
    loss_var_truth = np.asarray(built.loss_dist.ppf(levels), dtype=float)
    loss_es_truth = np.asarray(built.loss_dist.expected_shortfall(levels), dtype=float)

    sample_metrics = point_df[["subsample_id", "var_level", "h_var_sub", "loss_var_sub"]].drop_duplicates(subset=["subsample_id", "var_level"])
    sample_es = tail_df[["subsample_id", "var_level", "loss_es_sub"]].drop_duplicates(subset=["subsample_id", "var_level"])

    method_names = list(specs)
    pointwise_methods = [m for m in method_names if "pointwise" in specs[m].reports]
    tail_methods = [m for m in method_names if "tail" in specs[m].reports]
    marginal_methods = [m for m in method_names if "marginal" in specs[m].reports]

    records = []
    for j, level in enumerate(levels):
        at = sample_metrics["var_level"].eq(level)
        records.append(_summary_record("Hazard VaR", "Empirical subsample", level, sample_metrics.loc[at, "h_var_sub"], hazard_truth[j]))
    for j, level in enumerate(levels):
        at = sample_metrics["var_level"].eq(level)
        records.append(_summary_record("Loss VaR", "Empirical subsample", level, sample_metrics.loc[at, "loss_var_sub"], loss_var_truth[j]))

    for method in _method_order(point_df[point_df["method"].isin(marginal_methods)], marginal_methods):
        rows = point_df[point_df["method"].eq(method)]
        for j, level in enumerate(levels):
            at_rows = rows.loc[rows["var_level"].eq(level)]
            records.append(_summary_record("Loss VaR", method, level, at_rows["damage_var_sub"], loss_var_truth[j],
                                            status=_status_for(method, at_rows)))

    for method in _method_order(point_df[point_df["method"].isin(pointwise_methods)], pointwise_methods):
        rows = point_df[point_df["method"].eq(method)]
        for level in levels:
            at_rows = rows.loc[rows["var_level"].eq(level)]
            truth = float(at_rows["true_full"].iloc[0])
            records.append(_summary_record("Pointwise conditional expectation", method, level, at_rows["prediction"], truth,
                                            status=_status_for(method, at_rows)))

    for j, level in enumerate(levels):
        at = sample_es["var_level"].eq(level)
        records.append(_summary_record("Loss ES", "Empirical subsample", level, sample_es.loc[at, "loss_es_sub"], loss_es_truth[j]))

    for method in _method_order(tail_df[tail_df["method"].isin(marginal_methods)], marginal_methods):
        rows = tail_df[tail_df["method"].eq(method)]
        for j, level in enumerate(levels):
            at_rows = rows.loc[rows["var_level"].eq(level)]
            records.append(_summary_record("Loss ES", method, level, at_rows["damage_es_sub"], loss_es_truth[j],
                                            status=_status_for(method, at_rows)))

    sample_tail_mean = tail_df[["subsample_id", "var_level", "tail_mean_sub"]].drop_duplicates(subset=["subsample_id", "var_level"])
    for j, level in enumerate(levels):
        at = sample_tail_mean["var_level"].eq(level)
        truth = float(tail_df.loc[tail_df["var_level"].eq(level), "true_full"].iloc[0])
        records.append(_summary_record("Tail conditional expectation", "Empirical subsample", level, sample_tail_mean.loc[at, "tail_mean_sub"], truth))

    for method in _method_order(tail_df[tail_df["method"].isin(tail_methods)], tail_methods):
        rows = tail_df[tail_df["method"].eq(method)]
        for level in levels:
            at_rows = rows.loc[rows["var_level"].eq(level)]
            truth = float(at_rows["true_full"].iloc[0])
            records.append(_summary_record("Tail conditional expectation", method, level, at_rows["prediction"], truth,
                                            status=_status_for(method, at_rows)))

    results_table = pd.DataFrame.from_records(records)
    return {
        "Hazard — VaR": results_table.loc[results_table["Quantity"].eq("Hazard VaR")],
        "Losses — VaR": results_table.loc[results_table["Quantity"].eq("Loss VaR")],
        "Losses — Pointwise conditional expectation": results_table.loc[results_table["Quantity"].eq("Pointwise conditional expectation")],
        "Losses — ES": results_table.loc[results_table["Quantity"].eq("Loss ES")],
        "Losses — Tail conditional expectation": results_table.loc[results_table["Quantity"].eq("Tail conditional expectation")],
    }


def display_summary_tables(tables: dict[str, pd.DataFrame]):
    for title, table in tables.items():
        print(f"\n{'-' * 16} {title} {'-' * 16}")
        display(table.drop(columns="Quantity").reset_index(drop=True))


def _slugify(title):
    return re.sub(r"[^a-z0-9]+", "_", title.lower()).strip("_")


def save_results(result: BootstrapResult, out_dir: Path, *, cache_tag_suffix: str = "") -> dict[str, Path]:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    tag = result.config.cache_tag() + cache_tag_suffix
    paths = {
        "pointwise": out_dir / f"pointwise_{tag}.csv",
        "tail": out_dir / f"tail_{tag}.csv",
    }
    result.point_df.to_csv(paths["pointwise"], index=False)
    result.tail_df.to_csv(paths["tail"], index=False)
    table_dir = out_dir / f"tables_{tag}"
    table_dir.mkdir(parents=True, exist_ok=True)
    for title, table in build_summary_tables(result).items():
        table.drop(columns="Quantity").reset_index(drop=True).to_csv(table_dir / f"{_slugify(title)}.csv", index=False)
    return paths


def load_or_run_bootstrap_study(
    config: "StudyConfig", results_dir: Path = RESULTS_DIR, *, cache_tag_suffix: str = "",
) -> tuple[BootstrapResult, bool]:
    """Like `run_bootstrap_study`, but skips the expensive fitting pass if
    this scenario's results are already saved on disk (`pointwise_<tag>.csv`
    / `tail_<tag>.csv` under `results_dir`). Returns `(result, from_cache)`.

    Only the fitting pass is skipped here -- population/subsample generation
    already has its own independent cache (`generate_population_and_subsamples`,
    HDF5-backed). Both caches are bypassed together by `config.run.rebuild_cache
    = True`, the same flag that already governs the population cache.

    `cache_tag_suffix` is appended verbatim to `config.cache_tag()` to build the
    cache filename, mirroring `run_bootstrap_study`/`generate_population_and_subsamples`'s
    own `cache_tag_suffix`/`tag_suffix` parameters. `StudyConfig.cache_tag()` only
    encodes distribution *names* and `N`/`n`/`iterations`/`bins` -- never
    `hazard.params`/`loss.params`/`copula.params`, `seed`, `var_levels`, or
    `methods` -- so two configs differing only in one of those will silently
    share a cache file unless disambiguated by hand via this suffix. The one
    partial mitigation already in place is for `methods`/`var_levels`
    specifically: a cache hit is only trusted if the loaded tables actually
    cover the methods/levels this config asks for -- otherwise (e.g. a method
    was added since the cache was written) this falls back to a fresh run
    rather than silently returning incomplete results.
    """
    results_dir = Path(results_dir)
    tag = config.cache_tag() + cache_tag_suffix
    pointwise_path = results_dir / f"pointwise_{tag}.csv"
    tail_path = results_dir / f"tail_{tag}.csv"

    if not config.run.rebuild_cache and pointwise_path.exists() and tail_path.exists():
        point_df = pd.read_csv(pointwise_path)
        tail_df = pd.read_csv(tail_path)

        expected_methods = {m.name for m in config.run.methods}
        expected_levels = set(np.round(np.asarray(config.run.var_levels, dtype=float), 6))
        cached_methods = set(point_df["method"].unique()) | set(tail_df["method"].unique())
        cached_levels = set(np.round(point_df["var_level"].unique(), 6)) & set(np.round(tail_df["var_level"].unique(), 6))

        if expected_methods.issubset(cached_methods) and expected_levels.issubset(cached_levels):
            built = build_scenario(config.scenario)
            population, _ = generate_population_and_subsamples(config, built, tag_suffix=cache_tag_suffix)
            h_theory_true = built.hazard_dist.ppf(np.asarray(config.run.var_levels, dtype=float))
            print(f"Loaded cached results for {tag} from {results_dir} (skipped re-fitting).")
            result = BootstrapResult(
                config=config, built=built, population=population, h_theory_true=h_theory_true,
                output={}, point_df=point_df, tail_df=tail_df, diagnostics=None,
            )
            return result, True

        print(f"Cached results for {tag} don't cover the current methods/var_levels; re-running.")

    return run_bootstrap_study(config, cache_tag_suffix=cache_tag_suffix), False


### Plotting: the joint sample and grouped boxplots

`plot_joint_sample` is a scatter of the raw $(H,L)$ population with the
hazard/loss VaR thresholds overlaid, a quick visual check that the copula
and margins produced the intended dependence structure. `_plot_grouped_boxplots`
draws one box per (method, VaR level) pair — box = interquartile range across
the $S$ bootstrap estimates, whiskers at the $2.5$th/$97.5$th percentiles
(i.e. a $95\%$ empirical interval, not $\pm1.5\,\mathrm{IQR}$), mean marked
by a circle, and the exact theoretical value marked by an $\times$ — so
both *bias* (box offset from the $\times$) and *dispersion* (box/whisker
width) are visible in the same figure. `plot_grouped_boxplots` wraps this for
either the pointwise or tail target, restricting the method list to those
whose `MethodSpec.reports` actually includes that target.


In [ ]:
def plot_joint_sample(result: BootstrapResult, var_levels=(0.95, 0.99, 0.999, 0.9999)):
    """Joint hazard-loss sample and marginal VaR thresholds."""
    population, built, cfg = result.population, result.built, result.config
    fig, ax = plt.subplots(figsize=(9.2, 5.8), facecolor="white")
    ax.scatter(population.hazard, population.losses, s=3.0, alpha=0.15, color="#3f3f3f", linewidths=0, rasterized=True, label="Full sample")
    for q in var_levels:
        ax.axvline(built.hazard_dist.ppf(q), color=REFERENCE_COLORS["Hazard"], linestyle=(0, (6, 4)), linewidth=1.0, alpha=0.72,
                   label=rf"$\mathrm{{VaR}}_{{{q:g}}}(H)$")
    for q in var_levels:
        ax.axhline(built.loss_dist.ppf(q), color=REFERENCE_COLORS["Loss"], linestyle=(0, (1.2, 2.4)), linewidth=1.0, alpha=0.72,
                   label=rf"$\mathrm{{VaR}}_{{{q:g}}}(L)$")
    ax.set_xlabel(r"Hazard $H$", labelpad=8)
    ax.set_ylabel(r"Loss $L$", labelpad=10)
    ax.set_title(rf"Joint sample of $H$ and $L$ ({cfg.scenario.copula.name} copula, {cfg.scenario.loss.name} losses)", fontsize=14.0, pad=14)
    ax.legend(loc="upper left", fontsize=10.7, ncol=2, handlelength=1.8, columnspacing=1.0, handletextpad=0.5)
    fig.tight_layout(pad=0.7)
    save_figure(fig, f"joint_sample_{cfg.cache_tag()}")
    plt.show()


def _plot_grouped_boxplots(ax, df, method_names, ylabel, title):
    levels = np.sort(df["var_level"].unique().astype(float))
    palette = _palette_for(method_names)
    group_centers = np.arange(len(levels), dtype=float)
    offsets = np.linspace(-0.30, 0.30, len(method_names)) if len(method_names) > 1 else np.array([0.0])
    width = 0.65 / max(len(method_names), 1)

    for method_index, method in enumerate(method_names):
        distributions, positions = [], []
        for level_index, level in enumerate(levels):
            values = df.loc[df["method"].eq(method) & df["var_level"].eq(level), "prediction"].to_numpy(dtype=float)
            values = values[np.isfinite(values)]
            if values.size:
                distributions.append(values)
                positions.append(group_centers[level_index] + offsets[method_index])
        ax.boxplot(
            distributions, positions=positions, widths=width, patch_artist=True, showmeans=True, showfliers=False,
            manage_ticks=False, whis=(2.5, 97.5),
            boxprops={"facecolor": palette[method], "alpha": 0.20, "edgecolor": palette[method], "linewidth": 1.15},
            whiskerprops={"color": palette[method], "linewidth": 1.0}, capprops={"color": palette[method], "linewidth": 1.0},
            medianprops={"color": METHOD_COLORS["Truth"], "linewidth": 1.8},
            meanprops={"marker": "o", "markerfacecolor": "white", "markeredgecolor": palette[method], "markeredgewidth": 1.0, "markersize": 5},
        )

    truth = np.array([float(df.loc[df["var_level"].eq(level), "true_full"].iloc[0]) for level in levels])
    ax.scatter(group_centers, truth, color=METHOD_COLORS["Truth"], marker="X", s=65, edgecolor="white", linewidth=0.7, zorder=4)
    ax.set_xticks(group_centers)
    ax.set_xticklabels(_q_to_pct_labels(levels))
    ax.set_xlabel(r"Hazard quantile level $q$", labelpad=8)
    ax.set_ylabel(ylabel, labelpad=10)
    ax.set_title(title, fontsize=13.0, pad=12)
    # Log scale: VaR levels span multiple orders of magnitude (heavier right
    # tail at higher q), so a linear axis squashes the low-q boxes flat
    # against zero and they look like they've disappeared. All values here
    # are strictly positive losses, so log scale is always safe.
    ax.set_yscale("log")
    ax.margins(y=0.08)
    return palette


def plot_grouped_boxplots(result: BootstrapResult, target: Literal["pointwise", "tail"] = "pointwise", ax=None):
    """Grouped boxplots of every method's estimates by VaR level, against theory."""
    _, _, specs = build_methods_and_train_on(result.config.run.methods)
    df = result.point_df if target == "pointwise" else result.tail_df
    key = "pointwise" if target == "pointwise" else "tail"
    method_names = _plot_methods([m for m in specs if key in specs[m].reports])

    if ax is not None:
        return _plot_grouped_boxplots(ax, df, method_names,
                                      ylabel=r"$\mathbb{E}[L\mid H]$",
                                      title=f"{'Pointwise' if target == 'pointwise' else 'Tail'} conditional mean")

    fig, ax = plt.subplots(figsize=(7.5, 4.9), facecolor="white")
    cfg = result.config
    label = r"\mathbb{E}[L\mid H=\mathrm{VaR}_{q}(H)]" if target == "pointwise" else r"\mathbb{E}[L\mid H\geq \mathrm{VaR}_{q}(H)]"
    palette = _plot_grouped_boxplots(ax, df, method_names, ylabel=rf"${label}$",
                                     title=rf"${label}$ ({cfg.scenario.copula.name} copula, {cfg.scenario.loss.name} losses)")
    legend_handles = [Patch(facecolor=palette[name], edgecolor=palette[name], alpha=0.20, label=name) for name in method_names] + [
        Line2D([0], [0], color=METHOD_COLORS["Truth"], linewidth=1.8, label="Median"),
        Line2D([0], [0], marker="o", linestyle="none", markerfacecolor="white", markeredgecolor=METHOD_COLORS["Truth"], markersize=5, label="Mean"),
        Line2D([0], [0], marker="X", linestyle="none", color=METHOD_COLORS["Truth"], markeredgecolor="white", markersize=8, label="Theory"),
    ]
    fig.legend(handles=legend_handles, loc="lower center", bbox_to_anchor=(0.5, -0.05), ncol=min(len(legend_handles), 7), frameon=False, handlelength=1.7, columnspacing=1.2)
    fig.tight_layout(rect=(0, 0.12, 1, 1), pad=0.8)
    save_figure(fig, f"grouped_boxplots_{target}_{result.config.cache_tag()}")
    plt.show()


def plot_sanity_check(result: BootstrapResult):
    """Full-sample empirical vs. theoretical tail-expectation check."""
    cfg = result.config
    h_theory = result.h_theory_true
    true_tail = np.asarray(result.built.targets.tail_mean(h_theory), dtype=float)
    var_levels_check = np.asarray(result.config.run.var_levels, dtype=float)
    loss_full = result.population.losses.astype(float)
    hazard_full = result.population.hazard.astype(float)

    empirical_tail, tail_counts = [], []
    for threshold in h_theory:
        in_tail = hazard_full >= threshold
        tail_counts.append(int(in_tail.sum()))
        empirical_tail.append(float(np.mean(loss_full[in_tail])) if np.any(in_tail) else np.nan)
    empirical_tail = np.asarray(empirical_tail, dtype=float)
    difference = empirical_tail - true_tail
    relative_difference = np.where(np.abs(true_tail) > 0, 100.0 * difference / true_tail, np.nan)

    sanity_check_table = pd.DataFrame({
        "VaR level": var_levels_check, "Hazard threshold": h_theory, "Full-sample tail count": tail_counts,
        "Empirical tail expectation": empirical_tail, "Theoretical tail expectation": true_tail,
        "Difference": difference, "% difference": relative_difference,
    })
    display(sanity_check_table)

    x = np.arange(len(var_levels_check), dtype=float)
    fig, ax = plt.subplots(figsize=(8.2, 4.9), facecolor="white")
    ax.plot(x, true_tail, marker="X", markersize=8, color=REFERENCE_COLORS["Truth"], markeredgecolor="white", markeredgewidth=0.7, label="Theory", zorder=3)
    ax.plot(x, empirical_tail, marker="o", markersize=5.5, color=REFERENCE_COLORS["Empirical"], markeredgecolor="white", markeredgewidth=0.7, label="Empirical full sample", zorder=4)
    for index, count in enumerate(tail_counts):
        ax.annotate(rf"$n={count:,}$", (x[index], max(true_tail[index], empirical_tail[index])), xytext=(0, 7), textcoords="offset points", fontsize=11.0, color="0.35", ha="center")
    ax.set_xticks(x)
    ax.set_xticklabels(_q_to_pct_labels(var_levels_check))
    ax.set_xlabel(r"Hazard quantile level $q$", labelpad=8)
    ax.set_ylabel(r"$\mathbb{E}[L\mid H\geq \mathrm{VaR}_q(H)]$", labelpad=10)
    ax.set_title(rf"Full-sample validation $(N={len(hazard_full):,})$ ({cfg.scenario.copula.name} copula, {cfg.scenario.loss.name} losses)", fontsize=14.0, pad=14)
    ax.legend(loc="best", handlelength=1.8)
    ax.margins(y=0.12)
    fig.tight_layout(pad=0.7)
    save_figure(fig, f"sanity_check_{result.config.cache_tag()}")
    plt.show()
    return sanity_check_table


### 9b. Three-panel grouped boxplots (pointwise, tail, marginal ES)

`plot_grouped_boxplots` above draws one target (pointwise or tail) per call.
This combines pointwise, tail, **and** marginal Loss ES into one figure with
a single shared legend -- an estimator that appears in more than one panel
keeps the same color and horizontal slot in each, and the empirical-subsample
baseline (which has no pointwise/tail curve of its own) still contributes its
box to the ES panel, since $\mathrm{ES}_q(L)$ is a marginal quantity every
method reports.

Building the ES panel reuses Part 8's `tail_df` directly, rather than a
separate helper: it already carries each marginal-reporting method's
`damage_es_sub` alongside its `prediction` column, and the empirical
baseline's own `loss_es_sub`, so building the panel is just reshaping those
two into the same `(method, var_level, prediction, true_full)` shape
`_plot_grouped_boxplots` already expects.

**Cell below:** `plot_three_panel_boxplots`.


In [ ]:
def plot_three_panel_boxplots(result: "BootstrapResult"):
    """Pointwise, tail, and marginal-ES grouped boxplots side by side, with
    one legend shared across all three panels."""
    built = result.built
    cfg = result.config
    _, _, specs = build_methods_and_train_on(result.config.run.methods)
    point_df, tail_df = result.point_df, result.tail_df

    pointwise_methods = _plot_methods([m for m in specs if "pointwise" in specs[m].reports])
    tail_methods = _plot_methods([m for m in specs if "tail" in specs[m].reports])
    marginal_methods = _plot_methods([m for m in specs if "marginal" in specs[m].reports])
    panel_names = list(dict.fromkeys(pointwise_methods + tail_methods + marginal_methods))

    levels = np.sort(point_df["var_level"].unique().astype(float))
    loss_es_truth = dict(zip(levels, np.asarray(built.loss_dist.expected_shortfall(levels), dtype=float)))

    sample_es = tail_df[["subsample_id", "var_level", "loss_es_sub"]].drop_duplicates(subset=["subsample_id", "var_level"])
    es_parts = [sample_es.rename(columns={"loss_es_sub": "prediction"}).assign(method="Empirical subsample")]
    for method in marginal_methods:
        rows = tail_df[tail_df["method"].eq(method)][["method", "var_level", "damage_es_sub"]].rename(columns={"damage_es_sub": "prediction"})
        es_parts.append(rows)
    es_df = pd.concat(es_parts, ignore_index=True)
    es_df["true_full"] = es_df["var_level"].map(loss_es_truth)
    es_names = ["Empirical subsample"] + [m for m in panel_names if m in marginal_methods]

    fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.8), facecolor="white")
    _plot_grouped_boxplots(axes[0], point_df, [m for m in panel_names if m in pointwise_methods],
                          r"$\mathbb{E}[L\mid H=\mathrm{VaR}_{q}(H)]$", "Pointwise conditional mean")
    _plot_grouped_boxplots(axes[1], tail_df, [m for m in panel_names if m in tail_methods],
                          r"$\mathbb{E}[L\mid H\geq \mathrm{VaR}_{q}(H)]$", "Tail conditional mean")
    _plot_grouped_boxplots(axes[2], es_df, es_names, r"$\mathrm{ES}_q(L)$", "Loss ES")

    palette = _palette_for(panel_names + ["Empirical subsample"])
    handles = [Patch(facecolor=palette[name], edgecolor=palette[name], alpha=0.20, label=name) for name in panel_names]
    handles.append(Patch(facecolor=palette["Empirical subsample"], edgecolor=palette["Empirical subsample"], alpha=0.20, label="Empirical subsample"))
    handles.append(Line2D([0], [0], marker="X", linestyle="none", color=METHOD_COLORS["Truth"], markeredgecolor="white", markersize=8, label="Truth"))
    fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, -0.06), ncol=min(len(handles), 8),
              frameon=False, fontsize=11.1, handlelength=1.6, columnspacing=1.1)
    fig.suptitle(f"Grouped boxplots by VaR level ({cfg.scenario.copula.name} copula, {cfg.scenario.loss.name} losses)", fontsize=14.0, y=1.04)
    fig.tight_layout(rect=(0, 0.12, 1, 0.94), pad=0.8)
    save_figure(fig, f"three_panel_boxplots_{result.config.cache_tag()}")
    plt.show()


### Nonparametric copula validation

An independent check that the *simulated* population actually reproduces the
*intended* copula. The Capéraà–Fougères–Genest estimator recovers $A(t)$
nonparametrically from
pseudo-observations $(u_i,v_i)$, with $S_i=-\log u_i$, $T_i=-\log v_i$,
$\zeta_i(t)=\min\!\big(S_i/t,\;T_i/(1-t)\big)$, and Euler–Mascheroni constant
$\gamma\approx0.5772$:
$$\hat A(t)=\exp\!\Big(-\gamma-\operatorname{mean}_i\log\zeta_i(t)\Big)\Big/\hat A(\tfrac12)_{\text{norm.}}.$$
For Gumbel, Kendall's $\tau$ has the closed form $\tau=1-1/\theta$, checked
against the sample $\tau$. Upper-tail dependence
$\lambda_u=\lim_{q\to1}P(V>q\mid U>q)=2-2A(\tfrac12)$ is checked empirically at
$q\in\{0.99,0.999\}$ — it approaches its limit slowly from below, so a
sample value a few percent under the theoretical one at $q=0.999$ is expected
and is not by itself a sign of a sampling error. Generic to any registered
copula, not Gumbel-specific.


In [ ]:
# ---------------------------------------------------------------------------
# Nonparametric copula-validation diagnostics: the Capera-Fougeres-Genest
# estimate of A(t) against the theoretical Pickands function, empirical
# Kendall's tau against
# its theoretical value (Gumbel only, in closed form), and empirical vs.
# theoretical upper-tail dependence. Generic to any registered EV copula, not
# Gumbel-specific -- called once per run as a population sanity check.
# ---------------------------------------------------------------------------

_EULER = 0.5772156649015329


def pickands_cfg(u, v, t):
    """Capera-Fougeres-Genest estimate of A(t), with endpoint correction."""
    S = -np.log(np.clip(u, 1e-300, 1 - 1e-16))
    T = -np.log(np.clip(v, 1e-300, 1 - 1e-16))
    A_e1 = np.exp(-_EULER - np.mean(np.log(S)))
    out = []
    for tt in np.atleast_1d(np.asarray(t, float)):
        zeta = np.minimum(S / max(tt, 1e-12), T / max(1.0 - tt, 1e-12))
        out.append(np.exp(-_EULER - np.mean(np.log(zeta))) / A_e1)
    return np.asarray(out)


def validate_copula_sample(result: "BootstrapResult", n_check: int = 20_000):
    """Compare the simulated population against the theoretical copula."""
    built = result.built
    u_pop = built.hazard_dist.cdf(result.population.hazard)
    v_pop = built.loss_dist.cdf(result.population.losses)
    u_check, v_check = u_pop[:n_check], v_pop[:n_check]

    t_grid = np.array([0.1, 0.25, 0.5, 0.75, 0.9])
    tau = stats.kendalltau(u_check, v_check).statistic
    table = pd.DataFrame({"t": t_grid, "A(t) theoretical": built.pickands.A(t_grid), "A(t) estimated (CFG)": pickands_cfg(u_pop, v_pop, t_grid)})
    display(table)

    tau_msg = f"Kendall tau: sample {tau:.4f}"
    if result.config.scenario.copula.name.lower() == "gumbel":
        theta = result.config.scenario.copula.params.get("theta", 2.0)
        tau_msg += f", theoretical 1 - 1/theta = {1 - 1 / theta:.4f}"
    print(tau_msg)

    for q in (0.99, 0.999):
        lam = np.mean((u_pop > q) & (v_pop > q)) / (1.0 - q)
        print(f"upper tail dependence at level {q}: empirical {lam:.4f}, limit 2 - 2A(1/2) = {built.pickands.upper_tail_dependence():.4f}")


### 9c. Upper-tail damage-curve comparison

Every plot so far summarizes an estimator's bootstrap distribution at a
handful of discrete VaR levels (the grouped boxplots above) or its % error
against a swept parameter (Part 10c). This section instead shows what a
fitted damage function actually *looks like* as a curve over the hazard
axis, zoomed into the upper tail, alongside the theoretical curve
$d(h)=\mathbb E[L\mid H=h]$ (Part 6).

For one representative subsample (drawn once per call, fixed by
`subsample_seed`), a chosen subset of the pointwise-reporting methods
(`active_methods`, default just `"Sigmoid"` -- pass e.g.
`active_methods=("Sigmoid", "Isotonic")` to compare more) is fit and plotted
as a curve, each end-labeled directly on the line as well as in the legend.
Two VaR($H$) reference positions are marked: $\widehat{\mathrm{VaR}}_q(H)$,
the *subsample's own* empirical hazard quantile (what a single analyst's
dataset would actually estimate -- each fitted curve gets a marker/line at
its value there), and the true $\mathrm{VaR}_q(H)$ (diamond, on the
theoretical curve). Bootstrap uncertainty is shown as two shaded ribbons per
method -- the $95\%$ interval and the interquartile range across **all**
bootstrap subsamples, from the same `point_df` the grouped boxplots and
`build_summary_tables` already use, re-aggregated per `(method, var_level)`
here and connected across levels into a ribbon instead of a full
box-and-whisker plot.

**Cell below:** `_direct_label_line`, `_pointwise_bootstrap_interval` (the
per-`(method, var_level)` quantile summary), and
`plot_upper_tail_comparison`.


In [ ]:
_UPPER_TAIL_AVAILABLE_METHODS = ("Quadratic", "Sigmoid", "Isotonic", "Isotonic (raw)", "IDR")


def _direct_label_line(ax, x, y, label, color, x_pad_frac=0.012, y_offset=0.0, fontsize=12.1):
    """Place `label` directly at the right end of a plotted curve."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    if not np.any(ok):
        return
    x_ok, y_ok = x[ok], y[ok]
    x_span = max(float(np.max(x_ok) - np.min(x_ok)), 1e-9)
    ax.text(x_ok[-1] + x_pad_frac * x_span, y_ok[-1] + y_offset, label, color=color,
           fontsize=fontsize, va="center", ha="left", clip_on=False)


def _pointwise_bootstrap_interval(point_df: pd.DataFrame) -> pd.DataFrame:
    """Per-(method, var_level) quantile summary of the bootstrap `prediction`
    column, positioned at that level's mean empirical hazard-VaR.

    Always grouped by (method, var_level) together -- collapsing across
    var_level (e.g. dropping duplicates keyed on method alone) silently
    picks an arbitrary level's rows, the mistake the sensitivity-sweep cell
    (Part 10c) made and had fixed earlier.
    """
    rows = []
    for (method, level), group in point_df.groupby(["method", "var_level"], sort=False):
        pred = group["prediction"].to_numpy(dtype=float)
        pred = pred[np.isfinite(pred)]
        h_sub = group["h_var_sub"].to_numpy(dtype=float)
        h_sub = h_sub[np.isfinite(h_sub)]
        if pred.size == 0 or h_sub.size == 0:
            continue
        q025, q25, q75, q975 = np.quantile(pred, [0.025, 0.25, 0.75, 0.975], method="inverted_cdf")
        rows.append(dict(method=method, var_level=float(level), h_var_sub_mean=float(h_sub.mean()),
                         q025=float(q025), q25=float(q25), q75=float(q75), q975=float(q975), n=int(pred.size)))
    return pd.DataFrame(rows).sort_values(["method", "h_var_sub_mean"])


def plot_upper_tail_comparison(result: "BootstrapResult", *, active_methods: tuple[str, ...] = ("Sigmoid",),
                               q_highlight: float = 0.99, zoom_q: float = 0.95,
                               show_95_interval: bool = True, show_iqr: bool = True,
                               n_grid: int = 400, subsample_seed: int = 0):
    """Fitted damage curve(s) near the upper tail, with bootstrap-uncertainty
    ribbons from the pointwise results table (Part 8's `point_df`)."""
    cfg, built, population = result.config, result.built, result.population
    rng = np.random.default_rng(subsample_seed)
    idx = rng.choice(population.hazard.size, size=cfg.run.n, replace=False)
    h, y = population.hazard[idx], population.losses[idx]

    h_zoom_min = 0.95 * float(built.hazard_dist.ppf(zoom_q))
    intervals = _pointwise_bootstrap_interval(result.point_df)
    active_intervals = intervals[intervals["method"].isin(active_methods) & (intervals["h_var_sub_mean"] >= h_zoom_min)]
    if not active_intervals.empty:
        h_zoom_max = float(active_intervals["h_var_sub_mean"].max())
    else:
        h_zoom_max = float(built.hazard_dist.ppf(max(cfg.run.var_levels)))
    h_zoom_max += 0.03 * (h_zoom_max - h_zoom_min)
    h_grid = np.linspace(h_zoom_min, h_zoom_max, n_grid)

    xb, yb = bin_summary(np.column_stack([h, y]), num_bins=cfg.run.bins, summary_func=cfg.run.bin_summary_func, bin_x=cfg.run.bin_x)
    ok = np.isfinite(xb) & np.isfinite(yb)
    xb, yb = xb[ok], yb[ok]

    methods, train_on, _ = build_methods_and_train_on(MethodConfig(name) for name in active_methods)
    predictors, _ = _fit_predictors_for_subsample(methods, train_on, h, y, xb, yb)
    curves = {name: np.asarray(predictors[name].predict(h_grid), dtype=float)
             for name in active_methods if predictors.get(name) is not None and not getattr(predictors[name], "failed", False)}
    truth_curve = built.targets.d(h_grid)

    palette = _palette_for(active_methods)

    fig, ax = plt.subplots(figsize=(8.4, 5.5), facecolor="white")
    zoom = (h >= h_zoom_min) & (h <= h_zoom_max)
    ax.scatter(h[zoom], y[zoom], s=5.0, alpha=0.4, color="#3f3f3f", linewidths=0, rasterized=True, label="Subsample data", zorder=1.0)
    xb_zoom = (xb >= h_zoom_min) & (xb <= h_zoom_max)
    ax.scatter(xb[xb_zoom], yb[xb_zoom], s=22, marker="o", facecolor="white", edgecolor="0.18", linewidths=0.8, alpha=0.9,
              label="Subsample binned data", zorder=3.1)

    band_widths = {}
    for name in active_methods:
        rows = intervals[intervals["method"].eq(name)]
        band_widths[name] = float((rows["q975"] - rows["q025"]).mean()) if not rows.empty else -np.inf
    band_order = sorted(active_methods, key=lambda name: band_widths[name], reverse=True)
    for name in band_order:
        rows = intervals[intervals["method"].eq(name)].sort_values("h_var_sub_mean")
        if rows.empty:
            continue
        color = palette[name]
        if show_95_interval:
            ax.fill_between(rows["h_var_sub_mean"], rows["q025"], rows["q975"], color=color, alpha=0.10, linewidth=0, zorder=1.55)
        if show_iqr:
            ax.fill_between(rows["h_var_sub_mean"], rows["q25"], rows["q75"], color=color, alpha=0.22, linewidth=0, zorder=1.75)

    h_sub_var = float(np.quantile(h, q_highlight, method="inverted_cdf"))
    if h_zoom_min <= h_sub_var <= h_zoom_max:
        ax.axvline(h_sub_var, color="0.5", linewidth=1.0, linestyle=(0, (6, 4)), alpha=0.7, zorder=2.2)
        ax.text(h_sub_var, 0.985, rf"$\widehat{{\mathrm{{VaR}}}}_{{{q_highlight:.2f}}}(H)$", transform=ax.get_xaxis_transform(),
               rotation=90, ha="right", va="top", fontsize=12.1, color="0.4", zorder=4.0)

    for name, curve in curves.items():
        color = palette[name]
        ax.plot(h_grid, curve, color=color, linewidth=2.0, zorder=3.6, label=name)
        _direct_label_line(ax, h_grid, curve, name, color)
        if h_zoom_min <= h_sub_var <= h_zoom_max:
            var_y = float(predictors[name].predict(np.array([h_sub_var]))[0])
            ax.axhline(var_y, color=color, linewidth=1.0, linestyle=(0, (6, 4)), alpha=0.7, zorder=2.45)
            ax.scatter([h_sub_var], [var_y], marker="o", s=36, facecolor=color, edgecolor="white", linewidths=0.75, zorder=4.5)

    ax.plot(h_grid, truth_curve, color=METHOD_COLORS["Truth"], linewidth=2.0, alpha=0.95, zorder=2.9, label="Theoretical value")
    _direct_label_line(ax, h_grid, truth_curve, r"True $\mathbb{E}[L\mid H=h]$", METHOD_COLORS["Truth"])
    h_true_var = float(built.hazard_dist.ppf(q_highlight))
    if h_zoom_min <= h_true_var <= h_zoom_max:
        ax.axvline(h_true_var, color=METHOD_COLORS["Truth"], linewidth=0.95, linestyle=(0, (1.2, 2.4)), alpha=0.7, zorder=2.35)
        ax.text(h_true_var, 0.80, rf"$\mathrm{{VaR}}_{{{q_highlight:.2f}}}(H)$", transform=ax.get_xaxis_transform(),
               rotation=90, ha="right", va="top", fontsize=12.1, color=METHOD_COLORS["Truth"], zorder=4.0)
        true_var_y = float(built.targets.d(np.array([h_true_var]))[0])
        ax.axhline(true_var_y, color=METHOD_COLORS["Truth"], linewidth=0.95, linestyle=(0, (1.2, 2.4)), alpha=0.7, zorder=2.35)
        ax.scatter([h_true_var], [true_var_y], marker="D", s=32, facecolor=METHOD_COLORS["Truth"], edgecolor="white", linewidths=0.75, zorder=4.6)

    ax.set_xlim(h_zoom_min, h_zoom_max)
    y_candidates = [float(np.nanquantile(y[zoom], 0.98, method="inverted_cdf"))] if np.any(zoom) else []
    y_candidates += [float(np.nanmax(c)) for c in curves.values() if np.any(np.isfinite(c))]
    y_candidates.append(float(np.nanmax(truth_curve)))
    for name in active_methods:
        rows = intervals[intervals["method"].eq(name)]
        if not rows.empty:
            y_candidates.append(float(rows["q975"].max()))
    if y_candidates:
        ax.set_ylim(0.0, max(y_candidates) * 1.10)

    ax.set_xlabel(r"Hazard $H$", labelpad=8)
    ax.set_ylabel(r"Loss $L$", labelpad=10)
    ax.set_title(rf"Upper-tail zoom of $\mathbb{{E}}[L\mid H=h]$ ({cfg.scenario.copula.name} copula, {cfg.scenario.loss.name} losses)",
                fontsize=14.0, pad=26)
    handles, labels = ax.get_legend_handles_labels()
    unique = dict(zip(labels, handles))
    ax.legend(unique.values(), unique.keys(), loc="lower left", bbox_to_anchor=(0.0, 1.01), fontsize=11.9,
             frameon=False, ncol=min(4, len(unique)), handlelength=1.5, columnspacing=1.0, handletextpad=0.45, borderaxespad=0.0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    fig.tight_layout(pad=0.6)
    save_figure(fig, f"upper_tail_comparison_{result.config.cache_tag()}")
    plt.show()


### 9d. Hazard-loss risk diagram

A single composite figure putting every marginal/conditional risk quantity
from Parts 6 and 8 on one picture: the joint sample, the theoretical damage
curve $d(h)=\mathbb E[L\mid H=h]$ (the one purely theoretical element; every
other quantity below is read directly off the sample), and four reference
points/lines marked directly on it -- $\mathrm{VaR}_q(H)$, $\mathrm{VaR}_q(L)$,
$d(\mathrm{VaR}_q(H))$ (diamond, on the curve), $\mathbb E[L\mid H\geq
\mathrm{VaR}_q(H)]$ (circle), and $\mathbb E[L\mid L\geq \mathrm{VaR}_q(L)]$
(star) -- plus, optionally (`show_conditional_tail_var=True`, off by
default), $\mathrm{VaR}_q(L\mid H\geq\mathrm{VaR}_q(H))$. The square marker
just marks where the two VaR lines cross. Note $\mathbb E[L\mid L\geq
\mathrm{VaR}_q(L)]$ is the plain empirical tail mean of $L$ -- a simple
descriptive quantity for this population picture, not the coherent
$\mathrm{ES}_q(L)$ used to benchmark estimators elsewhere in this notebook
(Part 4's `_es_from_sorted_atoms` convention), so the two shouldn't be
compared directly.

`mode="full"` additionally draws the marginal loss density $f_L$ (mirrored
into the panel to the left of the main axis), the tail-conditional loss
density $f_{L\mid H\geq \mathrm{VaR}_q(H)}$ (right), and the marginal hazard
density $f_H$ (bottom, with the region beyond $\mathrm{VaR}_q(H)$ shaded);
`mode="mainright"` keeps only the main panel and the conditional-loss-density
panel. All densities are Gaussian KDEs, boundary-reflected on the
$\log(1+L)$ scale for the loss densities (positive support). Each panel is
plotted on its own true density scale, not rescaled to a common axis, so peak heights differ across panels.

**Cell below:** `_reflected_log_kde` (boundary-aware density estimates,
used for all three density panels), `plot_hazard_loss_risk_diagram`, and
`plot_hazard_loss_risk_diagram_curves`.

**`plot_hazard_loss_risk_diagram_curves`** is a second version of the same
diagram where three of the four reference quantities become curves over the
hazard axis instead of one fixed-$q$ point: $\mathrm{VaR}_{F_H(h)}(L)$,
$\mathrm{ES}_{F_H(h)}(L)$, and $\mathbb E[L\mid H\geq h]$, each swept over
$q=F_H(h)$ as $h$ varies -- $d(h)$ is unchanged, it was already a curve.
Unlike the figure above, **every** quantity here -- the three curves, the
vertical $\mathrm{VaR}_q(H)$ line, and all four marker dots -- is the exact
theoretical value (Parts 4/6's closed-form/quadrature machinery: `ppf`,
`expected_shortfall`, `targets.d`/`.tail_mean`), not anything derived from
the population sample, so there's no sampling noise to read past. The joint
scatter and the KDE density side-panels stay empirical, same as above --
only the reference quantities move from "one point" to "the whole curve."
The star marker on this version is a genuine $\mathrm{ES}_q(L)$ (the
coherent superquantile convention), unlike the empirical star above.


In [ ]:
def _reflected_log_kde(values: np.ndarray, grid: np.ndarray, bw_adjust: float = 1.05) -> np.ndarray:
    """Gaussian KDE density on log1p(values), boundary-reflected at zero.

    Reflecting the log1p-transformed sample about zero before fitting the
    KDE removes the boundary bias a plain KDE would have near L=0 (positive
    support only); the density is mapped back to the L scale via the
    change-of-variables factor 1/(1+L).
    """
    values = np.asarray(values, dtype=float)
    grid = np.asarray(grid, dtype=float)
    mask = np.isfinite(values) & (values >= 0.0)
    lv = np.log1p(values[mask])
    out = np.zeros_like(grid, dtype=float)
    if lv.size < 2:
        return np.full_like(grid, np.nan, dtype=float)
    kde = gaussian_kde(np.concatenate([lv, -lv]))
    kde.set_bandwidth(kde.factor * bw_adjust)
    positive = grid >= 0.0
    lg = np.log1p(grid[positive])
    out[positive] = 2.0 * kde(lg) / (1.0 + grid[positive])
    return out


def _fmt_density(val: float) -> str:
    return f"{val:.2g}"


def plot_hazard_loss_risk_diagram(result: "BootstrapResult", *, q: float = 0.99,
                                  mode: Literal["full", "mainright"] = "full",
                                  show_conditional_tail_var: bool = False, n_grid: int = 900):
    """Full-sample hazard-loss risk diagram: joint scatter, the theoretical
    damage curve, both empirical marginal VaRs, and the population's own
    tail-mean quantities, marked directly on the figure. `mode="full"` adds
    marginal loss/hazard density panels and the hazard-tail-conditional loss
    density; `mode="mainright"` keeps only the main panel and that
    conditional density.
    """
    cfg, built, population = result.config, result.built, result.population
    h, y = population.hazard, population.losses

    col_blue = METHOD_COLORS["Quadratic"]  # hazard-tail-conditioned family
    col_loss = METHOD_COLORS["Sigmoid"]  # loss-tail family
    col_haz = METHOD_COLORS["Isotonic"]  # hazard marginal only
    col_cond_var = METHOD_COLORS["IDR"]
    col_scatter = "#3f3f3f"

    h_var = float(np.quantile(h, q, method="inverted_cdf"))
    l_var = float(np.quantile(y, q, method="inverted_cdf"))
    tail_mask = h >= h_var
    l_given_h_tail = y[tail_mask]
    expected_l_given_h_tail = float(l_given_h_tail.mean()) if l_given_h_tail.size else np.nan
    l_tail_mask = y >= l_var
    expected_l_given_l_tail = float(y[l_tail_mask].mean()) if np.any(l_tail_mask) else np.nan
    conditional_l_var = float(np.quantile(l_given_h_tail, q, method="inverted_cdf")) if l_given_h_tail.size else np.nan

    x_max = float(np.quantile(h, 0.997, method="inverted_cdf"))
    y_candidates = [float(np.quantile(y, 0.995, method="inverted_cdf")), 1.08 * expected_l_given_l_tail]
    if show_conditional_tail_var and np.isfinite(conditional_l_var):
        y_candidates.append(1.08 * conditional_l_var)
    y_max = max(y_candidates)

    h_grid = np.linspace(0.0, x_max, n_grid)
    damage_grid = np.linspace(0.0, x_max, min(n_grid, 800))
    damage_values = built.targets.d(damage_grid)
    damage_at_h_var = float(built.targets.d(np.array([h_var]))[0])

    l_grid = np.linspace(0.0, y_max, n_grid)
    l_density = _reflected_log_kde(y, l_grid)
    cond_l_density = _reflected_log_kde(l_given_h_tail, l_grid)
    h_density = _reflected_log_kde(h, h_grid)
    l_peak = float(np.nanmax(l_density)) if np.any(np.isfinite(l_density)) else 1.0
    cond_peak = float(np.nanmax(cond_l_density)) if np.any(np.isfinite(cond_l_density)) else 1.0
    h_peak = float(np.nanmax(h_density)) if np.any(np.isfinite(h_density)) else 1.0

    ann_fs = 12.1

    if mode == "full":
        fig = plt.figure(figsize=(11.6, 7.9), dpi=160, facecolor="white")
        gs = fig.add_gridspec(2, 3, width_ratios=[1.4, 5.45, 1.22], height_ratios=[5.05, 1.28], wspace=0.16, hspace=0.08)
        ax_left = fig.add_subplot(gs[0, 0])
        ax_main = fig.add_subplot(gs[0, 1])
        ax_right = fig.add_subplot(gs[0, 2], sharey=ax_main)
        ax_bottom = fig.add_subplot(gs[1, 1], sharex=ax_main)
    else:
        fig = plt.figure(figsize=(10.2, 6.2), dpi=160, facecolor="white")
        gs = fig.add_gridspec(1, 2, width_ratios=[5.45, 1.22], wspace=0.10)
        ax_main = fig.add_subplot(gs[0, 0])
        ax_right = fig.add_subplot(gs[0, 1], sharey=ax_main)
        ax_left = ax_bottom = None

    ax_main.axvspan(h_var, x_max, ymin=0, ymax=l_var / y_max, color=col_blue, alpha=0.035, lw=0, zorder=0)
    ax_main.axhspan(l_var, y_max, color=col_loss, alpha=0.045, lw=0, zorder=0)
    ax_main.fill_between([h_var, x_max], l_var, y_max, color="#b8a2ff", alpha=0.075, zorder=0)

    in_view = (h <= x_max) & (y <= y_max)
    ax_main.scatter(h[in_view], y[in_view], s=2.5, c=col_scatter, alpha=0.08, linewidths=0, rasterized=True, zorder=1)
    ax_main.plot(damage_grid, damage_values, color="black", linewidth=2.0, zorder=5)

    ax_main.axvline(h_var, color=col_haz, linewidth=1.2, linestyle=(0, (6, 4)), zorder=2)
    ax_main.axhline(l_var, color=col_loss, linewidth=1.1, linestyle=(0, (7, 5)), zorder=2)
    ax_main.axhline(damage_at_h_var, color="black", linewidth=1.0, linestyle=(0, (5, 4)), zorder=2)
    ax_main.axhline(expected_l_given_h_tail, color=col_blue, linewidth=1.2, linestyle=(0, (1.2, 2.4)), zorder=2)
    ax_main.axhline(expected_l_given_l_tail, color=col_loss, linewidth=1.2, linestyle=(0, (1.2, 2.4)), zorder=2)
    if show_conditional_tail_var:
        ax_main.axhline(conditional_l_var, color=col_cond_var, linewidth=1.35, linestyle=(0, (9, 4)), zorder=2)

    marker_kw = dict(markeredgecolor="white", markeredgewidth=0.8, zorder=8, clip_on=False)
    ax_main.plot(h_var, expected_l_given_l_tail, marker="*", markersize=14, color=col_loss, **marker_kw)
    ax_main.plot(h_var, expected_l_given_h_tail, marker="o", markersize=8.5, color=col_blue, **marker_kw)
    ax_main.plot(h_var, l_var, marker="s", markersize=8.0, color=col_loss, **marker_kw)
    ax_main.plot(h_var, damage_at_h_var, marker="D", markersize=7.5, color="black", **marker_kw)
    if show_conditional_tail_var:
        ax_main.plot(h_var, conditional_l_var, marker="^", markersize=9.5, color=col_cond_var, **marker_kw)

    x_label = 0.10 * x_max
    ax_main.text(x_label, expected_l_given_l_tail + 0.022 * y_max, r"$\mathbb{E}[L\mid L\geq \mathrm{VaR}_q(L)]$",
                color=col_loss, fontsize=ann_fs, ha="left", va="bottom", zorder=9)
    ax_main.text(x_label, expected_l_given_h_tail + 0.018 * y_max, r"$\mathbb{E}[L\mid H\geq \mathrm{VaR}_q(H)]$",
                color=col_blue, fontsize=ann_fs, ha="left", va="bottom", zorder=9)
    if show_conditional_tail_var:
        va = "bottom" if conditional_l_var < 0.92 * y_max else "top"
        y_txt = conditional_l_var + 0.018 * y_max if va == "bottom" else conditional_l_var - 0.018 * y_max
        ax_main.text(x_label, y_txt, r"$\mathrm{VaR}_q(L\mid H\geq \mathrm{VaR}_q(H))$",
                    color=col_cond_var, fontsize=ann_fs, ha="left", va=va, zorder=9)
    ax_main.annotate(r"$\mathrm{VaR}_q(L)$", xy=(x_label, l_var), xytext=(0, 8), textcoords="offset points",
                     color=col_loss, fontsize=ann_fs, ha="left", va="bottom", zorder=9)
    ax_main.annotate(r"$d(\mathrm{VaR}_q(H))$", xy=(x_label, damage_at_h_var), xytext=(0, -8), textcoords="offset points",
                     color="black", fontsize=ann_fs, ha="left", va="top", zorder=9)
    ax_main.text(h_var + 0.025 * x_max, y_max * 0.975, r"$\mathrm{VaR}_q(H)$", color=col_haz,
                fontsize=ann_fs, ha="left", va="top", rotation=90, zorder=9)

    x_arrow = 0.78 * x_max
    y_arrow = float(np.interp(x_arrow, damage_grid, damage_values))
    x_text = 0.62 * x_max
    y_text = min(y_max * 0.72, y_arrow + 0.16 * y_max)
    ax_main.annotate(r"$d(h)=\mathbb{E}[L\mid H=h]$", xy=(x_arrow, y_arrow), xytext=(x_text, y_text),
                     fontsize=ann_fs + 1.2, color="black", ha="left", va="center",
                     arrowprops=dict(arrowstyle="->", lw=0.9, color="black", shrinkA=2, shrinkB=4, connectionstyle="arc3,rad=0.12"), zorder=9)

    ax_main.set_xlim(0, x_max)
    ax_main.set_ylim(0, y_max)
    ax_main.set_title(f"Full-sample hazard-loss risk diagram ({cfg.scenario.copula.name} copula, {cfg.scenario.loss.name} losses)", fontsize=14.0, pad=14)
    ax_main.spines["top"].set_visible(False)
    ax_main.spines["right"].set_visible(False)
    ax_main.yaxis.set_major_locator(mpl.ticker.MaxNLocator(nbins=7))

    if mode == "full":
        ax_main.tick_params(axis="x", labelbottom=False)
        ax_main.set_ylabel(r"Loss $L$", labelpad=10)

        loss_pad = 0.18 * l_peak
        left_zero = -loss_pad
        left_xmin = -(l_peak + loss_pad)
        ax_left.fill_betweenx(l_grid, -(l_density + loss_pad), left_zero, color=col_loss, alpha=0.22, lw=0)
        ax_left.plot(-(l_density + loss_pad), l_grid, color=col_loss, linewidth=1.55)
        ax_left.hlines(l_var, xmin=left_xmin, xmax=left_zero, color=col_loss, linewidth=0.85, linestyle=(0, (7, 5)))
        ax_left.hlines(expected_l_given_l_tail, xmin=left_xmin, xmax=left_zero, color=col_loss, linewidth=0.9, linestyle=(0, (1.2, 2.4)))
        ax_left.set_xlim(left_xmin, loss_pad)
        ax_left.set_ylim(0, y_max)
        ax_left.set_xticks([left_zero, -(0.5 * l_peak + loss_pad), left_xmin])
        ax_left.set_xticklabels(["0", _fmt_density(0.5 * l_peak), _fmt_density(l_peak)])
        ax_left.tick_params(axis="x", labelsize=8.5, length=3)
        ax_left.set_yticks([])
        ax_left.set_xlabel(r"$f_L(\ell)$", labelpad=8)
        ax_left.spines["bottom"].set_bounds(left_xmin, left_zero)
        ax_left.spines["top"].set_visible(False)
        ax_left.spines["right"].set_visible(False)
        ax_left.spines["left"].set_visible(False)
    else:
        ax_main.set_xlabel(r"Hazard $H$", labelpad=8)
        ax_main.set_ylabel(r"Loss $L$", labelpad=10)

    ax_right.fill_betweenx(l_grid, 0, cond_l_density, color=col_blue, alpha=0.18, lw=0)
    ax_right.plot(cond_l_density, l_grid, color=col_blue, linewidth=1.65)
    ax_right.axhline(expected_l_given_h_tail, color=col_blue, linewidth=0.9, linestyle=(0, (1.2, 2.4)))
    if show_conditional_tail_var:
        ax_right.axhline(conditional_l_var, color=col_cond_var, linewidth=1.05, linestyle=(0, (9, 4)))
    ax_right.set_xlim(0, 1.05 * cond_peak)
    ax_right.set_ylim(0, y_max)
    ax_right.set_xticks([0, 0.5 * cond_peak, cond_peak])
    ax_right.set_xticklabels(["0", _fmt_density(0.5 * cond_peak), _fmt_density(cond_peak)])
    ax_right.tick_params(axis="x", labelsize=8.5, length=3)
    plt.setp(ax_right.get_yticklabels(), visible=False)
    ax_right.tick_params(axis="y", length=0)
    ax_right.set_xlabel(r"$f_{L\mid H\geq \mathrm{VaR}_q(H)}(\ell)$", color=col_blue, labelpad=8)
    ax_right.spines["top"].set_visible(False)
    ax_right.spines["right"].set_visible(False)

    if mode == "full":
        beyond = h_grid >= h_var
        ax_bottom.fill_between(h_grid, 0, h_density, color=col_haz, alpha=0.25, lw=0)
        ax_bottom.fill_between(h_grid[beyond], 0, h_density[beyond], color=col_blue, alpha=0.10, lw=0)
        ax_bottom.plot(h_grid, h_density, color=col_haz, linewidth=1.55)
        ax_bottom.axvline(h_var, color=col_haz, linewidth=1.1, linestyle=(0, (6, 4)))
        ax_bottom.set_xlim(0, x_max)
        ax_bottom.set_ylim(0, 1.05 * h_peak)
        ax_bottom.set_yticks([0, 0.5 * h_peak, h_peak])
        ax_bottom.set_yticklabels(["0", _fmt_density(0.5 * h_peak), _fmt_density(h_peak)])
        ax_bottom.set_xlabel(r"Hazard $H$", labelpad=8)
        ax_bottom.set_ylabel(r"$f_H(h)$", labelpad=8)
        ax_bottom.spines["top"].set_visible(False)
        ax_bottom.spines["right"].set_visible(False)
        fig.subplots_adjust(left=0.075, right=0.98, top=0.93, bottom=0.09)
    else:
        fig.subplots_adjust(left=0.09, right=0.97, top=0.90, bottom=0.11)

    save_figure(fig, f"hazard_loss_risk_diagram_{mode}_{cfg.cache_tag()}")
    plt.show()


def plot_hazard_loss_risk_diagram_curves(result: "BootstrapResult", *, q: float = 0.99,
                                         mode: Literal["full", "mainright"] = "full", n_grid: int = 900):
    """Theoretical-curves variant of the hazard-loss risk diagram: the same
    layout as `plot_hazard_loss_risk_diagram`, but VaR_q(L), ES_q(L), and
    E[L|H>=h] are drawn as curves over h (each evaluated at q=F_H(h))
    instead of pinned to one q. Every curve and marker here is the exact
    theoretical value (Parts 4/6's closed-form/quadrature machinery), not
    anything derived from the population sample -- only the scatter and the
    KDE density side-panels stay empirical, same as the figure above.
    """
    cfg, built, population = result.config, result.built, result.population
    h, y = population.hazard, population.losses

    col_blue = METHOD_COLORS["Quadratic"]
    col_loss = METHOD_COLORS["Sigmoid"]
    col_haz = METHOD_COLORS["Isotonic"]
    col_scatter = "#3f3f3f"

    h_var = float(built.hazard_dist.ppf(q))
    l_var = float(built.loss_dist.ppf(q))
    loss_es = float(built.loss_dist.expected_shortfall(q))
    damage_at_h_var = float(built.targets.d(np.array([h_var]))[0])
    tail_mean_at_h_var = float(built.targets.tail_mean(np.array([h_var]))[0])

    x_max = float(np.quantile(h, 0.997, method="inverted_cdf"))
    y_max = max(float(np.quantile(y, 0.995, method="inverted_cdf")), 1.08 * loss_es, 1.08 * tail_mean_at_h_var)

    h_grid = np.linspace(0.0, x_max, n_grid)
    damage_grid = np.linspace(0.0, x_max, min(n_grid, 800))
    damage_values = built.targets.d(damage_grid)

    q_grid = built.hazard_dist.cdf(h_grid)
    var_curve = built.loss_dist.ppf(q_grid)
    es_curve = built.loss_dist.expected_shortfall(q_grid)
    tail_mean_curve = built.targets.tail_mean(h_grid)

    l_grid = np.linspace(0.0, y_max, n_grid)
    tail_mask = h >= h_var
    l_given_h_tail = y[tail_mask]
    l_density = _reflected_log_kde(y, l_grid)
    cond_l_density = _reflected_log_kde(l_given_h_tail, l_grid)
    h_density = _reflected_log_kde(h, h_grid)
    l_peak = float(np.nanmax(l_density)) if np.any(np.isfinite(l_density)) else 1.0
    cond_peak = float(np.nanmax(cond_l_density)) if np.any(np.isfinite(cond_l_density)) else 1.0
    h_peak = float(np.nanmax(h_density)) if np.any(np.isfinite(h_density)) else 1.0

    ann_fs = 12.1

    if mode == "full":
        fig = plt.figure(figsize=(11.6, 7.9), dpi=160, facecolor="white")
        gs = fig.add_gridspec(2, 3, width_ratios=[1.4, 5.45, 1.22], height_ratios=[5.05, 1.28], wspace=0.16, hspace=0.08)
        ax_left = fig.add_subplot(gs[0, 0])
        ax_main = fig.add_subplot(gs[0, 1])
        ax_right = fig.add_subplot(gs[0, 2], sharey=ax_main)
        ax_bottom = fig.add_subplot(gs[1, 1], sharex=ax_main)
    else:
        fig = plt.figure(figsize=(10.2, 6.2), dpi=160, facecolor="white")
        gs = fig.add_gridspec(1, 2, width_ratios=[5.45, 1.22], wspace=0.10)
        ax_main = fig.add_subplot(gs[0, 0])
        ax_right = fig.add_subplot(gs[0, 1], sharey=ax_main)
        ax_left = ax_bottom = None

    ax_main.axvspan(h_var, x_max, ymin=0, ymax=l_var / y_max, color=col_blue, alpha=0.07, lw=0, zorder=0)
    ax_main.axhspan(l_var, y_max, color=col_loss, alpha=0.09, lw=0, zorder=0)
    ax_main.fill_between([h_var, x_max], l_var, y_max, color="#b8a2ff", alpha=0.15, zorder=0)

    in_view = (h <= x_max) & (y <= y_max)
    ax_main.scatter(h[in_view], y[in_view], s=2.5, c=col_scatter, alpha=0.08, linewidths=0, rasterized=True, zorder=1)

    ax_main.plot(damage_grid, damage_values, color="black", linewidth=2.0, zorder=5)
    ax_main.plot(h_grid, var_curve, color=col_loss, linewidth=1.6, linestyle=(0, (7, 5)), zorder=4, label=r"$\mathrm{VaR}_{F_H(h)}(L)$")
    ax_main.plot(h_grid, es_curve, color=col_loss, linewidth=1.6, linestyle=(0, (1.2, 2.4)), zorder=4, label=r"$\mathrm{ES}_{F_H(h)}(L)$")
    ax_main.plot(h_grid, tail_mean_curve, color=col_blue, linewidth=1.6, linestyle=(0, (1.2, 2.4)), zorder=4, label=r"$\mathbb{E}[L\mid H\geq h]$")
    ax_main.axvline(h_var, color=col_haz, linewidth=1.2, linestyle=(0, (6, 4)), zorder=2)
    ax_main.axhline(l_var, color=col_loss, linewidth=1.1, linestyle=(0, (7, 5)), zorder=2)

    marker_kw = dict(markeredgecolor="white", markeredgewidth=0.8, zorder=8, clip_on=False)
    ax_main.plot(h_var, loss_es, marker="*", markersize=14, color=col_loss, **marker_kw)
    ax_main.plot(h_var, tail_mean_at_h_var, marker="o", markersize=8.5, color=col_blue, **marker_kw)
    ax_main.plot(h_var, l_var, marker="s", markersize=8.0, color=col_loss, **marker_kw)
    ax_main.plot(h_var, damage_at_h_var, marker="D", markersize=7.5, color="black", **marker_kw)

    x_arrow = 0.78 * x_max
    y_arrow = float(np.interp(x_arrow, damage_grid, damage_values))
    x_text = 0.62 * x_max
    y_text = min(y_max * 0.72, y_arrow + 0.16 * y_max)
    ax_main.annotate(r"$d(h)=\mathbb{E}[L\mid H=h]$", xy=(x_arrow, y_arrow), xytext=(x_text, y_text),
                     fontsize=ann_fs + 1.2, color="black", ha="left", va="center",
                     arrowprops=dict(arrowstyle="->", lw=0.9, color="black", shrinkA=2, shrinkB=4, connectionstyle="arc3,rad=0.12"), zorder=9)

    ax_main.set_xlim(0, x_max)
    ax_main.set_ylim(0, y_max)
    ax_main.set_title(f"Hazard-loss risk diagram, theoretical curves ({cfg.scenario.copula.name} copula, {cfg.scenario.loss.name} losses)",
                      fontsize=14.0, pad=14)
    ax_main.legend(loc="upper left", fontsize=11.1, frameon=False, handlelength=1.8)
    ax_main.spines["top"].set_visible(False)
    ax_main.spines["right"].set_visible(False)
    ax_main.yaxis.set_major_locator(mpl.ticker.MaxNLocator(nbins=7))

    if mode == "full":
        ax_main.tick_params(axis="x", labelbottom=False)
        ax_main.set_ylabel(r"Loss $L$", labelpad=10)

        loss_pad = 0.18 * l_peak
        left_zero = -loss_pad
        left_xmin = -(l_peak + loss_pad)
        ax_left.fill_betweenx(l_grid, -(l_density + loss_pad), left_zero, color=col_loss, alpha=0.22, lw=0)
        ax_left.plot(-(l_density + loss_pad), l_grid, color=col_loss, linewidth=1.55)
        ax_left.hlines(l_var, xmin=left_xmin, xmax=left_zero, color=col_loss, linewidth=0.85, linestyle=(0, (7, 5)))
        ax_left.hlines(loss_es, xmin=left_xmin, xmax=left_zero, color=col_loss, linewidth=0.9, linestyle=(0, (1.2, 2.4)))
        ax_left.set_xlim(left_xmin, loss_pad)
        ax_left.set_ylim(0, y_max)
        ax_left.set_xticks([left_zero, -(0.5 * l_peak + loss_pad), left_xmin])
        ax_left.set_xticklabels(["0", _fmt_density(0.5 * l_peak), _fmt_density(l_peak)])
        ax_left.tick_params(axis="x", labelsize=8.5, length=3)
        ax_left.set_yticks([])
        ax_left.set_xlabel(r"$f_L(\ell)$", labelpad=8)
        ax_left.spines["bottom"].set_bounds(left_xmin, left_zero)
        ax_left.spines["top"].set_visible(False)
        ax_left.spines["right"].set_visible(False)
        ax_left.spines["left"].set_visible(False)

        x_label_left = 0.5 * (left_xmin + left_zero)
        ax_left.text(x_label_left, l_var + 0.025 * y_max, r"$\mathrm{VaR}_q(L)$", color=col_loss, fontsize=ann_fs,
                    ha="center", va="bottom", clip_on=False)
        ax_left.text(x_label_left, loss_es + 0.025 * y_max, r"$\mathrm{ES}_q(L)$", color=col_loss, fontsize=ann_fs,
                    ha="center", va="bottom", clip_on=False)
    else:
        ax_main.set_xlabel(r"Hazard $H$", labelpad=8)
        ax_main.set_ylabel(r"Loss $L$", labelpad=10)

    ax_right.fill_betweenx(l_grid, 0, cond_l_density, color=col_blue, alpha=0.18, lw=0)
    ax_right.plot(cond_l_density, l_grid, color=col_blue, linewidth=1.65)
    ax_right.axhline(tail_mean_at_h_var, color=col_blue, linewidth=0.9, linestyle=(0, (1.2, 2.4)))
    ax_right.set_xlim(0, 1.05 * cond_peak)
    ax_right.set_ylim(0, y_max)
    ax_right.set_xticks([0, 0.5 * cond_peak, cond_peak])
    ax_right.set_xticklabels(["0", _fmt_density(0.5 * cond_peak), _fmt_density(cond_peak)])
    ax_right.tick_params(axis="x", labelsize=8.5, length=3)
    plt.setp(ax_right.get_yticklabels(), visible=False)
    ax_right.tick_params(axis="y", length=0)
    ax_right.set_xlabel(r"$f_{L\mid H\geq \mathrm{VaR}_q(H)}(\ell)$", color=col_blue, labelpad=8)
    ax_right.spines["top"].set_visible(False)
    ax_right.spines["right"].set_visible(False)
    va_tm = "bottom" if tail_mean_at_h_var < 0.90 * y_max else "top"
    y_txt_tm = tail_mean_at_h_var + 0.025 * y_max if va_tm == "bottom" else tail_mean_at_h_var - 0.025 * y_max
    ax_right.text(0.03, y_txt_tm, r"$\mathbb{E}[L\mid H\geq \mathrm{VaR}_q(H)]$", color=col_blue, fontsize=ann_fs,
                 ha="left", va=va_tm, clip_on=False,
                 bbox=dict(facecolor="white", alpha=0.72, edgecolor="none", pad=1.5))

    if mode == "full":
        beyond = h_grid >= h_var
        ax_bottom.fill_between(h_grid, 0, h_density, color=col_haz, alpha=0.25, lw=0)
        ax_bottom.fill_between(h_grid[beyond], 0, h_density[beyond], color=col_blue, alpha=0.10, lw=0)
        ax_bottom.plot(h_grid, h_density, color=col_haz, linewidth=1.55)
        ax_bottom.axvline(h_var, color=col_haz, linewidth=1.1, linestyle=(0, (6, 4)))
        ax_bottom.set_xlim(0, x_max)
        ax_bottom.set_ylim(0, 1.05 * h_peak)
        ax_bottom.set_yticks([0, 0.5 * h_peak, h_peak])
        ax_bottom.set_yticklabels(["0", _fmt_density(0.5 * h_peak), _fmt_density(h_peak)])
        ax_bottom.set_xlabel(r"Hazard $H$", labelpad=8)
        ax_bottom.set_ylabel(r"$f_H(h)$", labelpad=8)
        ax_bottom.spines["top"].set_visible(False)
        ax_bottom.spines["right"].set_visible(False)
        ax_bottom.text(h_var + 0.02 * x_max, 0.95 * h_peak, r"$\mathrm{VaR}_q(H)$", color=col_haz, fontsize=ann_fs,
                      ha="left", va="top")
        fig.subplots_adjust(left=0.075, right=0.98, top=0.93, bottom=0.09)
    else:
        fig.subplots_adjust(left=0.09, right=0.97, top=0.90, bottom=0.11)

    save_figure(fig, f"hazard_loss_risk_diagram_curves_{mode}_{cfg.cache_tag()}")
    plt.show()

### 9e. IDR conditional-law figure

IDR (Part 7d) is the one estimator here that targets a whole conditional
*distribution*, not just a conditional mean -- every other figure in this
notebook only ever shows its predicted mean curve, discarding that extra
information. This figure keeps it: for one representative subsample, IDR's
fitted mean curve $d_{IDR}(h)$ is compared to the true curve $d(h)$, and its
full fitted tail law
$$\hat F_{L\mid H\geq \widehat{\mathrm{VaR}}_q(H)}(z)\approx
\frac{1}{|\{i:H_i\geq \widehat{\mathrm{VaR}}_q(H)\}|}\sum_{i:H_i\geq \widehat{\mathrm{VaR}}_q(H)}\hat F_{L\mid H=H_i}(z)$$
(`OfficialIDRDamageModel.exceedance_distribution`, the same mixture Part 8
uses for IDR's tail-summary reporting) is shown as a probability-weighted
horizontal-bar histogram.

Two $\mathrm{VaR}_q(H)$ positions are marked and labeled separately, since
they generally differ: $\widehat{\mathrm{VaR}}_q(H)$ is the *subsample's
own* empirical hazard quantile (what IDR was actually fit and queried
against), and $\mathrm{VaR}_q(H)$ is the full population's empirical hazard
quantile -- each with its own curve point, tail-mean marker, and (if
`show_conditional_tail_var=True`, off by default) tail-VaR marker, so the
figure also shows how much IDR's *own* estimate of the query point differs
from the truth, not just how well its curve tracks $d(h)$.

**Cell below:** `_hist_bar_from_atoms` and `plot_idr_conditional_law`.


In [ ]:
def _hist_bar_from_atoms(thresholds: np.ndarray, probs: np.ndarray, bins: np.ndarray):
    """Bin a discrete (thresholds, probability) law into a peak-normalized
    weighted histogram, for a horizontal probability-mass bar plot."""
    hist, edges = np.histogram(thresholds, bins=bins, weights=np.nan_to_num(probs, nan=0.0))
    peak = np.max(hist) if hist.size else 0.0
    scaled = hist / peak if peak > 0 else hist
    centers = 0.5 * (edges[:-1] + edges[1:])
    heights = np.diff(edges)
    return centers, heights, scaled


def _stack_labels_at_x(ax, x, entries, y_max, *, hug_frac=0.012, y_min_frac=0.02,
                       avoid_artists=(), pad_px=4.0):
    """Place a column of text labels anchored at `x` with no overlap, measured
    from Matplotlib's own rendered text extents rather than a guessed height.
    Each entry is a dict with `y` (the value it annotates), `text`, `color`,
    and optionally `alpha`/`fontsize`/`ha`. Labels are drawn at their target
    y (offset up slightly by `hug_frac * y_max` so they don't sit exactly on
    their own line), then walked bottom-to-top in a single sweep: each
    label's minimum allowed bottom is the max of `y_min_frac * y_max` (a
    floor), the previous label's *actual* rendered top, and the top of any
    `avoid_artists` obstacle whose x-range it overlaps -- all resolved
    together in one pass, so clearing one constraint can't silently
    reintroduce another. There is deliberately no upper ceiling/compression:
    an earlier version tried to squeeze the whole stack under a fixed
    fraction of y_max whenever the top label ran past it, which -- when a
    handful of labels span a huge range (e.g. a tiny mean next to a huge
    optional tail-VaR value) -- could shift small, already-separated labels
    down far enough that the final floor-clamp collapsed several of them
    onto the exact same position. Labels are free to run close to (or
    slightly past) the top of the axes instead; pass the plot's own
    reference-line labels (e.g. the VaR gridline text) as `avoid_artists` if
    they need to be kept clear of specifically.
    `avoid_artists` are other Text (or any) artists already on the axes --
    e.g. a reference-line label placed elsewhere, possibly at a different x
    -- checked via each one's real bounding box, since a wide obstacle
    anchored far away may not actually reach every row."""
    if not entries:
        return
    lo = y_min_frac * y_max
    ordered = sorted(entries, key=lambda e: e["y"])
    texts = []
    for e in ordered:
        t = ax.text(x, e["y"] + hug_frac * y_max, e["text"], color=e["color"], alpha=e.get("alpha", 1.0),
                   fontsize=e.get("fontsize", 12.1), ha=e.get("ha", "left"), va="bottom", zorder=9,
                   bbox=dict(facecolor="white", edgecolor="none", alpha=0.72, pad=1.5))
        texts.append(t)

    fig = ax.figure
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    inv = ax.transData.inverted()

    def data_bbox(artist):
        bb = artist.get_window_extent(renderer=renderer)
        x0, y0 = inv.transform((bb.x0, bb.y0))
        x1, y1 = inv.transform((bb.x1, bb.y1))
        return [x0, y0, x1, y1]

    ax_bb = ax.get_window_extent(renderer=renderer)
    y0_lim, y1_lim = ax.get_ylim()
    px_per_data_y = ax_bb.height / max(y1_lim - y0_lim, 1e-12)
    pad_data = pad_px / max(px_per_data_y, 1e-12)

    def x_overlaps(b1, b2):
        return not (b1[2] + pad_data < b2[0] or b2[2] + pad_data < b1[0])

    boxes = [data_bbox(t) for t in texts]
    obstacle_boxes = [data_bbox(a) for a in avoid_artists]

    top_so_far = None
    for t, box in zip(texts, boxes):
        min_bottom = lo
        if top_so_far is not None:
            min_bottom = max(min_bottom, top_so_far + pad_data)
        for ob in obstacle_boxes:
            if x_overlaps(box, ob):
                min_bottom = max(min_bottom, ob[3] + pad_data)
        if box[1] < min_bottom:
            dy = min_bottom - box[1]
            cur_x, cur_y = t.get_position()
            t.set_position((cur_x, cur_y + dy))
            box[1] += dy
            box[3] += dy
        top_so_far = box[3]

def plot_idr_conditional_law(result: "BootstrapResult", *, q: float = 0.99, hist_bins_loss: int = 34,
                             subsample_seed: int = 0, show_conditional_tail_var: bool = False, n_grid: int = 400):
    """True vs. IDR-estimated damage curve on one representative subsample,
    with IDR's tail-conditional law shown as a horizontal probability-mass
    histogram."""
    cfg, built, population = result.config, result.built, result.population
    rng = np.random.default_rng(subsample_seed)
    idx = rng.choice(population.hazard.size, size=cfg.run.n, replace=False)
    h, y = population.hazard[idx], population.losses[idx]

    model = OfficialIDRDamageModel().fit(h, y)
    col_idr = METHOD_COLORS["IDR"]
    col_haz_var = METHOD_COLORS["Quadratic"]

    h_var = float(np.quantile(h, q, method="inverted_cdf"))
    true_h_var = float(np.quantile(population.hazard, q, method="inverted_cdf"))

    damage_at_h_var = float(model.predict(np.array([h_var]))[0])
    true_damage_at_h_var = float(built.targets.d(np.array([true_h_var]))[0])

    tail_law = model.exceedance_distribution(np.array([h_var]), hazard_data=h)
    idr_tail_mean = float(tail_law.mean()[0])
    idr_tail_var = float(tail_law.value_at_risk(q)[0]) if show_conditional_tail_var else np.nan

    true_tail_mask = population.hazard >= true_h_var
    true_l_given_h_tail = population.losses[true_tail_mask]
    true_expected_l_given_h_tail = float(true_l_given_h_tail.mean()) if true_l_given_h_tail.size else np.nan
    true_conditional_l_var = (float(np.quantile(true_l_given_h_tail, q, method="inverted_cdf"))
                              if show_conditional_tail_var and true_l_given_h_tail.size else np.nan)

    x_max = float(max(np.quantile(h, 0.997, method="inverted_cdf"), h_var + 0.25))
    y_candidates = [float(np.quantile(y, 0.995, method="inverted_cdf")), 1.08 * damage_at_h_var, 1.08 * idr_tail_mean,
                   1.08 * true_damage_at_h_var, 1.08 * true_expected_l_given_h_tail]
    if show_conditional_tail_var:
        y_candidates += [1.08 * idr_tail_var, 1.08 * true_conditional_l_var]
    y_max = max(y_candidates)

    damage_grid = np.linspace(0.0, x_max, n_grid)
    idr_curve = model.predict(damage_grid)
    true_curve = built.targets.d(damage_grid)

    loss_bins = np.linspace(0.0, y_max, hist_bins_loss + 1)
    centers, heights, hist_scaled = _hist_bar_from_atoms(tail_law.thresholds, tail_law.probabilities()[0], loss_bins)

    fig = plt.figure(figsize=(9.5, 5.9), facecolor="white")
    gs = fig.add_gridspec(1, 2, width_ratios=[5.45, 1.22], wspace=0.08)
    ax_main = fig.add_subplot(gs[0, 0])
    ax_right = fig.add_subplot(gs[0, 1], sharey=ax_main)

    ax_main.axvspan(h_var, x_max, color=col_haz_var, alpha=0.025, lw=0, zorder=0)
    visible = (h <= x_max) & (y <= y_max)
    ax_main.scatter(h[visible], y[visible], s=8.0, c="#3f3f3f", alpha=0.22, linewidths=0, rasterized=True, zorder=1)
    ax_main.plot(damage_grid, true_curve, color="black", linewidth=2.0, zorder=5)
    ax_main.plot(damage_grid, idr_curve, color=col_idr, linewidth=2.0, linestyle=(0, (5, 3)), zorder=6)

    ax_main.axvline(h_var, color=col_idr, linewidth=1.2, linestyle=(0, (6, 4)), zorder=2)
    ax_main.axvline(true_h_var, color="black", linewidth=1.0, linestyle=(0, (1.2, 2.4)), alpha=0.75, zorder=2)
    ax_main.axhline(damage_at_h_var, color=col_idr, linewidth=1.0, linestyle=(0, (5, 4)), zorder=2)
    ax_main.axhline(true_damage_at_h_var, color="black", linewidth=1.0, linestyle=(0, (1.2, 2.4)), alpha=0.75, zorder=2)
    ax_main.axhline(idr_tail_mean, color=col_idr, linewidth=1.15, linestyle=(0, (1.2, 2.2)), zorder=2)
    ax_main.axhline(true_expected_l_given_h_tail, color="black", linewidth=1.0, linestyle=(0, (1.2, 2.4)), alpha=0.52, zorder=2)
    if show_conditional_tail_var:
        ax_main.axhline(idr_tail_var, color=col_idr, linewidth=1.35, linestyle=(0, (9, 4)), zorder=2)
        ax_main.axhline(true_conditional_l_var, color=col_idr, linewidth=1.15, linestyle=(0, (9, 4)), alpha=0.62, zorder=2)

    marker_kw = dict(markeredgecolor="white", markeredgewidth=0.8, zorder=8, clip_on=False)
    ax_main.plot(h_var, damage_at_h_var, marker="D", markersize=7.5, color=col_idr, **marker_kw)
    ax_main.plot(true_h_var, true_damage_at_h_var, marker="D", markersize=7.5, color="black", **marker_kw)
    ax_main.plot(h_var, idr_tail_mean, marker="o", markersize=8.5, color=col_idr, **marker_kw)
    ax_main.plot(true_h_var, true_expected_l_given_h_tail, marker="o", markersize=8.0, color="black", **marker_kw)
    if show_conditional_tail_var:
        ax_main.plot(h_var, idr_tail_var, marker="^", markersize=9.5, color=col_idr, **marker_kw)
        ax_main.plot(true_h_var, true_conditional_l_var, marker="^", markersize=9.0, color=col_idr, alpha=0.62, **marker_kw)

    x_label = 0.10 * x_max
    if h_var <= true_h_var:
        hat_ha, hat_dx, true_ha, true_dx = "right", -0.018 * x_max, "left", 0.018 * x_max
    else:
        hat_ha, hat_dx, true_ha, true_dx = "left", 0.018 * x_max, "right", -0.018 * x_max
    label_bbox = dict(facecolor="white", edgecolor="none", alpha=0.72, pad=1.2)
    hat_var_text = ax_main.text(h_var + hat_dx, y_max * 0.975, rf"$\widehat{{\mathrm{{VaR}}}}_{{{q:.2f}}}(H)$", color=col_idr,
                fontsize=12.1, ha=hat_ha, va="top", rotation=90, zorder=9, bbox=label_bbox)
    true_var_text = ax_main.text(true_h_var + true_dx, y_max * 0.975, rf"$\mathrm{{VaR}}_{{{q:.2f}}}(H)$", color="black",
                fontsize=12.1, ha=true_ha, va="top", rotation=90, zorder=9, bbox=label_bbox)

    label_entries = [
        dict(y=idr_tail_mean, text=rf"$\mathbb{{E}}^{{IDR}}[L\mid H\geq \widehat{{\mathrm{{VaR}}}}_{{{q:.2f}}}(H)]$", color=col_idr),
        dict(y=true_expected_l_given_h_tail, text=rf"$\mathbb{{E}}[L\mid H\geq \mathrm{{VaR}}_{{{q:.2f}}}(H)]$", color="black"),
        dict(y=damage_at_h_var,
            text=rf"$d_{{IDR}}(\widehat{{\mathrm{{VaR}}}}_{{{q:.2f}}}(H))=\mathbb{{E}}^{{IDR}}[L\mid H=\widehat{{\mathrm{{VaR}}}}_{{{q:.2f}}}(H)]$", color=col_idr),
        dict(y=true_damage_at_h_var, text=rf"$d(\mathrm{{VaR}}_{{{q:.2f}}}(H))$", color="black"),
    ]
    if show_conditional_tail_var:
        label_entries.append(dict(y=true_conditional_l_var,
            text=rf"$\mathrm{{VaR}}_{{{q:.2f}}}(L\mid H\geq \mathrm{{VaR}}_{{{q:.2f}}}(H))$", color=col_idr, alpha=0.62))
        label_entries.append(dict(y=idr_tail_var,
            text=rf"$\mathrm{{VaR}}^{{IDR}}_{{{q:.2f}}}(L\mid H\geq \widehat{{\mathrm{{VaR}}}}_{{{q:.2f}}}(H))$", color=col_idr, alpha=0.62))
    label_entries = [e for e in label_entries if np.isfinite(e["y"])]
    # hat_var_text/true_var_text are rotated 90 degrees; Matplotlib's
    # get_window_extent() on rotated text returns an unreliable, often far
    # too large axis-aligned bounding box under this renderer, so they are
    # not used as obstacles here -- they're anchored near h_var/true_h_var,
    # spatially separate from this left-anchored column in every normal case.
    _stack_labels_at_x(ax_main, x_label, label_entries, y_max)

    x_arrow = 0.78 * x_max
    y_arrow = float(np.interp(x_arrow, damage_grid, idr_curve))
    ax_main.annotate(r"$d_{IDR}(h)$", xy=(x_arrow, y_arrow), xytext=(0.58 * x_max, min(y_max * 0.72, y_arrow + 0.16 * y_max)),
                     fontsize=13.3, color=col_idr, ha="left", va="center",
                     arrowprops=dict(arrowstyle="->", lw=0.9, color=col_idr, shrinkA=2, shrinkB=4, connectionstyle="arc3,rad=0.12"),
                     bbox=label_bbox, zorder=9)
    x_true_arrow = 0.70 * x_max
    y_true_arrow = float(np.interp(x_true_arrow, damage_grid, true_curve))
    ax_main.annotate(r"$d(h)$", xy=(x_true_arrow, y_true_arrow), xytext=(0.40 * x_max, min(y_max * 0.50, y_true_arrow + 0.10 * y_max)),
                     fontsize=13.3, color="black", ha="left", va="center",
                     arrowprops=dict(arrowstyle="->", lw=0.9, color="black", shrinkA=2, shrinkB=4, connectionstyle="arc3,rad=0.12"),
                     bbox=label_bbox, zorder=9)

    ax_main.set_xlim(0, x_max)
    ax_main.set_ylim(0, y_max)
    ax_main.set_title(rf"IDR tail estimate $\widehat{{F}}_{{L\mid H\geq \widehat{{\mathrm{{VaR}}}}_{{{q:.2f}}}(H)}}$ ({cfg.scenario.copula.name} copula, {cfg.scenario.loss.name} losses)",
                      fontsize=14.0, pad=14)
    ax_main.set_xlabel(r"Hazard $H$", labelpad=8)
    ax_main.set_ylabel(r"Loss $L$", labelpad=10)
    ax_main.spines["top"].set_visible(False)
    ax_main.spines["right"].set_visible(False)
    ax_main.yaxis.set_major_locator(mpl.ticker.MaxNLocator(nbins=7))

    ax_right.barh(centers, hist_scaled, height=0.90 * heights, left=0, color=col_haz_var,
                 alpha=0.18, edgecolor=col_haz_var, linewidth=0.45)
    ax_right.axhline(idr_tail_mean, color=col_idr, linewidth=0.9, linestyle=(0, (1.2, 2.2)))
    ax_right.axhline(true_expected_l_given_h_tail, color="black", linewidth=0.8, linestyle=(0, (1.2, 2.4)), alpha=0.62)
    if show_conditional_tail_var:
        ax_right.axhline(idr_tail_var, color=col_idr, linewidth=1.05, linestyle=(0, (9, 4)))
        ax_right.axhline(true_conditional_l_var, color=col_idr, linewidth=0.95, linestyle=(0, (9, 4)), alpha=0.62)
    ax_right.set_xlim(0, 1.03)
    ax_right.set_ylim(0, y_max)
    ax_right.set_xticks([])
    plt.setp(ax_right.get_yticklabels(), visible=False)
    ax_right.tick_params(axis="y", length=0)
    ax_right.set_xlabel(rf"$\widehat{{p}}^{{IDR}}(L \mid H\geq \widehat{{\mathrm{{VaR}}}}_{{{q:.2f}}}(H))$", color=col_idr, labelpad=8, fontsize=10.2)
    ax_right.spines["top"].set_visible(False)
    ax_right.spines["right"].set_visible(False)

    fig.subplots_adjust(left=0.090, right=0.965, top=0.905, bottom=0.155)
    save_figure(fig, f"idr_conditional_law_{cfg.cache_tag()}")
    plt.show()


def plot_idr_risk_diagram(result: "BootstrapResult", *, q: float = 0.99, hist_bins_loss: int = 34,
                          subsample_seed: int = 0, n_grid: int = 400):
    """IDR's fitted curve and tail-conditional law against the true theory,
    in the cleaner style of `plot_hazard_loss_risk_diagram_curves`: both true
    quantities (`d(h)` and `E[L|H>=h]`) are drawn as curves over the full h
    range -- like that plot's `tail_mean_curve` -- instead of `plot_idr_conditional_law`'s
    single-q true markers/axhlines, which just restated points already
    readable off those curves. Non-IDR elements reuse that plot's own color
    convention (`col_blue`/`col_haz`) instead of plain black. IDR's own
    values stay marked with axvline/axhline + markers exactly as
    `plot_idr_conditional_law` has them."""
    cfg, built, population = result.config, result.built, result.population
    rng = np.random.default_rng(subsample_seed)
    idx = rng.choice(population.hazard.size, size=cfg.run.n, replace=False)
    h, y = population.hazard[idx], population.losses[idx]

    model = OfficialIDRDamageModel().fit(h, y)
    col_idr = METHOD_COLORS["IDR"]
    col_haz_var = METHOD_COLORS["Quadratic"]
    col_blue = METHOD_COLORS["Quadratic"]  # true E[L|H>=h] curve/marker -- matches the theory plot's tail-mean color
    col_haz = METHOD_COLORS["Isotonic"]    # true VaR_q(H) reference line -- matches the theory plot's axvline color

    h_var = float(np.quantile(h, q, method="inverted_cdf"))
    true_h_var = float(np.quantile(population.hazard, q, method="inverted_cdf"))

    damage_at_h_var = float(model.predict(np.array([h_var]))[0])
    true_damage_at_h_var = float(built.targets.d(np.array([true_h_var]))[0])

    tail_law = model.exceedance_distribution(np.array([h_var]), hazard_data=h)
    idr_tail_mean = float(tail_law.mean()[0])

    true_expected_l_given_h_tail = float(built.targets.tail_mean(np.array([true_h_var]))[0])

    x_max = float(max(np.quantile(h, 0.997, method="inverted_cdf"), h_var + 0.25))

    damage_grid = np.linspace(0.0, x_max, n_grid)
    idr_curve = model.predict(damage_grid)
    true_curve = built.targets.d(damage_grid)
    true_tail_mean_curve = built.targets.tail_mean(damage_grid)

    y_candidates = [float(np.quantile(y, 0.995, method="inverted_cdf")), 1.08 * damage_at_h_var, 1.08 * idr_tail_mean,
                   1.08 * true_damage_at_h_var, 1.08 * true_expected_l_given_h_tail,
                   1.05 * float(np.nanmax(true_tail_mean_curve)),
                   1.05 * float(np.nanmax(idr_curve)), 1.05 * float(np.nanmax(true_curve))]
    y_max = max(y_candidates)

    loss_bins = np.linspace(0.0, y_max, hist_bins_loss + 1)
    centers, heights, hist_scaled = _hist_bar_from_atoms(tail_law.thresholds, tail_law.probabilities()[0], loss_bins)

    fig = plt.figure(figsize=(9.5, 5.9), facecolor="white")
    gs = fig.add_gridspec(1, 2, width_ratios=[5.45, 1.22], wspace=0.08)
    ax_main = fig.add_subplot(gs[0, 0])
    ax_right = fig.add_subplot(gs[0, 1], sharey=ax_main)

    ax_main.axvspan(h_var, x_max, color=col_haz_var, alpha=0.025, lw=0, zorder=0)
    visible = (h <= x_max) & (y <= y_max)
    ax_main.scatter(h[visible], y[visible], s=8.0, c="#3f3f3f", alpha=0.22, linewidths=0, rasterized=True, zorder=1)
    ax_main.plot(damage_grid, true_curve, color="black", linewidth=2.0, zorder=5)
    ax_main.plot(damage_grid, true_tail_mean_curve, color=col_blue, linewidth=1.6, linestyle=(0, (4, 1.8)), zorder=4)
    ax_main.plot(damage_grid, idr_curve, color=col_idr, linewidth=2.0, linestyle=(0, (5, 3)), zorder=6)

    ax_main.axvline(true_h_var, color=col_haz, linewidth=1.2, linestyle=(0, (6, 4)), zorder=2)
    ax_main.axvline(h_var, color=col_idr, linewidth=1.2, linestyle=(0, (6, 4)), zorder=2)
    ax_main.axhline(damage_at_h_var, color=col_idr, linewidth=1.0, linestyle=(0, (5, 4)), zorder=2)
    ax_main.axhline(idr_tail_mean, color=col_idr, linewidth=1.15, linestyle=(0, (1.2, 2.2)), zorder=2)
    ax_main.axhline(true_damage_at_h_var, color="black", linewidth=0.95, linestyle=(0, (1.2, 2.4)), alpha=0.7, zorder=2)

    marker_kw = dict(markeredgecolor="white", markeredgewidth=0.8, zorder=8, clip_on=False)
    ax_main.plot(h_var, damage_at_h_var, marker="D", markersize=7.5, color=col_idr, **marker_kw)
    ax_main.plot(h_var, idr_tail_mean, marker="o", markersize=8.5, color=col_idr, **marker_kw)
    # True-side markers, at the curves' own VaR_q(H) intersection -- same
    # marker shapes/colors the theory plot uses for d(h) and E[L|H>=h].
    ax_main.plot(true_h_var, true_damage_at_h_var, marker="D", markersize=7.5, color="black", **marker_kw)
    ax_main.plot(true_h_var, true_expected_l_given_h_tail, marker="o", markersize=8.5, color=col_blue, **marker_kw)

    label_bbox = dict(facecolor="white", edgecolor="none", alpha=0.72, pad=1.2)

    hat_ha, hat_dx = ("right", -0.018 * x_max) if h_var <= true_h_var else ("left", 0.018 * x_max)
    true_ha, true_dx = ("left", 0.018 * x_max) if h_var <= true_h_var else ("right", -0.018 * x_max)
    ax_main.text(h_var + hat_dx, y_max * 0.975, rf"$\widehat{{\mathrm{{VaR}}}}_{{{q:.2f}}}(H)$", color=col_idr,
                fontsize=12.1, ha=hat_ha, va="top", rotation=90, zorder=9, bbox=label_bbox)
    ax_main.text(true_h_var + true_dx, y_max * 0.975, rf"$\mathrm{{VaR}}_{{{q:.2f}}}(H)$", color=col_haz,
                fontsize=12.1, ha=true_ha, va="top", rotation=90, zorder=9, bbox=label_bbox)

    # Curve identification via a compact legend, matching the theory plot's
    # own approach, instead of stacking a third arrow-callout next to the
    # two IDR/true `d(h)` ones -- avoids competing with the rotated VaR
    # labels near the top of the axes.
    ax_main.plot([], [], color="black", linewidth=2.0, label=r"$d(h)=\mathbb{E}[L\mid H=h]$")
    ax_main.plot([], [], color=col_blue, linewidth=1.6, linestyle=(0, (4, 1.8)), label=r"$\mathbb{E}[L\mid H\geq h]$")
    ax_main.plot([], [], color=col_idr, linewidth=2.0, linestyle=(0, (5, 3)), label=r"$d_{IDR}(h)$")
    ax_main.plot([], [], color=col_idr, linewidth=1.15, linestyle=(0, (1.2, 2.2)),
                label=rf"$\mathbb{{E}}^{{IDR}}[L\mid H\geq \widehat{{\mathrm{{VaR}}}}_{{{q:.2f}}}(H)]$")
    ax_main.plot([], [], color=col_idr, linewidth=1.0, linestyle=(0, (5, 4)),
                label=rf"$d_{{IDR}}(\widehat{{\mathrm{{VaR}}}}_{{{q:.2f}}}(H))=\mathbb{{E}}^{{IDR}}[L\mid H=\widehat{{\mathrm{{VaR}}}}_{{{q:.2f}}}(H)]$")
    ax_main.legend(loc="upper left", fontsize=11.1, frameon=False, handlelength=1.8)

    ax_main.set_xlim(0, x_max)
    ax_main.set_ylim(0, y_max)
    ax_main.set_title(rf"IDR risk diagram ({cfg.scenario.copula.name} copula, {cfg.scenario.loss.name} losses)",
                      fontsize=14.0, pad=14)
    ax_main.set_xlabel(r"Hazard $H$", labelpad=8)
    ax_main.set_ylabel(r"Loss $L$", labelpad=10)
    ax_main.spines["top"].set_visible(False)
    ax_main.spines["right"].set_visible(False)
    ax_main.yaxis.set_major_locator(mpl.ticker.MaxNLocator(nbins=7))

    ax_right.barh(centers, hist_scaled, height=0.90 * heights, left=0, color=col_haz_var,
                 alpha=0.18, edgecolor=col_haz_var, linewidth=0.45)
    ax_right.axhline(idr_tail_mean, color=col_idr, linewidth=0.9, linestyle=(0, (1.2, 2.2)))
    ax_right.axhline(true_expected_l_given_h_tail, color=col_blue, linewidth=0.9, linestyle=(0, (1.2, 2.4)), alpha=0.85)
    ax_right.set_xlim(0, 1.03)
    ax_right.set_ylim(0, y_max)
    ax_right.set_xticks([])
    plt.setp(ax_right.get_yticklabels(), visible=False)
    ax_right.tick_params(axis="y", length=0)
    ax_right.set_xlabel(rf"$\widehat{{p}}^{{IDR}}(L \mid H\geq \widehat{{\mathrm{{VaR}}}}_{{{q:.2f}}}(H))$", color=col_idr, labelpad=8, fontsize=10.2)
    ax_right.spines["top"].set_visible(False)
    ax_right.spines["right"].set_visible(False)

    va_idr = "bottom" if idr_tail_mean < 0.90 * y_max else "top"
    y_txt_idr = idr_tail_mean + 0.025 * y_max if va_idr == "bottom" else idr_tail_mean - 0.025 * y_max
    ax_right.text(0.03, y_txt_idr, rf"$\mathbb{{E}}^{{IDR}}[L\mid H\geq \widehat{{\mathrm{{VaR}}}}_{{{q:.2f}}}(H)]$",
                 color=col_idr, fontsize=10.3, ha="left", va=va_idr, clip_on=False, bbox=label_bbox)
    va_true = "bottom" if true_expected_l_given_h_tail < 0.90 * y_max else "top"
    y_txt_true = true_expected_l_given_h_tail + 0.025 * y_max if va_true == "bottom" else true_expected_l_given_h_tail - 0.025 * y_max
    ax_right.text(0.03, y_txt_true, rf"$\mathbb{{E}}[L\mid H\geq \mathrm{{VaR}}_{{{q:.2f}}}(H)]$",
                 color=col_blue, fontsize=10.3, ha="left", va=va_true, clip_on=False, bbox=label_bbox)

    fig.subplots_adjust(left=0.090, right=0.965, top=0.905, bottom=0.155)
    save_figure(fig, f"idr_risk_diagram_{cfg.cache_tag()}")
    plt.show()

## Part 10 — Runs

Everything above is generic. From here on, each subsection is purely a
config (from Part 2, or a variant built from it) plus calls into Parts 4-9.

**Toggling distributions and copulas:** every run below is built from a
`StudyConfig`, and a `StudyConfig`'s hazard, loss, and copula are each just a
name plus a parameter dict (`DistributionSpec`/`CopulaSpec`, Part 2). To point
any run below at a different choice, take a preset (`LIGHT_PRESET`/
`FULL_PRESET`) or an existing config and override just that piece with
`dataclasses.replace` -- nothing else needs to change. For example, to swap
the loss margin for a GPD instead of whatever the base config uses:
```python
alt_config = replace(LIGHT_PRESET, scenario=replace(
    LIGHT_PRESET.scenario, loss=DistributionSpec("gpd", dict(alpha=2.5)),
))
```
and the same pattern with `hazard=` or `copula=` swaps the other two pieces.
Part 2's "Available distributions"/"Available copulas" tables list every
registered name and its parameters. This is exactly the substitution 10a uses
below (swapping `LIGHT_PRESET`'s lognormal loss for a Dagum one) and the one
10c uses to sweep a parameter grid.

### 10a. Light-scale run

`LIGHT_PRESET` (Part 2) is a small-scale scenario -- Gumbel copula, lognormal
losses, `N=100_000`, `n=500` -- kept available as a fast, ready-to-run
baseline and as the base config Part 10c's sweep builds from. The next cell
is a **parameter panel**: every `ScenarioConfig`/`RunConfig` field for the
scenario this notebook actually runs by default (`ACTIVE`), each as its own
named, commented variable -- edit values there directly rather than writing
a `dataclasses.replace` call. As shipped it reproduces `LIGHT_PRESET` with
the loss margin swapped from lognormal to **Dagum** (`alpha=2.0, p=1.0`),
hazard left at Weibull. Two related global toggles live elsewhere, not
duplicated in the panel: `SAVE_FIGURES` (Part 3) and `RUN_FULL_SCALE`
(Part 10b).

**A reading note on the VaR tables below:** with a subsample size of `n=500`,
no empirical quantile is defined above `1 - 1/n = 0.998`, so the `q=0.999`
and `q=0.9999` rows of `"Empirical subsample"` collapse to the same number --
the subsample maximum -- for every subsample, by construction of
`np.quantile(..., method="inverted_cdf")`. IDR's marginal VaR at those same
levels lands on exactly the same value for a related reason: its mixture CDF
is closed at $1.0$ at the largest observed loss (`IDRDiscreteDistribution`'s
`close_at_max` convention), which *is* the subsample maximum. Both are
correct, expected behavior of a small-$n$ empirical estimate at an extreme
quantile, not a computation bug -- and closing exactly that gap is what a
dedicated extreme-value tail model would be for (not implemented here).

**Two cells below:** the parameter panel (builds the Weibull/Dagum
`ACTIVE` config from the named variables above it), then
`load_or_run_bootstrap_study(ACTIVE, RESULTS_DIR)` -- skipping the expensive
fitting pass if this scenario is already cached on disk (Part 9), otherwise
falling through to `run_bootstrap_study`, the single entry point that wires
together everything above: `build_scenario` (Parts 4-6) constructs the hazard/loss
distributions, copula, and exact `ConditionalTargets`; `generate_population_and_subsamples`
(Part 8) draws the population and bootstrap subsamples; `run_fullbaseline_predictions_both`
(Part 8) fits every configured method on every subsample and aggregates;
`build_summary_tables`/`plot_*` (Part 9) report the results -- purely a
config plus function calls.

In [ ]:
# ============================================================================
# Primary scenario parameter panel -- edit these, then rerun this cell and
# everything below in Part 10a/10e. See Part 2's "Available distributions"/
# "Available copulas" tables for every registered name and its parameters,
# and the "Metaparameters" note earlier in this Part for what each
# RunConfig field controls. Unrelated global toggles: SAVE_FIGURES (Part 3),
# RUN_FULL_SCALE (Part 10b) -- not duplicated here, see those cells.
# ============================================================================

# -- Data-generating process (ScenarioConfig) --
PRIMARY_SEED = 1                     # population/subsample RNG seed
PRIMARY_N = 100_000                  # synthetic population size

PRIMARY_HAZARD_DIST = "weibull_min"
PRIMARY_HAZARD_PARAMS = dict(c=2 / 3, loc=0.0, scale=1.0)

PRIMARY_LOSS_DIST = "dagum"
PRIMARY_LOSS_PARAMS = dict(alpha=2.0, p=1.0)

PRIMARY_COPULA = "gumbel"
PRIMARY_COPULA_PARAMS = dict(theta=2.0)   # dependence strength (larger theta = stronger tail dependence, Gumbel)
PRIMARY_COPULA_MIRROR = None              # None -> registry default; only meaningful for "lorenz"

# -- Bootstrap/evaluation mechanics (RunConfig) --
PRIMARY_SUBSAMPLE_SIZE = 500          # n: size of one bootstrap resample ("one real dataset")
PRIMARY_ITERATIONS = 100              # number of bootstrap resamples
PRIMARY_BINS = 50                     # bins for methods trained on "binned" data
PRIMARY_VAR_LEVELS = (0.95, 0.99, 0.999, 0.9999)
PRIMARY_METHODS = _DEFAULT_METHODS    # override with a custom tuple of MethodConfig(...) to change the method roster
PRIMARY_DATA_BACKEND = "hdf5"        # "memory" regenerates each call; "hdf5" persists to disk
                                      # (used by every Part 10 run below, including this one)

# -- Plot-level selection --
PLOT_SUBSAMPLE_SEED = 10               # which random subsample plot_upper_tail_comparison (Part 10e) displays

ACTIVE = StudyConfig(
    scenario=ScenarioConfig(
        seed=PRIMARY_SEED, N=PRIMARY_N,
        hazard=DistributionSpec(PRIMARY_HAZARD_DIST, PRIMARY_HAZARD_PARAMS),
        loss=DistributionSpec(PRIMARY_LOSS_DIST, PRIMARY_LOSS_PARAMS),
        copula=CopulaSpec(PRIMARY_COPULA, PRIMARY_COPULA_PARAMS, mirror=PRIMARY_COPULA_MIRROR),
    ),
    run=RunConfig(
        n=PRIMARY_SUBSAMPLE_SIZE, iterations=PRIMARY_ITERATIONS, bins=PRIMARY_BINS,
        var_levels=PRIMARY_VAR_LEVELS, methods=PRIMARY_METHODS, data_backend=PRIMARY_DATA_BACKEND,
        collect_diagnostics=True,
    ),
    label="light",
)


In [ ]:
light_result, light_from_cache = load_or_run_bootstrap_study(ACTIVE, RESULTS_DIR)
if not light_from_cache:
    save_results(light_result, RESULTS_DIR)
plot_joint_sample(light_result)
validate_copula_sample(light_result)

light_tables = build_summary_tables(light_result)
display_summary_tables(light_tables)

plot_three_panel_boxplots(light_result)

_ = plot_sanity_check(light_result)


### 10b. Full-scale run

A production-scale sweep over every combination of loss margin (dagum,
pareto, lognormal), hazard margin (Weibull, and a Generalised Pareto hazard
with $\xi=0.3$ -- the same `gpd` distribution registered in Part 4, used
in the `hazard=` slot instead of `loss=`; nothing in the registry ties a
distribution to one role or the other), and copula (Gumbel, Lorenz) -- $3
\times 2 \times 2 = 12$ scenarios, each at `N=500_000`, `n=500`,
`iterations=100_000`, with an HDF5-cached population + subsample-index table
(`DATA_DIR`) so re-running doesn't regenerate data already on disk. This is
very expensive (12x a single already-expensive full-scale run) -- guarded by
`RUN_FULL_SCALE`, off by default; see `docs/reproducibility.md` for expected
runtime.

**Cell below:** builds each scenario from `FULL_PRESET` via
`dataclasses.replace`, swapping only `hazard`/`loss`/`copula`; runs
`run_bootstrap_study`; plots the joint sample and three-panel boxplots and
runs the sanity check (each figure saved to `PLOTS_DIR` if `SAVE_FIGURES` is
on); and saves results to `RESULTS_DIR` via `save_results` -- all keyed by
that scenario's own `StudyConfig.cache_tag()`, so the 12 scenarios' cached
populations, results, and figures never collide.


In [ ]:
FULL_SWEEP_LOSSES = {
    "dagum": DistributionSpec("dagum", dict(alpha=2.0, p=1.0)),
}
FULL_SWEEP_HAZARDS = {
    "weibull": DistributionSpec("weibull_min", dict(c=2 / 3, loc=0.0, scale=1.0)),
    "gpd": DistributionSpec("gpd", dict(alpha=1 / 0.3)),  # xi = 1/alpha = 0.3
}
FULL_SWEEP_COPULAS = {
    "gumbel": CopulaSpec("gumbel", dict(theta=2.0)),
    "lorenz": CopulaSpec("lorenz", dict(), mirror=True),
}

RUN_FULL_SCALE = True  # 12 scenarios x N=500_000 x iterations=100_000 -- very expensive, see docs/reproducibility.md

import os

# Caps how many NEWLY-COMPUTED (not cache-hit) scenarios one invocation of
# this cell runs before stopping -- unset/0 means unlimited, i.e. today's
# "whole sweep in one process" behavior. Set by the chained orchestrator
# (notebooks/run_full_sweep.sh) to 1, so a crash never costs more than one
# scenario's fitting progress; already-cached scenarios are still
# fast-forwarded through for free within the same invocation.
SWEEP_MAX_NEW_SCENARIOS = int(os.environ.get("SWEEP_MAX_NEW_SCENARIOS", "0")) or None

if RUN_FULL_SCALE:
    full_results = {}
    new_scenario_count = 0
    stop_sweep = False
    for loss_name, loss_spec in FULL_SWEEP_LOSSES.items():
        if stop_sweep:
            break
        for hazard_name, hazard_spec in FULL_SWEEP_HAZARDS.items():
            if stop_sweep:
                break
            for copula_name, copula_spec in FULL_SWEEP_COPULAS.items():
                if stop_sweep:
                    break
                scenario_config = replace(
                    FULL_PRESET,
                    scenario=replace(FULL_PRESET.scenario, hazard=hazard_spec, loss=loss_spec, copula=copula_spec),
                )
                print(f"--- {copula_name} / {hazard_name} hazard / {loss_name} loss ---")
                result, from_cache = load_or_run_bootstrap_study(scenario_config, RESULTS_DIR)
                full_results[(loss_name, hazard_name, copula_name)] = result

                plot_joint_sample(result)
                plot_three_panel_boxplots(result)
                _ = plot_sanity_check(result)
                if from_cache:
                    print(f"Results already present for {scenario_config.cache_tag()}; skipped re-saving.")
                else:
                    saved_paths = save_results(result, RESULTS_DIR)
                    print("Saved:", saved_paths)
                    new_scenario_count += 1
                    if SWEEP_MAX_NEW_SCENARIOS is not None and new_scenario_count >= SWEEP_MAX_NEW_SCENARIOS:
                        print(f"Reached SWEEP_MAX_NEW_SCENARIOS={SWEEP_MAX_NEW_SCENARIOS}; stopping this invocation.")
                        stop_sweep = True

    print(f"Completed {len(full_results)} full-scale scenarios this invocation ({new_scenario_count} newly computed).")
else:
    print("RUN_FULL_SCALE is False; skipping the full-scale sweep. Set it to True to run it.")


### 10c. Tail-heaviness sensitivity sweep

`run_sensitivity_sweep` varies **one** loss-margin parameter over a grid,
re-running `run_bootstrap_study` at each grid point and summarizing the %
error in Loss VaR/ES at one hazard-quantile level `q` for every method plus
the empirical-subsample baseline. Because it is built on `StudyConfig`, it
works for any registered loss distribution -- lognormal `s`, Pareto `b`,
Dagum `alpha`/`p`, or GPD `alpha` (Part 2's "Available distributions" table
has the full parameter list).

**To point the sweep at a different distribution or parameter:** build a
`base_config` whose `scenario.loss` is that distribution's `DistributionSpec`
-- any other parameter already set there stays fixed while `param` varies,
which is how the two Dagum sweeps below hold `p`/`alpha` at a reference value
while sweeping the other -- then call `run_sensitivity_sweep(base_config,
param="the_parameter_name", values=[...])`. The hazard and copula pass
through `base_config.scenario` untouched, so swapping either one (same
`dataclasses.replace` pattern as Part 10's intro) changes the whole sweep's
setting without touching `run_sensitivity_sweep` itself.

**Cell below:** `_sweep_row`/`run_sensitivity_sweep`/`plot_sensitivity_sweep`/
`plot_sweep_grid`, plus two example sweeps over the Dagum loss margin's tail
index $\alpha$ and body shape $p$. For each grid value of the swept
parameter, a fresh `StudyConfig` is built (via `dataclasses.replace`),
`run_bootstrap_study` is re-run, and the % error in
$\mathrm{VaR}_q(L)$/$\mathrm{ES}_q(L)$ at one fixed level $q$ is recorded per
method -- same formula as Part 8's `%error`, just tabulated across the
parameter grid instead of across VaR levels. Each panel plots each method's
**median** % error across bootstrap iterations (solid; the empirical
baseline dashed), with an $\times$ marking grid points where the quantity
failed to exist (e.g. an infinite ES) in more than a fifth of the bootstrap
iterations -- a legend entry in `plot_sweep_grid`'s shared legend explains
what that marker means.

`plot_sweep_grid` keeps the figure itself to title + one shared $y$-axis
label + legend; each panel keeps its own $x$-axis label (a single label
centered under both panels read as ambiguous about which panel it belonged
to). Run-specific detail (which hazard margin, which parameter is held fixed
at what value, which direction is the heavier tail) is instead printed as a
plain caption line immediately above each figure. `methods_to_plot` controls
which methods are drawn without recomputing anything.

**Why Quadratic's curve looks jagged as a function of $\alpha$:** this is not
Monte Carlo noise. Quadratic (Part 7a) fits its quadratic coefficient with a
significance test and deliberately returns NaN for the whole bootstrap
iteration -- with no linear fallback -- whenever that test doesn't reject.
The resulting per-$\alpha$ failure rate (an $\times$ marker once it passes
`failure_threshold_pct`) is a structural property of the signal-to-noise in
the binned design at that $\alpha$, not sampling jitter -- it peaks near
$\alpha=3$ (roughly half of all iterations, at the 10k-iteration reference
scale) and is low at both ends of the swept range. More bootstrap iterations
sharpen the estimate of that failure rate; they do not smooth it away, so
raising `iterations` further is not expected to make this curve look
cleaner.

The Dagum example below is pinned to its own population size and seed
rather than inheriting them from `LIGHT_PRESET` -- so its numbers can be
checked directly against a companion implementation of the same sweep. Any other sweep built from this cell's functions is free to use
`LIGHT_PRESET`/`ACTIVE`'s settings and the full default method list instead,
as `run_sensitivity_sweep`'s docstring above describes.

In [ ]:
def _sweep_row(distribution, parameter_name, parameter_value, method, risk_measure, values, true_value, q):
    values = np.asarray(values, dtype=float)
    finite = values[np.isfinite(values)]
    n_total = values.size
    mean_estimate = float(np.mean(finite)) if finite.size else np.nan
    median_estimate = float(np.median(finite)) if finite.size else np.nan
    pct_error = (100.0 * (mean_estimate - true_value) / true_value
                 if finite.size and np.isfinite(true_value) and true_value != 0 else np.nan)
    median_pct_error = (100.0 * (median_estimate - true_value) / true_value
                 if finite.size and np.isfinite(true_value) and true_value != 0 else np.nan)
    return {
        "distribution": distribution, "parameter": parameter_name, "parameter_value": float(parameter_value),
        "method": method, "risk_measure": risk_measure, "q": q, "true_value": true_value,
        "mean_estimate": mean_estimate, "median_estimate": median_estimate,
        "pct_error": pct_error, "median_pct_error": median_pct_error,
        "n_total": n_total, "n_valid": int(finite.size),
        "failure_rate_pct": 100.0 * (n_total - finite.size) / n_total if n_total else np.nan,
    }



def _sweep_grid_point_rows(base_config: "StudyConfig", param: str, value: float, q: float) -> list[dict]:
    """Run one sensitivity-sweep grid point (one parameter value) and
    return its per-(method, risk_measure) rows -- the unit both
    `run_sensitivity_sweep` and `load_or_run_sensitivity_sweep` build on."""
    loss_params = {**base_config.scenario.loss.params, param: value}
    scenario = replace(base_config.scenario, loss=DistributionSpec(base_config.scenario.loss.name, loss_params))
    config = replace(base_config, scenario=scenario)
    result = run_bootstrap_study(config, cache_tag_suffix=f"_{param}{value:g}")

    true_var = float(result.built.loss_dist.ppf(q))
    true_es = float(result.built.loss_dist.expected_shortfall(q))
    dist_name = config.scenario.loss.name

    point_at_q = result.point_df[np.isclose(result.point_df["var_level"], q)]
    sample_metrics = point_at_q[["subsample_id", "loss_var_sub", "loss_es_sub"]].drop_duplicates(subset=["subsample_id"])
    sample_var = sample_metrics["loss_var_sub"].to_numpy(dtype=float)
    sample_es = sample_metrics["loss_es_sub"].to_numpy(dtype=float)
    rows = [
        _sweep_row(dist_name, param, value, "Empirical subsample", "VaR", sample_var, true_var, q),
        _sweep_row(dist_name, param, value, "Empirical subsample", "ES", sample_es, true_es, q),
    ]
    for method, group in point_at_q.groupby("method"):
        rows.append(_sweep_row(dist_name, param, value, method, "VaR", group["damage_var_sub"].to_numpy(dtype=float), true_var, q))
        rows.append(_sweep_row(dist_name, param, value, method, "ES", group["damage_es_sub"].to_numpy(dtype=float), true_es, q))

    # Tail CE: E[L | H >= VaR_q(H)], hazard-conditioned rather than
    # loss-conditioned like VaR/ES above -- pulled from tail_df, which
    # already carries the empirical `tail_mean_sub` baseline (Part 9) and
    # each method's `prediction` at this q from the same `run_bootstrap_study`
    # call, at no extra fitting cost.
    tail_at_q = result.tail_df[np.isclose(result.tail_df["var_level"], q)]
    sample_tail = tail_at_q[["subsample_id", "tail_mean_sub"]].drop_duplicates(subset=["subsample_id"])
    true_tail = float(tail_at_q["true_full"].iloc[0])
    rows.append(_sweep_row(dist_name, param, value, "Empirical subsample", "Tail CE",
                            sample_tail["tail_mean_sub"].to_numpy(dtype=float), true_tail, q))
    for method, group in tail_at_q.groupby("method"):
        rows.append(_sweep_row(dist_name, param, value, method, "Tail CE", group["prediction"].to_numpy(dtype=float), true_tail, q))
    return rows


def run_sensitivity_sweep(base_config: "StudyConfig", param: str, values: Iterable[float], *, q: float = 0.99) -> pd.DataFrame:
    """Vary one loss-margin parameter over a grid, rerunning run_bootstrap_study
    at each point; summarize % error in Loss VaR/ES at level q.

    Any parameter already set on `base_config.scenario.loss.params` other than
    `param` stays fixed at its `base_config` value -- e.g. sweeping Dagum's
    `alpha` with `p` fixed, or vice versa. To sweep a different loss family
    entirely, build `base_config` with that family's `DistributionSpec` (Part
    2's "Available distributions" table lists every registered name and its
    parameters) and pass the matching `param=`.
    """
    if not any(np.isclose(q, lvl) for lvl in base_config.run.var_levels):
        raise ValueError(f"q={q} must be one of base_config.run.var_levels={base_config.run.var_levels}.")

    rows = []
    for value in values:
        rows.extend(_sweep_grid_point_rows(base_config, param, value, q))
    return pd.DataFrame(rows)


SENSITIVITY_NEW_POINTS_THIS_INVOCATION = 0


def load_or_run_sensitivity_sweep(base_config: "StudyConfig", param: str, values: Iterable[float],
                                  cache_path: Path, *, q: float = 0.99) -> pd.DataFrame:
    """Like `run_sensitivity_sweep`, but persists each grid point's rows to
    `cache_path` as soon as they're computed and skips any parameter value
    already present there -- the sweep-level counterpart to
    `load_or_run_bootstrap_study` (Part 9). A crash mid-sweep only loses the
    grid point in progress, not the whole sweep. Respects
    `base_config.run.rebuild_cache` the same way the scenario-level cache
    does (bypasses and rebuilds this cache too when True).

    Capped by `SENSITIVITY_SWEEP_MAX_NEW_POINTS` (env var, unset/0 =
    unlimited) -- shared across every call in the same process, so calling
    this once for the alpha-sweep and once for the p-sweep in the same
    invocation stops at the cap regardless of which sweep it falls in.
    """
    import os

    if not any(np.isclose(q, lvl) for lvl in base_config.run.var_levels):
        raise ValueError(f"q={q} must be one of base_config.run.var_levels={base_config.run.var_levels}.")

    cache_path = Path(cache_path)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    if base_config.run.rebuild_cache and cache_path.exists():
        cache_path.unlink()
    cached_df = pd.read_csv(cache_path) if cache_path.exists() else pd.DataFrame()
    done_values = set(np.round(cached_df["parameter_value"].to_numpy(dtype=float), 9)) if not cached_df.empty else set()

    max_new = int(os.environ.get("SENSITIVITY_SWEEP_MAX_NEW_POINTS", "0")) or None
    global SENSITIVITY_NEW_POINTS_THIS_INVOCATION

    all_rows = [cached_df] if not cached_df.empty else []
    for value in values:
        key = round(float(value), 9)
        if key in done_values:
            continue
        if max_new is not None and SENSITIVITY_NEW_POINTS_THIS_INVOCATION >= max_new:
            print(f"Reached SENSITIVITY_SWEEP_MAX_NEW_POINTS={max_new}; stopping this invocation.")
            break
        rows = _sweep_grid_point_rows(base_config, param, value, q)
        row_df = pd.DataFrame(rows)
        all_rows.append(row_df)
        row_df.to_csv(cache_path, mode="a" if cache_path.exists() else "w", header=not cache_path.exists(), index=False)
        SENSITIVITY_NEW_POINTS_THIS_INVOCATION += 1
        print(f"Computed and saved grid point {param}={value:g} ({SENSITIVITY_NEW_POINTS_THIS_INVOCATION} new this invocation).")

    return pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()


_SWEEP_DISPLAY_NAMES = {"Empirical subsample": "Empirical"}


def plot_sensitivity_sweep(sweep_df: pd.DataFrame, risk_measure: Literal["VaR", "ES", "Tail CE"] = "VaR", title: str | None = None,
                            xlabel: str | None = None, ylabel: str | None = None, methods_to_plot: list[str] | None = None,
                            ax=None, failure_threshold_pct: float = 20.0):
    """One sweep panel: the solid line is each method's **median** % error
    across bootstrap iterations, the dashed line is the empirical
    (no-damage-function) baseline, and an $\times$ marks a grid point where
    the quantity failed to exist (e.g. an infinite ES) in more than
    `failure_threshold_pct` of iterations -- see `plot_sweep_grid`'s shared
    legend for that marker's meaning when this panel is composed there.
    `methods_to_plot`, if given, both restricts and orders which methods are
    drawn (default: every method present in `sweep_df`, in first-seen order).
    """
    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(7.5, 4.8), facecolor="white")

    sub = sweep_df[sweep_df["risk_measure"].eq(risk_measure)]
    available = list(dict.fromkeys(sub["method"]))
    method_order = [m for m in methods_to_plot if m in available] if methods_to_plot is not None else available
    palette = _palette_for(method_order)

    value_to_pos = {v: i for i, v in enumerate(sorted(sub["parameter_value"].unique()))}

    for method in method_order:
        mdf = sub[sub["method"].eq(method)].sort_values("parameter_value")
        if mdf.empty:
            continue
        empirical = method == "Empirical subsample"
        color = palette.get(method)
        ax.plot(mdf["parameter_value"].map(value_to_pos), mdf["median_pct_error"], marker="o", markersize=4.2, markeredgecolor="white",
                markeredgewidth=0.6, linewidth=1.5, color=color, linestyle="--" if empirical else "-",
                label=_SWEEP_DISPLAY_NAMES.get(method, method), zorder=3)
        bad = mdf["failure_rate_pct"] > failure_threshold_pct
        if bad.any():
            ax.plot(mdf.loc[bad, "parameter_value"].map(value_to_pos), mdf.loc[bad, "median_pct_error"], "x",
                    markersize=9, markeredgewidth=1.6, color=color, zorder=4)

    ax.axhline(0.0, color=METHOD_COLORS["Truth"], linewidth=1.0, alpha=0.8, zorder=2)
    if xlabel:
        ax.set_xlabel(xlabel, labelpad=8)
    resolved_ylabel = ylabel if ylabel is not None else (f"error in {risk_measure} (%)" if own_fig else None)
    if resolved_ylabel:
        ax.set_ylabel(resolved_ylabel, labelpad=8)
    ax.set_xticks(list(value_to_pos.values()))
    ax.set_xticklabels([f"{v:g}" for v in value_to_pos])
    if title:
        ax.set_title(title, fontsize=13.0, pad=10)

    if own_fig:
        ax.legend(loc="best", fontsize=11.0, ncol=2)
        fig.tight_layout(pad=0.7)
        tag = f"{sub['distribution'].iloc[0]}_{sub['parameter'].iloc[0]}" if not sub.empty else "sweep"
        save_figure(fig, f"sensitivity_sweep_{risk_measure}_{tag}")
        plt.show()


def plot_sweep_grid(sweep_df: pd.DataFrame, title: str, xlabel: str, methods_to_plot: list[str] | None = None,
                     failure_threshold_pct: float = 20.0):
    """VaR and ES panels side by side for one sweep, with one shared legend
    and one shared y-axis label below/beside both panels -- the layout used
    below. Each panel keeps its own x-axis label (a single label centered
    under both panels reads as ambiguous about which panel it belongs to).
    Panels are **not** forced onto a common y-range: ES errors are
    intrinsically larger than VaR errors, and a shared scale would flatten
    the VaR panel.
    """
    fig, axes = plt.subplots(1, 3, figsize=(19.0, 4.8), facecolor="white")
    q_val = float(sweep_df["q"].iloc[0]) if not sweep_df.empty else None
    var_title = rf"$\mathrm{{VaR}}_{{{q_val:g}}}(L)$" if q_val is not None else "VaR"
    es_title = rf"$\mathrm{{ES}}_{{{q_val:g}}}(L)$" if q_val is not None else "ES"
    tail_title = rf"$\mathbb{{E}}[L\mid H\geq \mathrm{{VaR}}_{{{q_val:g}}}(H)]$" if q_val is not None else "Tail CE"
    plot_sensitivity_sweep(sweep_df, risk_measure="VaR", title=var_title, xlabel=xlabel, ylabel="",
                            methods_to_plot=methods_to_plot, ax=axes[0], failure_threshold_pct=failure_threshold_pct)
    plot_sensitivity_sweep(sweep_df, risk_measure="ES", title=es_title, xlabel=xlabel, ylabel="",
                            methods_to_plot=methods_to_plot, ax=axes[1], failure_threshold_pct=failure_threshold_pct)
    plot_sensitivity_sweep(sweep_df, risk_measure="Tail CE", title=tail_title, xlabel=xlabel, ylabel="",
                            methods_to_plot=methods_to_plot, ax=axes[2], failure_threshold_pct=failure_threshold_pct)

    handles, labels = axes[0].get_legend_handles_labels()
    # Restricted to the plotted methods -- checking the raw sweep_df would add this
    # legend entry even when the only method exceeding failure_threshold_pct is one
    # methods_to_plot excludes, with no "x" actually visible anywhere in the figure.
    plotted = sweep_df[sweep_df["method"].isin(methods_to_plot)] if methods_to_plot is not None else sweep_df
    if (plotted["failure_rate_pct"] > failure_threshold_pct).any():
        handles.append(Line2D([0], [0], marker="x", linestyle="none", color=METHOD_COLORS["Truth"],
                               markersize=9, markeredgewidth=1.6))
        labels.append(f">{failure_threshold_pct:g}% failure rate")
    fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, -0.02), ncol=min(len(labels), 4),
              frameon=False, fontsize=10.5)

    fig.suptitle(title, y=1.03, fontsize=13.5)
    fig.supylabel("Median relative error (%)", fontsize=14.0)
    fig.tight_layout(rect=(0.02, 0.14, 1, 0.94), pad=0.8)
    tag = f"{sweep_df['distribution'].iloc[0]}_{sweep_df['parameter'].iloc[0]}" if not sweep_df.empty else "sweep"
    save_figure(fig, f"sweep_grid_{tag}")
    plt.show()


# Example: a two-parameter Dagum sweep. Dagum's tail index alpha and body
# shape p are swept independently -- each sweep holds the other parameter
# fixed at a reference value -- at q=0.99, small scale (raise `iterations`
# for a production-fidelity sweep, at proportionally higher cost).
#
# This example deliberately does NOT use LIGHT_PRESET's population size and
# seed: it is pinned to a specific (seed, N) so its numbers are directly
# comparable to a companion implementation of this same sweep, at a larger
# population for less small-N sampling noise in the "true" targets.
ALPHA_REF, P_REF = 2.0, 1.0
SWEEP_ALPHAS = [1.25, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]
SWEEP_PS = [0.25, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0]
SWEEP_Q = 0.99
SWEEP_SEED = 20260805
SWEEP_METHOD_NAMES = {"Quadratic", "Sigmoid", "Isotonic", "Isotonic (raw)", "IDR"}

methods_to_plot = ["Empirical subsample", "IDR", "Isotonic", "Isotonic (raw)", "Quadratic", "Sigmoid"]


def _hazard_label(name: str) -> str:
    """Registry name -> the short hazard label used in sweep-grid titles,
    e.g. "weibull_min" -> "weibull"."""
    return {"weibull_min": "weibull"}.get(name, name)


def _sweep_grid_title(scenario: "ScenarioConfig", suffix: str) -> str:
    """Main title + trailing "(copula, loss, hazard)" parenthetical -- same
    convention as every other figure in this notebook that reports
    `cfg.scenario.copula.name`/`.loss.name` (Part 9's `plot_joint_sample`,
    `plot_three_panel_boxplots`, etc.), extended with the hazard margin
    since this cell's two alpha-sweep figures are otherwise identical."""
    return (f"{suffix} ({scenario.copula.name} copula, {scenario.loss.name} losses, "
            f"{_hazard_label(scenario.hazard.name)} hazard)")

# Alpha-sweep: N=500k, iterations=20,000, run under both hazard margins used
# in the main full-scale sweep (cell cb90f4e3) -- same Weibull/GPD specs,
# Gumbel copula fixed both times. hdf5-cached and resumable via
# load_or_run_sensitivity_sweep + run_sensitivity_sweep.sh.
# The CSV cache path folds in the iteration count (unlike the hdf5 cache,
# `load_or_run_sensitivity_sweep`'s cache only keys on `parameter_value`, so
# a bare filename would silently keep serving rows computed at a stale
# iteration count after bumping `iterations` here) -- raising it again later
# starts a fresh cache file rather than reusing this one.
#
# On iterations=20,000 rather than 100,000: Quadratic's curve here is
# visibly jagged as a function of alpha, but that ruggedness is NOT Monte
# Carlo noise reducible by more iterations. Quadratic (Part 7a) fits its
# quadratic coefficient with a significance test and returns NaN for the
# whole iteration -- deliberately, with no linear fallback -- whenever that
# test doesn't reject; the resulting per-alpha failure rate is a structural
# property of the signal-to-noise in the binned design at that alpha (it
# peaks near alpha=3 at ~49% of iterations at the 10k-iteration reference
# scale, vs. ~4-11% at the swept range's extremes), not sampling jitter. More
# iterations only estimate that same failure rate more precisely -- they
# don't change it, so 100,000 iterations reproduced the same hump, just
# measured more tightly, at 10x the cost and enough memory to reliably OOM
# this machine. 20,000 is a modest, safely-scoped bump over the original
# 10,000 for a somewhat tighter failure-rate estimate, not an attempt to
# smooth the hump away.
SWEEP_HAZARDS = {
    "weibull": DistributionSpec("weibull_min", dict(c=2 / 3, loc=0.0, scale=1.0)),
    "gpd": DistributionSpec("gpd", dict(alpha=1 / 0.3)),  # xi = 1/alpha = 0.3
}

dagum_sweep_base_large = replace(
    LIGHT_PRESET,
    scenario=replace(LIGHT_PRESET.scenario, loss=DistributionSpec("dagum", dict(alpha=ALPHA_REF, p=P_REF)),
                     seed=SWEEP_SEED, N=500_000),
    run=replace(LIGHT_PRESET.run, iterations=20_000, data_backend="hdf5",
               methods=tuple(m for m in _DEFAULT_METHODS if m.name in SWEEP_METHOD_NAMES)),
)

for hazard_name, hazard_spec in SWEEP_HAZARDS.items():
    base = replace(dagum_sweep_base_large, scenario=replace(dagum_sweep_base_large.scenario, hazard=hazard_spec))
    print(f"Fixed hazard: {base.scenario.hazard}  |  Fixed copula: {base.scenario.copula}")
    alpha_sweep_cache = RESULTS_DIR / f"sweep_dagum_alpha_{hazard_name}_iter{dagum_sweep_base_large.run.iterations}.csv"
    alpha_sweep_df = load_or_run_sensitivity_sweep(base, param="alpha", values=SWEEP_ALPHAS,
                                                   cache_path=alpha_sweep_cache, q=SWEEP_Q)
    if alpha_sweep_df.empty:
        # SENSITIVITY_SWEEP_MAX_NEW_POINTS is one shared budget across both
        # hazards in this loop (and the p-sweep below) -- when the other
        # hazard already spent it this invocation and this hazard has no
        # cached points of its own yet, there's nothing to display/plot.
        print(f"No cached or newly-computed points yet for {hazard_name}; skipping its plot this invocation.")
        continue
    display(alpha_sweep_df)
    print(f"Dagum tail index alpha swept, p={P_REF:g} fixed. Smaller alpha = heavier loss tail.")
    plot_sweep_grid(alpha_sweep_df, title=_sweep_grid_title(base.scenario, "Sensitivity to the Dagum tail index"),
                     xlabel=r"$\alpha$", methods_to_plot=methods_to_plot)

# P-sweep: small scale, as originally coded -- Weibull hazard only, no
# scale-up, no caching (cheap enough not to need it).
dagum_sweep_base_small = replace(
    LIGHT_PRESET,
    scenario=replace(LIGHT_PRESET.scenario, loss=DistributionSpec("dagum", dict(alpha=ALPHA_REF, p=P_REF)),
                     seed=SWEEP_SEED, N=200_000),
    run=replace(LIGHT_PRESET.run, iterations=100,
               methods=tuple(m for m in _DEFAULT_METHODS if m.name in SWEEP_METHOD_NAMES)),
)

print(f"Fixed hazard: {dagum_sweep_base_small.scenario.hazard}  |  Fixed copula: {dagum_sweep_base_small.scenario.copula}")
p_sweep_df = run_sensitivity_sweep(dagum_sweep_base_small, param="p", values=SWEEP_PS, q=SWEEP_Q)
display(p_sweep_df)
print(f"Dagum body shape p swept, alpha={ALPHA_REF:g} fixed.")
plot_sweep_grid(p_sweep_df, title=_sweep_grid_title(dagum_sweep_base_small.scenario, "Sensitivity to the Dagum body shape"),
                 xlabel=r"$p$", methods_to_plot=methods_to_plot)

### 10e. Extended diagnostic figures

Five views of each of two scenarios, back to back: whichever run is
currently `light_result` (or, if `RUN_FULL_SCALE` above was switched on,
the first scenario in `full_results` -- the sweep's dict of every scenario
it has processed, keyed by `(loss_name, hazard_name, copula_name)`) --
labelled "Weibull hazard" below -- and the same n/iterations with Copula:
Gumbel / Hazard: `gpd` ($\xi=0.3$) / Loss: Dagum instead, the exact
canonical combination Part 10b's `FULL_SWEEP_HAZARDS["gpd"]`/
`FULL_SWEEP_LOSSES["dagum"]`/`FULL_SWEEP_COPULAS["gumbel"]` define, labelled
"GPD hazard" below. Everything below is generic in `result` and works
identically for any config, exactly like
`plot_joint_sample`/`plot_grouped_boxplots` above. Both scenarios go
through `load_or_run_bootstrap_study` (Part 9), the same light-scale
bootstrap cost `light_result` already paid (n=500, iterations=100), so a
first run of this cell after clearing the cache takes roughly twice as
long as 10a alone; a re-run with both scenarios already cached on disk
pays neither cost.

**Cell below:** builds the GPD-hazard scenario, then for each of the two
scenarios: the upper-tail damage-curve comparison (9c), the full and
trimmed hazard-loss risk diagrams (9d), and the IDR conditional-law figure
(9e) -- 10 figures total.

In [ ]:
demo_result = next(iter(full_results.values())) if RUN_FULL_SCALE else light_result

# Second scenario: same n/iterations, but Copula: Gumbel / Hazard: gpd /
# Loss: Dagum -- the exact canonical combination Part 10b's full sweep uses
# (FULL_SWEEP_HAZARDS["gpd"] is xi=0.3), reused by name here rather than
# re-declaring a new ad hoc DistributionSpec for the same distribution.
gpd_hazard_config = replace(ACTIVE, scenario=replace(
    ACTIVE.scenario,
    hazard=FULL_SWEEP_HAZARDS["gpd"], loss=FULL_SWEEP_LOSSES["dagum"], copula=FULL_SWEEP_COPULAS["gumbel"],
))
demo_result_gpd_hazard, gpd_hazard_from_cache = load_or_run_bootstrap_study(
    gpd_hazard_config, RESULTS_DIR, cache_tag_suffix="_gpdhazard",
)
if not gpd_hazard_from_cache:
    save_results(demo_result_gpd_hazard, RESULTS_DIR, cache_tag_suffix="_gpdhazard")

for hazard_label, result in [("Weibull hazard", demo_result), ("GPD hazard", demo_result_gpd_hazard)]:
    display(HTML(f"<h4>{hazard_label}</h4>"))
    plot_upper_tail_comparison(result, active_methods=("Sigmoid",), subsample_seed=PLOT_SUBSAMPLE_SEED)
    plot_hazard_loss_risk_diagram(result, mode="full")
    plot_idr_risk_diagram(result, subsample_seed=PLOT_SUBSAMPLE_SEED)
    plot_hazard_loss_risk_diagram_curves(result, mode="full")
    plot_idr_conditional_law(result, subsample_seed=PLOT_SUBSAMPLE_SEED)

## Part 11 — Notes

**Numerical-precision conventions**, used throughout Parts 5-8: copula
sampling and density evaluation are done in log-safe `(u, z)` form with
`v = expit(z)`, and losses are always drawn via `.isf(1 - v)`, never
`.ppf(v)` on a value clipped near 1 -- this keeps full double precision deep
in the loss tail, which matters most for the sensitivity sweep (10c) and the
EVT diagnostics (10d).

**Extending this notebook:**

- **New distribution** (hazard or loss): write a class exposing
  `cdf`/`ppf`/`mean`/`rvs` (optionally `sf`/`isf`/a closed-form
  `expected_shortfall`), wrap it in `DistributionAdapter`, and call
  `register_distribution("name", factory)` in Part 4.
- **New copula**: write its Pickands dependence function `A(t)` and first two
  derivatives, and call `register_copula("name", builder)` in Part 5.
- **New method**: write a fit function or a `.fit`/`.predict` class, and
  register a `MethodSpec` in Part 7.

In every case, nothing outside that one `register_...` call needs to change —
`DistributionSpec`/`CopulaSpec`/`MethodConfig` in Part 2 just need the new
name.


## Part 12 — HANZE real-data case study

Everything above fits estimators to *synthetic* data with a known
theoretical damage function, to measure each estimator's own bias/variance
against ground truth. This closing part instead fits the same estimator
families (Part 7) to **real** flood-loss data -- the HANZE v2.1.1 European
flood event database (Paprotny et al.) -- where there is no known "true"
damage function to benchmark against, only the fitted curves themselves.
Every VaR/ES/conditional-mean table below reports fitted values, not a %
error against theory, unlike every earlier Part.

The interface is a small `ipywidgets` dashboard: filter events by country/
region/year/hazard type, then click "Render analysis" to fit and plot.
Unlike every earlier Part, this one is interactive rather than
run-top-to-bottom -- the last cell still renders one default (unfiltered)
view automatically, so something appears even without interacting further.

### 12a. Loading the event data

`HANZE_events.csv` -- one row per aggregate flood event (event ID, country,
year, affected NUTS regions, reported loss in 2020 EUR, and a free-text
cause description) -- is not part of this notebook; it is located on disk
via a portable search (a sibling `data/` folder first, then a couple of
other likely locations), so this Part works regardless of exactly where the
data file ends up relative to the notebook.

**Cell below:** `_find_hanze_csv`, `HANZE_RENAME_MAP`, `split_region_codes`,
`load_hanze_events`.


In [ ]:
def _find_hanze_csv(filename: str = "HANZE_events.csv") -> Path:
    """`HANZE_events.csv` lives in the shared DATA_DIR (Part 1) alongside
    any generated population caches."""
    path = DATA_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(f"Could not find {filename!r} in {DATA_DIR}.")
    return path


HANZE_RENAME_MAP = {
    "ID": "event_id", "Country code": "country_code", "Year": "year", "Country name": "country",
    "Start date": "start_date", "End date": "end_date", "Type": "flood_type", "Flood source": "source",
    "Regions affected (v2010)": "region2010", "Regions affected (v2021)": "region2021",
    "Area flooded": "area_flooded_total", "Fatalities": "fatalities", "Persons affected": "persons_affected",
    "Losses (nominal value)": "losses_nominal", "Losses (original currency)": "currency",
    "Losses (2020 euro)": "losses_euro_total", "Cause": "cause_raw", "Notes": "notes_raw",
    "References": "references_raw", "Changes": "changes_raw",
}
HANZE_REGION_COL = "region2021"


def split_region_codes(value):
    if pd.isna(value):
        return tuple()
    return tuple(code.strip() for code in str(value).split(";") if code.strip())


def load_hanze_events(path):
    events = pd.read_csv(path, encoding="utf-8-sig").rename(columns=HANZE_RENAME_MAP)
    required = {"event_id", "year", "country", HANZE_REGION_COL, "losses_euro_total", "cause_raw"}
    missing = sorted(required.difference(events.columns))
    if missing:
        raise ValueError(f"HANZE_events.csv is missing required columns: {missing}")

    numeric_cols = ["year", "losses_euro_total", "losses_nominal", "area_flooded_total", "fatalities", "persons_affected"]
    for column in numeric_cols:
        if column in events:
            events[column] = pd.to_numeric(events[column], errors="coerce")
    for column in ["start_date", "end_date"]:
        if column in events:
            events[column] = pd.to_datetime(events[column], errors="coerce")

    events["regions"] = events[HANZE_REGION_COL].apply(split_region_codes)
    events["n_regions_affected"] = events["regions"].str.len()

    duplicate_ids = int(events["event_id"].duplicated().sum())
    if duplicate_ids:
        raise ValueError(f"Expected one row per aggregate event, but found {duplicate_ids} duplicate event IDs.")
    return events.sort_values(["year", "event_id"]).reset_index(drop=True)


HANZE_PATH = _find_hanze_csv()
hanze_events = load_hanze_events(HANZE_PATH)
print(f"Loaded {HANZE_PATH}")
print(f"Aggregate events: {len(hanze_events):,}")
print(f"Countries: {hanze_events['country'].nunique():,}")
print(f"Years: {int(hanze_events['year'].min())}–{int(hanze_events['year'].max())}")
display(hanze_events.head())


### 12b. Parsing the hazard cause text

`Cause` is free text (e.g. "Heavy rainfall, up to 120 mm in 24 hours"). A
small set of regexes extracts a coarse hazard type and, where the text gives
an amount-over-duration, both the raw rainfall amount and a derived
intensity `I = x/y`. **The main analysis uses the raw amount, not the
intensity**, as the hazard variable in the damage-function fits below --
event durations are reported too inconsistently across records for the
derived intensity to provide a sufficiently reliable, harmonised hazard
proxy, even though it remains available (e.g. in the dashboard's hazard-
variable selector) for anyone who wants to explore it anyway.

**Cell below:** `RAIN_PATTERNS`, `HAZARD_LABELS`, `parse_cause_text`, and `load_or_parse_hanze_causes` -- the Part 9-style cache (`load_or_run_bootstrap_study`) for this step: `cause_raw` parsing is the one non-vectorized, row-by-row pass in Part 12, so it's persisted to `HANZE_events_parsed.csv` under `DATA_DIR` and skipped on every later run once every current event id is already covered.


In [ ]:
RAIN_PATTERNS = [
    re.compile(
        r"(?P<amount>\d+(?:\.\d+)?)\s*mm\s*(?:in|/)\s*(?P<duration>\d+(?:\.\d+)?)\s*"
        r"(?P<unit>hours?|hrs?|hr|h|days?|day)\b", flags=re.I,
    ),
    re.compile(
        r"(?:up to\s*)?(?P<amount>\d+(?:\.\d+)?)\s*mm\b.*?\b"
        r"(?P<duration>\d+(?:\.\d+)?)\s*(?P<unit>hours?|hrs?|hr|h|days?|day)\b", flags=re.I,
    ),
    re.compile(r"(?P<amount>\d+(?:\.\d+)?)\s*mm\b", flags=re.I),
]

HAZARD_LABELS = [
    (re.compile(r"\btorrential rain\b", re.I), "torrential_rain"),
    (re.compile(r"\bheavy rainfall?\b|\bheavy rain\b", re.I), "heavy_rainfall"),
    (re.compile(r"\bextreme rainfall?\b|\bextreme rain\b", re.I), "extreme_rainfall"),
    (re.compile(r"\bsnowmelt\b", re.I), "snowmelt"),
    (re.compile(r"\briver overflow\b|\boverbank\b|\briver flood\b", re.I), "river_overflow"),
    (re.compile(r"\bstorm surge\b", re.I), "storm_surge"),
]


def parse_cause_text(text):
    result = {
        "hazard_main_type": "unknown", "rain_amount_mm": np.nan, "duration_h": np.nan,
        "duration_days": np.nan,
        # intensity_mm_per_h / equiv_mm_per_24h: computed below when both an
        # amount and a duration are matched, but NOT the hazard variable the
        # main analysis uses (see Part 12b) -- duration reporting is too
        # inconsistent across records for these derived values to be a
        # reliable hazard proxy; the raw "rain_amount_mm" is used instead.
        "intensity_mm_per_h": np.nan, "equiv_mm_per_24h": np.nan,
        "hazard_parse_confidence": "low",
    }
    if pd.isna(text):
        return result
    text = str(text)
    for pattern, label in HAZARD_LABELS:
        if pattern.search(text):
            result["hazard_main_type"] = label
            break
    match = next((pattern.search(text) for pattern in RAIN_PATTERNS if pattern.search(text)), None)
    if match is None:
        return result
    groups = match.groupdict()
    amount = pd.to_numeric(groups.get("amount"), errors="coerce")
    duration = pd.to_numeric(groups.get("duration"), errors="coerce")
    unit = (groups.get("unit") or "").lower()
    duration_h = np.nan
    if pd.notna(duration):
        duration_h = float(duration) * 24.0 if unit.startswith("day") else float(duration)
    result["rain_amount_mm"] = amount
    result["duration_h"] = duration_h
    result["duration_days"] = duration_h / 24.0 if pd.notna(duration_h) else np.nan
    if pd.notna(amount) and pd.notna(duration_h) and duration_h > 0:
        result["intensity_mm_per_h"] = amount / duration_h
        result["equiv_mm_per_24h"] = amount * 24.0 / duration_h
        result["hazard_parse_confidence"] = "high"
    elif pd.notna(amount):
        result["hazard_parse_confidence"] = "medium"
    return result


PARSED_HAZARD_COLUMNS = [
    "hazard_main_type", "rain_amount_mm", "duration_h", "duration_days",
    "intensity_mm_per_h", "equiv_mm_per_24h", "hazard_parse_confidence",
]
HANZE_PARSED_CACHE = DATA_DIR / "HANZE_events_parsed.csv"


def load_or_parse_hanze_causes(events: pd.DataFrame, cache_path: Path = HANZE_PARSED_CACHE, *, rebuild_cache: bool = False) -> pd.DataFrame:
    """Like `load_or_run_bootstrap_study` (Part 9): skip the free-text cause
    parse -- the one non-vectorized, row-by-row step in Part 12 -- whenever
    it's already cached. The cache is a standalone document, one row per
    event: id, time, NUTS region(s), cause text, and every parsed hazard/loss magnitude
    (`NaN` wherever the free text or the reported loss doesn't supply one) --
    not just the columns merged back onto `events` below.
    """
    if not rebuild_cache and cache_path.exists():
        cached = pd.read_csv(cache_path, encoding="utf-8-sig")
        if set(events["event_id"]).issubset(set(cached["event_id"])):
            print(f"Loaded cached cause-text parse for {len(cached)} events from {cache_path}")
            return events.merge(cached[["event_id", *PARSED_HAZARD_COLUMNS]], on="event_id", how="left")
        print(f"Cached parse at {cache_path} doesn't cover every current event; re-parsing.")

    parsed_hazards = events["cause_raw"].apply(parse_cause_text).apply(pd.Series)
    merged = pd.concat([events, parsed_hazards], axis=1)

    document = merged[["event_id", "start_date", "end_date", "year", HANZE_REGION_COL, "cause_raw",
                        "losses_euro_total", *PARSED_HAZARD_COLUMNS]]
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    document.to_csv(cache_path, index=False)
    print(f"Parsed {len(document)} events' cause text and cached to {cache_path}")
    return merged


hanze_events = load_or_parse_hanze_causes(hanze_events)

HANZE_HAZARD_COLUMNS = ["rain_amount_mm", "intensity_mm_per_h", "equiv_mm_per_24h"]
HANZE_HAZARD_LABELS = {
    "rain_amount_mm": "Rainfall amount (mm)",
    "intensity_mm_per_h": "Rainfall intensity (mm/h)",
    "equiv_mm_per_24h": "Rainfall equivalent (mm/24h)",
}
hazard_coverage = pd.DataFrame({
    "nonmissing_hazard": hanze_events[HANZE_HAZARD_COLUMNS].notna().sum(),
    "positive_hazard_and_loss": [
        ((hanze_events[column] > 0) & (hanze_events["losses_euro_total"] > 0)).sum()
        for column in HANZE_HAZARD_COLUMNS
    ],
})
display(hazard_coverage)


### 12c. Fitting real damage functions

Empirical VaR/ES (`empirical_var`/`empirical_es`) are the same functionals
used throughout this notebook, computed directly on the real sample.

Fitting reuses Part 8's own machinery exactly as the bootstrap loop does --
`build_methods_and_train_on`/`_fit_predictors_for_subsample` fit every
method in `_DEFAULT_METHODS` **once** on the real (hazard, loss) pairs (no
bootstrap resampling -- there is only one dataset here), and
`_pointwise_summary`/`_tail_summary`/`_marginal_risk_summary` (the exact
functions that already dispatch mean-curve vs. IDR vs. EVT models inside the
bootstrap loop) compute each method's fitted mean/VaR/ES at the requested
quantile levels, reusing that same dispatch instead of a second
implementation.
`plot_hanze_hazard_loss_fits` is this Part's primary figure: the raw
hazard-loss scatter with every fitted method's prediction curve overlaid.

**Cell below:** `positive_xy`, `empirical_var`/`empirical_es`,
`fit_hanze_damage_models`/`summarize_hanze_damage_models`,
`plot_hanze_hazard_loss_fits`, `display_scrollable_results`.


In [ ]:
HANZE_QUANTILES = (0.50, 0.95, 0.99, 0.999, 0.9999)


def positive_xy(data, hazard_var):
    xy = data[[hazard_var, "losses_euro_total"]].apply(pd.to_numeric, errors="coerce").dropna()
    return xy[(xy[hazard_var] > 0) & (xy["losses_euro_total"] > 0)].copy()


def empirical_var(values, q):
    """Empirical VaR, delegating to `_sample_var_es` (Part 7) so HANZE uses
    the exact same inverted-cdf convention as the bootstrap study rather than
    a separately maintained implementation."""
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.nan
    var, _ = _sample_var_es(values, np.asarray([q]))
    return float(var[0])


def empirical_es(values, q):
    """Empirical (coherent/superquantile) ES, delegating to `_sample_var_es`
    (Part 7) so HANZE uses the exact same ES functional as the bootstrap
    study instead of a plain tail mean that over-counts the boundary atom."""
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.nan
    _, es = _sample_var_es(values, np.asarray([q]))
    return float(es[0])


def fit_hanze_damage_models(x, y, n_bins=10):
    """Fit every _DEFAULT_METHODS entry once on real (hazard, loss) pairs --
    the same dispatch Part 8's bootstrap loop uses, just called a single
    time instead of per subsample."""
    xb, yb = bin_summary(np.column_stack([x, y]), num_bins=min(max(2, int(n_bins)), len(x)), summary_func=np.mean)
    ok = np.isfinite(xb) & np.isfinite(yb) & (xb > 0) & (yb > 0)
    xb, yb = xb[ok], yb[ok]

    methods, train_on, specs = build_methods_and_train_on(_DEFAULT_METHODS)
    predictors, diagnostics = _fit_predictors_for_subsample(methods, train_on, x, y, xb, yb, collect_diagnostics=True)
    return predictors, specs, diagnostics


def summarize_hanze_damage_models(x, y, predictors, quantiles=HANZE_QUANTILES):
    """VaR/ES and conditional-mean tables for every fitted method, using
    Part 8's own per-method summary dispatch -- no ground truth here, so
    only fitted values are reported."""
    risk_rows, expectation_rows = [], []
    for q in quantiles:
        hazard_var = empirical_var(x, q)
        hazard_tail_mask = x >= hazard_var
        order, idxs = _prepare_tail_indexing(x, np.array([hazard_var]))

        risk = {
            "q": q, "Hazard VaR": hazard_var,
            "Hazard ES": empirical_es(x, q),
            "Tail n": int(np.sum(hazard_tail_mask)),
            "Observed loss VaR": empirical_var(y, q), "Observed loss ES": empirical_es(y, q),
        }
        expectation = {
            "q": q, "Hazard VaR": hazard_var,
            "Observed E[L | H >= VaR]": float(y[hazard_tail_mask].mean()) if np.any(hazard_tail_mask) else np.nan,
        }

        for name, model in predictors.items():
            if getattr(model, "failed", False):
                risk[f"{name} loss VaR"] = risk[f"{name} loss ES"] = np.nan
                expectation[f"{name} E[L | H = VaR]"] = expectation[f"{name} E[L | H >= VaR]"] = np.nan
                continue
            point = _pointwise_summary(model, np.array([hazard_var]), np.array([q]))
            tail = _tail_summary(model, x, np.array([hazard_var]), np.array([q]), order, idxs)
            marginal = _marginal_risk_summary(model, x, np.array([q]))
            expectation[f"{name} E[L | H = VaR]"] = float(point["mean"][0])
            expectation[f"{name} E[L | H >= VaR]"] = float(tail["mean"][0])
            risk[f"{name} loss VaR"] = float(marginal["value_at_risk"][0])
            risk[f"{name} loss ES"] = float(marginal["expected_shortfall"][0])

        risk_rows.append(risk)
        expectation_rows.append(expectation)

    return pd.DataFrame(risk_rows), pd.DataFrame(expectation_rows)


def plot_hanze_hazard_loss_fits(x, y, predictors, hazard_var_name, title=None, ax=None, fig_name="hanze_hazard_loss_fits"):
    """Hazard-loss scatter with every fitted method's prediction curve --
    this Part's primary figure."""
    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(9.0, 5.6), facecolor="white")
    ax.scatter(x, y, s=22, alpha=0.45, color="0.55", edgecolor="none", label=f"{len(x):,} events", zorder=1)
    if x.max() > x.min():
        x_grid = np.geomspace(x.min(), x.max(), 300)
        for name, model in predictors.items():
            if getattr(model, "failed", False):
                continue
            prediction = np.asarray(model.predict(x_grid), dtype=float)
            valid = np.isfinite(prediction) & (prediction > 0)
            if np.any(valid):
                ax.plot(x_grid[valid], prediction[valid], color=METHOD_COLORS.get(name, "#999999"), linewidth=2.0, label=name, zorder=3)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(HANZE_HAZARD_LABELS.get(hazard_var_name, hazard_var_name), fontsize=10.5, labelpad=8)
    ax.set_ylabel("Aggregate loss (€ 2020)", fontsize=10.5, labelpad=10)
    ax.set_title(title or "Hazard-loss pairs and fitted damage functions", fontsize=11.0, pad=12)
    ax.legend(loc="best", fontsize=10.8, ncol=2)
    if own_fig:
        fig.tight_layout(pad=0.7)
        save_figure(fig, fig_name)
        display(fig)
        plt.close(fig)
    return ax


def display_scrollable_results(table, title):
    formatted = table.copy()
    for column in formatted.select_dtypes(include=[np.number]).columns:
        if column == "Tail n":
            formatted[column] = formatted[column].map(lambda value: f"{int(value)}")
        elif column == "q":
            formatted[column] = formatted[column].map(lambda value: f"{value:.4g}")
        else:
            formatted[column] = formatted[column].map(lambda value: "" if pd.isna(value) else f"{value:.5g}")
    html = formatted.to_html(index=False, escape=True, border=0)
    display(HTML(
        f"<h4>{title}</h4>"
        f"<div style='overflow-x:auto; max-width:100%; max-height:360px'>{html}</div>"
    ))


### 12d. Interactive dashboard

Filters (country, NUTS region, year range, hazard type, loss-reporting
status, hazard variable, bin count) select a subset of `hanze_events`;
"Render analysis" fits and plots for that subset. The dashboard shows, in
order: **(1)** a short description of the current selection, **(2)** the
primary figure -- the hazard-loss scatter with every fitted method's curve
-- then supporting panels carried over from the original analysis:
distribution histograms, VaR/ES and
conditional-expectation tables, and the sortable selected-events table.

The bin-count control defaults to `50` (Part 7a/7b's `Quadratic`/`Sigmoid`
significance screens need enough binned design points to detect a real
trend -- with real, noisy HANZE data and only a handful of bins, both can
be rejected outright even when a signal is genuinely there; `50` matches
the bin count used everywhere else in this notebook). Any method whose
curve is still absent for the current selection -- rejected by its
significance screen, or genuinely failed to fit -- is listed under the main
figure with its reason, rather than silently missing.

**Cell below:** the widget controls, `filtered_events`, `render_analysis`,
and the dashboard layout -- rendered once automatically (unfiltered) so
something appears even without interacting, with the widgets left live for
further filtering.


In [ ]:
import io
import html as _html


if widgets is None:
    print(
        "ipywidgets is not installed; skipping the interactive HANZE dashboard. "
        "Install it (`pip install ipywidgets`) and re-run this cell to use it."
    )

else:
    # -------------------------------------------------------------------------
    # Available filter values
    # -------------------------------------------------------------------------

    _hanze_all_region_codes = sorted({
        code
        for regions in hanze_events["regions"]
        for code in regions
    })

    _hanze_all_countries = sorted(
        hanze_events["country"].dropna().astype(str).unique()
    )

    _hanze_all_hazard_types = sorted(
        hanze_events["hazard_main_type"].dropna().astype(str).unique()
    )

    _hanze_default_hazard = hazard_coverage[
        "positive_hazard_and_loss"
    ].idxmax()

    _hanze_year_min = int(hanze_events["year"].min())
    _hanze_year_max = int(hanze_events["year"].max())


    # -------------------------------------------------------------------------
    # Controls
    # -------------------------------------------------------------------------

    hanze_country_widget = widgets.SelectMultiple(
        options=_hanze_all_countries,
        description="Country",
        rows=7,
        layout=widgets.Layout(
            width="310px",
            height="165px",
        ),
    )

    hanze_prefix_widget = widgets.Text(
        description="NUTS prefix",
        placeholder="e.g. CH, DE1, FR",
        layout=widgets.Layout(width="310px"),
    )

    hanze_region_widget = widgets.SelectMultiple(
        options=_hanze_all_region_codes,
        description="Exact NUTS",
        rows=8,
        layout=widgets.Layout(
            width="310px",
            height="185px",
        ),
    )

    hanze_refresh_regions_button = widgets.Button(
        description="Refresh region list"
    )

    hanze_year_widget = widgets.IntRangeSlider(
        value=[_hanze_year_min, _hanze_year_max],
        min=_hanze_year_min,
        max=_hanze_year_max,
        description="Years",
        continuous_update=False,
        layout=widgets.Layout(width="650px"),
    )

    hanze_hazard_type_widget = widgets.Dropdown(
        options=["all", *_hanze_all_hazard_types],
        value="all",
        description="Hazard type",
    )

    hanze_hazard_var_widget = widgets.Dropdown(
        options=HANZE_HAZARD_COLUMNS,
        value=_hanze_default_hazard,
        description="Hazard variable",
    )

    hanze_loss_mode_widget = widgets.Dropdown(
        options=[
            ("All events", "all"),
            ("Nonmissing losses", "nonmissing"),
            ("Positive losses", "positive"),
        ],
        value="positive",
        description="Loss filter",
    )

    hanze_bin_count_widget = widgets.IntSlider(
        value=10,
        min=2,
        max=50,
        step=1,
        description="Number of bins",
        continuous_update=False,
    )

    hanze_run_button = widgets.Button(
        description="Render analysis",
        button_style="primary",
    )


    # -------------------------------------------------------------------------
    # Dashboard output widgets
    #
    # IMPORTANT:
    # These deliberately do NOT use widgets.Output.
    #
    # figures -> widgets.Image
    # tables  -> widgets.HTML
    # text    -> widgets.HTML
    #
    # That bypasses VS Code's Output-widget IOPub capture entirely.
    # -------------------------------------------------------------------------

    _hanze_panel_layout = widgets.Layout(
        width="100%",
        max_width="100%",
    )

    hanze_summary_widget = widgets.HTML(
        layout=_hanze_panel_layout
    )

    hanze_fits_box = widgets.VBox(
        layout=_hanze_panel_layout
    )

    hanze_dist_box = widgets.VBox(
        layout=_hanze_panel_layout
    )

    hanze_results_box = widgets.VBox(
        layout=_hanze_panel_layout
    )

    hanze_table_box = widgets.VBox(
        layout=_hanze_panel_layout
    )


    # -------------------------------------------------------------------------
    # Figure -> Image widget
    # -------------------------------------------------------------------------

    def _hanze_figure_widget(fig, name=None, dpi=140):
        """
        Convert a Matplotlib figure directly to a widgets.Image.

        No display(fig), no widgets.Output, and the figure is closed
        immediately after conversion.
        """

        if name is not None:
            save_figure(fig, name)

        buffer = io.BytesIO()

        fig.savefig(
            buffer,
            format="png",
            dpi=dpi,
            bbox_inches="tight",
            facecolor=fig.get_facecolor(),
        )

        plt.close(fig)

        return widgets.Image(
            value=buffer.getvalue(),
            format="png",
            layout=widgets.Layout(
                width="100%",
                max_width="100%",
            ),
        )


    # -------------------------------------------------------------------------
    # Small HTML helpers
    # -------------------------------------------------------------------------

    def _hanze_message_widget(message):
        return widgets.HTML(
            value=(
                "<div style='"
                "padding:8px 10px;"
                "margin:4px 0 8px 0;"
                "background:#f7f7f7;"
                "border-left:3px solid #888;"
                "font-family:monospace;"
                "white-space:pre-wrap;"
                "'>"
                f"{_html.escape(str(message))}"
                "</div>"
            ),
            layout=_hanze_panel_layout,
        )


    def _hanze_results_widget(table, title):
        """
        Replacement for display_scrollable_results() that returns an
        ordinary HTML widget instead of calling display().
        """

        formatted = table.copy()

        for column in formatted.select_dtypes(
            include=[np.number]
        ).columns:

            if column == "Tail n":
                formatted[column] = formatted[column].map(
                    lambda value:
                    "" if pd.isna(value)
                    else f"{int(value)}"
                )

            elif column == "q":
                formatted[column] = formatted[column].map(
                    lambda value:
                    "" if pd.isna(value)
                    else f"{value:.4g}"
                )

            else:
                formatted[column] = formatted[column].map(
                    lambda value:
                    "" if pd.isna(value)
                    else f"{value:.5g}"
                )

        table_html = formatted.to_html(
            index=False,
            escape=True,
            border=0,
        )

        return widgets.HTML(
            value=(
                f"<h4>{_html.escape(title)}</h4>"
                "<div style='"
                "overflow-x:auto;"
                "max-width:100%;"
                "max-height:360px;"
                "overflow-y:auto;"
                "'>"
                f"{table_html}"
                "</div>"
            ),
            layout=_hanze_panel_layout,
        )


    def _hanze_events_table_widget(table):
        table_html = table.to_html(
            index=False,
            escape=True,
            border=0,
        )

        return widgets.HTML(
            value=(
                "<div style='"
                "overflow-x:auto;"
                "overflow-y:auto;"
                "max-width:100%;"
                "max-height:520px;"
                "'>"
                f"{table_html}"
                "</div>"
            ),
            layout=_hanze_panel_layout,
        )


    # -------------------------------------------------------------------------
    # Filtering helpers
    # -------------------------------------------------------------------------

    def _hanze_event_has_region(
        regions,
        prefix="",
        exact=(),
    ):
        if exact and not any(
            code in exact
            for code in regions
        ):
            return False

        if prefix and not any(
            code.upper().startswith(prefix)
            for code in regions
        ):
            return False

        return True


    def _hanze_candidate_region_codes():
        data = hanze_events

        selected_countries = tuple(
            hanze_country_widget.value
        )

        if selected_countries:
            data = data[
                data["country"].isin(
                    selected_countries
                )
            ]

        prefix = (
            hanze_prefix_widget
            .value
            .strip()
            .upper()
        )

        return sorted({
            code
            for regions in data["regions"]
            for code in regions
            if (
                not prefix
                or code.upper().startswith(prefix)
            )
        })


    def hanze_refresh_region_options(_=None):
        old_selection = set(
            hanze_region_widget.value
        )

        options = (
            _hanze_candidate_region_codes()
        )

        hanze_region_widget.options = options

        hanze_region_widget.value = tuple(
            code
            for code in options
            if code in old_selection
        )


    def filtered_events():
        data = hanze_events.copy()

        # Year
        y0, y1 = hanze_year_widget.value

        data = data[
            data["year"].between(y0, y1)
        ]

        # Country
        selected_countries = tuple(
            hanze_country_widget.value
        )

        if selected_countries:
            data = data[
                data["country"].isin(
                    selected_countries
                )
            ]

        # NUTS region
        prefix = (
            hanze_prefix_widget
            .value
            .strip()
            .upper()
        )

        exact = set(
            hanze_region_widget.value
        )

        if prefix or exact:
            mask = data["regions"].apply(
                _hanze_event_has_region,
                prefix=prefix,
                exact=exact,
            )

            data = data[mask]

        # Hazard type
        if (
            hanze_hazard_type_widget.value
            != "all"
        ):
            data = data[
                data["hazard_main_type"]
                == hanze_hazard_type_widget.value
            ]

        # Loss filter
        if (
            hanze_loss_mode_widget.value
            == "nonmissing"
        ):
            data = data[
                data["losses_euro_total"].notna()
            ]

        elif (
            hanze_loss_mode_widget.value
            == "positive"
        ):
            data = data[
                data["losses_euro_total"]
                .fillna(0)
                > 0
            ]

        if data["event_id"].duplicated().any():
            raise AssertionError(
                "Widget filtering unexpectedly "
                "produced duplicate event rows."
            )

        return data


    # -------------------------------------------------------------------------
    # Main render function
    # -------------------------------------------------------------------------

    def render_analysis(_=None):

        # Remove previous widget children immediately.
        # This is a direct trait assignment: no Output.clear_output().
        hanze_fits_box.children = ()
        hanze_dist_box.children = ()
        hanze_results_box.children = ()
        hanze_table_box.children = ()

        data = filtered_events()

        hazard_var = (
            hanze_hazard_var_widget.value
        )

        xy = positive_xy(
            data,
            hazard_var,
        )

        x = xy[
            hazard_var
        ].to_numpy(dtype=float)

        y = xy[
            "losses_euro_total"
        ].to_numpy(dtype=float)

        n_selected = len(data)
        positive_pairs = len(xy)

        total_loss = data[
            "losses_euro_total"
        ].sum(min_count=1)
        n_reported_loss = int(
            data["losses_euro_total"].notna().sum()
        )

        positive_loss_mask = data["losses_euro_total"] > 0
        median_loss = data.loc[
            positive_loss_mask,
            "losses_euro_total",
        ].median()
        n_positive_loss = int(positive_loss_mask.sum())

        positive_hazard_mask = data[hazard_var] > 0
        median_hazard = data.loc[
            positive_hazard_mask,
            hazard_var,
        ].median()
        n_positive_hazard = int(positive_hazard_mask.sum())

        pair_share = (
            100.0
            * positive_pairs
            / n_selected
            if n_selected
            else np.nan
        )

        # Estimation-sample stats: x/y (from `xy = positive_xy(data,
        # hazard_var)`) already restrict to events with *both* a positive
        # hazard_var and a positive reported loss -- the exact sample the
        # damage models are fit on and the risk table's "Observed"/VaR/ES
        # figures below are computed from. Reported separately from the
        # wider-selection stats above, since the two samples can differ --
        # events with a recorded hazard value are not a random subset of
        # all positive-loss events.
        est_median_loss = float(np.median(y)) if len(y) else np.nan
        est_total_loss = float(np.sum(y)) if len(y) else np.nan
        est_median_hazard = float(np.median(x)) if len(x) else np.nan


        # ---------------------------------------------------------------------
        # 1. Selection summary
        # ---------------------------------------------------------------------

        year_range = (
            f"{int(data['year'].min())}\u2013{int(data['year'].max())}"
            if n_selected else "n/a"
        )

        hanze_summary_widget.value = (
            "<div style='"
            "padding:10px 12px;"
            "border-left:4px solid #1f77b4;"
            "background:#f5f8fb"
            "'>"

            f"<b>{n_selected:,} unique aggregate events "
            f"selected</b> across "
            f"{data['country'].nunique():,} countries, "
            f"covering <b>{year_range}</b>. "

            "The damage models and the VaR/ES table below are fit and "
            "evaluated on the <b>estimation sample</b>: "
            f"<b>{positive_pairs:,} positive "
            "hazard–loss pairs</b> "
            f"({pair_share:.1f}% of selected events) with both a positive "
            f"{_html.escape(hazard_var)} and a positive reported loss. "
            f"Within this sample: median loss "
            f"<b>€{est_median_loss:,.0f}</b>, aggregate loss "
            f"<b>€{est_total_loss:,.0f}</b>, median "
            f"{_html.escape(hazard_var)} "
            f"<b>{est_median_hazard:,.3g}</b>. "

            "<br>For context, over the wider selection above "
            "(<i>not</i> the estimation sample used below): "
            "reported aggregate loss total across "
            f"<b>{n_reported_loss:,}</b> events with a reported loss: "
            f"<b>€{total_loss:,.0f}</b>; "
            f"median positive loss across <b>{n_positive_loss:,}</b> "
            "events with loss > 0 (hazard not required): "
            f"<b>€{median_loss:,.0f}</b>; "
            f"median positive {_html.escape(hazard_var)} across "
            f"<b>{n_positive_hazard:,}</b> events with "
            f"{_html.escape(hazard_var)} > 0 (loss not required): "
            f"<b>{median_hazard:,.3g}</b>."

            "</div>"
        )


        # ---------------------------------------------------------------------
        # 2. Hazard-loss scatter + fitted damage functions
        # ---------------------------------------------------------------------

        if len(xy) < 4:

            hanze_fits_box.children = (
                _hanze_message_widget(
                    "At least four positive hazard-loss "
                    "pairs are needed to fit; "
                    f"found {len(xy)}."
                ),
            )

            predictors = None

        else:

            predictors, specs, diagnostics = (
                fit_hanze_damage_models(
                    x,
                    y,
                    n_bins=hanze_bin_count_widget.value,
                )
            )

            # Pass an explicit axis so
            # plot_hanze_hazard_loss_fits()
            # does NOT call display(fig).
            fig, ax = plt.subplots(
                figsize=(9.0, 5.6),
                facecolor="white",
            )

            plot_hanze_hazard_loss_fits(
                x,
                y,
                predictors,
                hazard_var,
                title=(
                    f"{hazard_var} vs. aggregate loss "
                    f"({positive_pairs:,} events)"
                ),
                ax=ax,
            )

            fig.tight_layout(pad=0.7)

            fits_children = [
                _hanze_figure_widget(
                    fig,
                    name="hanze_hazard_loss_fits",
                )
            ]

            # Report methods whose curves are absent.
            not_shown = {
                d["method"]: d["message"]
                for d in diagnostics
                if d["status"] != "ok"
            }

            if not_shown:
                details = "; ".join(
                    f"{name} ({msg})"
                    for name, msg
                    in not_shown.items()
                )

                fits_children.append(
                    _hanze_message_widget(
                        "No curve shown for: "
                        f"{details}"
                    )
                )

            hanze_fits_box.children = tuple(
                fits_children
            )


        # ---------------------------------------------------------------------
        # 3. Distributions
        # ---------------------------------------------------------------------

        dist_children = []

        fig, axes = plt.subplots(
            1,
            2,
            figsize=(11, 4.0),
            facecolor="white",
        )

        losses = data.loc[
            data["losses_euro_total"] > 0,
            "losses_euro_total",
        ]

        hazards = data.loc[
            data[hazard_var] > 0,
            hazard_var,
        ]

        if len(losses):
            axes[0].hist(
                np.log10(losses),
                bins=min(
                    30,
                    max(
                        6,
                        int(np.sqrt(len(losses))),
                    ),
                ),
                color=METHOD_COLORS["Sigmoid"],
            )

            axes[0].set_xlabel(
                "log10 aggregate loss (€2020)",
                fontsize=10.5,
            )

        else:
            axes[0].text(
                0.5,
                0.5,
                "No positive loss data",
                ha="center",
                va="center",
                transform=axes[0].transAxes,
            )

        if len(hazards):
            axes[1].hist(
                np.log10(hazards),
                bins=min(
                    30,
                    max(
                        6,
                        int(np.sqrt(len(hazards))),
                    ),
                ),
                color=METHOD_COLORS["Quadratic"],
            )

            axes[1].set_xlabel(
                "log10 "
                + HANZE_HAZARD_LABELS.get(
                    hazard_var,
                    hazard_var,
                ),
                fontsize=10.5,
            )

        else:
            axes[1].text(
                0.5,
                0.5,
                "No positive hazard data",
                ha="center",
                va="center",
                transform=axes[1].transAxes,
            )

        fig.suptitle(
            "Distributions of the selected events",
            fontsize=11.5,
        )

        fig.tight_layout()

        dist_children.append(
            _hanze_figure_widget(
                fig,
                name="hanze_distributions",
            )
        )

        hanze_dist_box.children = tuple(
            dist_children
        )


        # ---------------------------------------------------------------------
        # 4. Damage-model tables
        # ---------------------------------------------------------------------

        if len(xy) < 4:

            hanze_results_box.children = (
                _hanze_message_widget(
                    "At least four positive hazard-loss "
                    "pairs are needed; "
                    f"found {len(xy)}."
                ),
            )

        else:

            risk_table, expectation_table = (
                summarize_hanze_damage_models(
                    x,
                    y,
                    predictors,
                )
            )

            hanze_results_box.children = (

                _hanze_results_widget(
                    risk_table,
                    "Hazard and loss VaR / ES results",
                ),

                _hanze_results_widget(
                    expectation_table,
                    "Conditional expected-loss results",
                ),
            )


        # ---------------------------------------------------------------------
        # 5. Selected aggregate-event table
        # ---------------------------------------------------------------------

        table_cols = [
            "event_id",
            "year",
            "country",
            HANZE_REGION_COL,
            "losses_euro_total",
            "hazard_main_type",
            hazard_var,
            "cause_raw",
        ]

        max_rows = 200

        selected_table = (
            data[table_cols]
            .sort_values(
                "losses_euro_total",
                ascending=False,
                na_position="last",
            )
            .head(max_rows)
        )

        table_children = [
            _hanze_events_table_widget(
                selected_table
            )
        ]

        if len(data) > max_rows:

            table_children.append(
                _hanze_message_widget(
                    f"Showing the first {max_rows:,} "
                    f"of {len(data):,} selected "
                    "aggregate events."
                )
            )

        hanze_table_box.children = tuple(
            table_children
        )


    # -------------------------------------------------------------------------
    # Button callback
    # -------------------------------------------------------------------------

    def _hanze_on_render(_=None):

        if hanze_run_button.disabled:
            return

        hanze_run_button.disabled = True

        old_description = (
            hanze_run_button.description
        )

        hanze_run_button.description = (
            "Rendering…"
        )

        try:
            render_analysis()

        finally:
            hanze_run_button.description = (
                old_description
            )

            hanze_run_button.disabled = False


    hanze_refresh_regions_button.on_click(
        hanze_refresh_region_options
    )

    hanze_run_button.on_click(
        _hanze_on_render
    )


    # -------------------------------------------------------------------------
    # Responsive control layout
    # -------------------------------------------------------------------------

    _hanze_row_layout = widgets.Layout(
        width="100%",
        display="flex",
        flex_flow="row wrap",
        align_items="flex-start",
    )

    for control in [
        hanze_country_widget,
        hanze_prefix_widget,
        hanze_region_widget,
        hanze_hazard_type_widget,
        hanze_hazard_var_widget,
        hanze_loss_mode_widget,
        hanze_bin_count_widget,
    ]:

        control.layout.width = "auto"
        control.layout.min_width = "280px"
        control.layout.flex = "1 1 280px"

        control.style.description_width = (
            "initial"
        )

    hanze_year_widget.style.description_width = (
        "initial"
    )

    hanze_year_widget.layout.width = "100%"


    hanze_location_controls = widgets.VBox(
        [
            hanze_country_widget,
            hanze_prefix_widget,
            hanze_refresh_regions_button,
        ],
        layout=widgets.Layout(
            width="auto",
            min_width="280px",
            flex="1 1 280px",
        ),
    )


    hanze_controls = widgets.VBox(
        [

            widgets.HBox(
                [
                    hanze_location_controls,
                    hanze_region_widget,
                ],
                layout=_hanze_row_layout,
            ),

            hanze_year_widget,

            widgets.HBox(
                [
                    hanze_hazard_type_widget,
                    hanze_hazard_var_widget,
                    hanze_loss_mode_widget,
                ],
                layout=_hanze_row_layout,
            ),

            widgets.HBox(
                [
                    hanze_bin_count_widget,
                    hanze_run_button,
                ],
                layout=_hanze_row_layout,
            ),
        ],
        layout=widgets.Layout(
            width="100%",
            max_width="100%",
        ),
    )


    # -------------------------------------------------------------------------
    # Dashboard
    # -------------------------------------------------------------------------

    hanze_dashboard = widgets.VBox(
        [

            widgets.HTML(
                "<h3>Controls</h3>"
            ),
            hanze_controls,

            widgets.HTML(
                "<h3>Selection summary</h3>"
            ),
            hanze_summary_widget,

            widgets.HTML(
                "<h3>Hazard-loss pairs and "
                "fitted damage functions</h3>"
            ),
            hanze_fits_box,

            widgets.HTML(
                "<h3>Distributions</h3>"
            ),
            hanze_dist_box,

            widgets.HTML(
                "<h3>Damage-model results</h3>"
            ),
            hanze_results_box,

            widgets.HTML(
                "<h3>Selected aggregate events</h3>"
            ),
            hanze_table_box,
        ],
        layout=widgets.Layout(
            width="100%",
            max_width="100%",
            overflow_x="hidden",
        ),
    )


    # -------------------------------------------------------------------------
    # Initial render
    #
    # Populate ordinary widget traits first, then display the dashboard once.
    # -------------------------------------------------------------------------

    hanze_refresh_region_options()

    render_analysis()

    display(hanze_dashboard)